# Q-MedAI Final Experiment

This notebook reproduces the FINAL EXPERIMENT methodology: a leakage-safe full-data comparison, a separate fair 150-row matched-data comparison, a 4-qubit/3-layer/100-iteration VQC, and a 150-row quantum kernel. It does not perform a sweep and does not claim GPU use, hardware execution, or quantum advantage.

In [19]:
# First execution cell: locate the supplied CSV and print the exact path used.
from pathlib import Path

candidates = []
for root in (Path('/kaggle/input'), Path.cwd(), Path('/kaggle/working')):
    if root.exists():
        candidates.extend(root.glob('**/breast_cancer.csv'))
DATASET_PATH = next((path for path in candidates if path.is_file()), None)
if DATASET_PATH is None:
    raise FileNotFoundError(
        'breast_cancer.csv was not found. Add or upload the Q-MedAI dataset, then run this cell again.'
    )
print(f'DATASET PATH USED: {DATASET_PATH.resolve()}')


DATASET PATH USED: /kaggle/input/datasets/adityagp7/q-medai-breast-cancer-dataset/breast_cancer.csv


In [32]:
# ============================================================
# CELL 1 — VERIFY KAGGLE INPUTS
# Q-MedAI | SIH 2026 | PS 26139
# ============================================================

from pathlib import Path
import os
import sys

print("=" * 70)
print("Q-MedAI — Kaggle Input Verification")
print("=" * 70)

# ------------------------------------------------------------
# 1. Candidate dataset directories
# ------------------------------------------------------------

DATASET_DIR_CANDIDATES = [
    Path("/kaggle/input/datasets/adityagp7/q-medai-breast-cancer-dataset"),
    Path("/kaggle/input/q-medai-breast-cancer-dataset"),
    Path("/kaggle/input/adityagp7/q-medai-breast-cancer-dataset"),
]

SOURCE_DIR_CANDIDATES = [
    Path("/kaggle/input/datasets/adityagp7/q-medai-project-source"),
    Path("/kaggle/input/q-medai-project-source"),
    Path("/kaggle/input/adityagp7/q-medai-project-source"),
]


def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return None


DATASET_INPUT_ROOT = first_existing(DATASET_DIR_CANDIDATES)
SOURCE_INPUT_ROOT = first_existing(SOURCE_DIR_CANDIDATES)


# ------------------------------------------------------------
# 2. Fall back to searching /kaggle/input
# ------------------------------------------------------------

KAGGLE_INPUT = Path("/kaggle/input")

if DATASET_INPUT_ROOT is None and KAGGLE_INPUT.exists():
    csv_matches = list(KAGGLE_INPUT.glob("**/breast_cancer.csv"))

    if csv_matches:
        DATASET_INPUT_ROOT = csv_matches[0].parent


if SOURCE_INPUT_ROOT is None and KAGGLE_INPUT.exists():
    source_matches = list(KAGGLE_INPUT.glob("**/src/final_experiment.py"))

    if source_matches:
        # .../<project>/src/final_experiment.py
        SOURCE_INPUT_ROOT = source_matches[0].parents[1]


# ------------------------------------------------------------
# 3. Locate breast_cancer.csv
# ------------------------------------------------------------

DATASET_PATH = None

if DATASET_INPUT_ROOT is not None:
    direct_csv = DATASET_INPUT_ROOT / "breast_cancer.csv"

    if direct_csv.exists():
        DATASET_PATH = direct_csv
    else:
        matches = list(DATASET_INPUT_ROOT.glob("**/breast_cancer.csv"))

        if matches:
            DATASET_PATH = matches[0]


# ------------------------------------------------------------
# 4. Locate project root
# ------------------------------------------------------------

PROJECT_ROOT = None
FINAL_EXPERIMENT_FILE = None

if SOURCE_INPUT_ROOT is not None:

    # Case A:
    # SOURCE_INPUT_ROOT itself is the project root
    candidate = SOURCE_INPUT_ROOT / "src" / "final_experiment.py"

    if candidate.exists():
        PROJECT_ROOT = SOURCE_INPUT_ROOT
        FINAL_EXPERIMENT_FILE = candidate

    else:
        # Case B:
        # Dataset contains an extra top-level project directory
        matches = list(
            SOURCE_INPUT_ROOT.glob("**/src/final_experiment.py")
        )

        if matches:
            FINAL_EXPERIMENT_FILE = matches[0]
            PROJECT_ROOT = FINAL_EXPERIMENT_FILE.parents[1]


# ------------------------------------------------------------
# 5. Print diagnostic information
# ------------------------------------------------------------

print()
print("Dataset input root:")
print(DATASET_INPUT_ROOT)

print()
print("Dataset CSV:")
print(DATASET_PATH)

print()
print("Project source input root:")
print(SOURCE_INPUT_ROOT)

print()
print("Detected project root:")
print(PROJECT_ROOT)

print()
print("Final experiment module:")
print(FINAL_EXPERIMENT_FILE)


# ------------------------------------------------------------
# 6. Fail early with useful errors
# ------------------------------------------------------------

if DATASET_PATH is None:
    print("\nFiles currently visible under /kaggle/input:")
    
    if KAGGLE_INPUT.exists():
        for path in list(KAGGLE_INPUT.glob("**/*"))[:100]:
            print(" ", path)

    raise FileNotFoundError(
        "Could not locate breast_cancer.csv. "
        "Make sure q-medai-breast-cancer-dataset is attached "
        "to this Kaggle notebook."
    )


if PROJECT_ROOT is None:
    print("\nPython files currently visible under /kaggle/input:")

    if KAGGLE_INPUT.exists():
        for path in list(KAGGLE_INPUT.glob("**/*.py"))[:100]:
            print(" ", path)

    raise FileNotFoundError(
        "Could not locate src/final_experiment.py. "
        "Make sure q-medai-project-source is attached "
        "to this Kaggle notebook."
    )


print()
print("=" * 70)
print("INPUT CHECK PASSED")
print("=" * 70)

print(f"DATASET PATH USED : {DATASET_PATH}")
print(f"PROJECT ROOT      : {PROJECT_ROOT}")

Q-MedAI — Kaggle Input Verification

Dataset input root:
/kaggle/input/datasets/adityagp7/q-medai-breast-cancer-dataset

Dataset CSV:
/kaggle/input/datasets/adityagp7/q-medai-breast-cancer-dataset/breast_cancer.csv

Project source input root:
/kaggle/input/datasets/adityagp7/q-medai-project-source

Detected project root:
/kaggle/input/datasets/adityagp7/q-medai-project-source/Q-MedAI

Final experiment module:
/kaggle/input/datasets/adityagp7/q-medai-project-source/Q-MedAI/src/final_experiment.py

INPUT CHECK PASSED
DATASET PATH USED : /kaggle/input/datasets/adityagp7/q-medai-breast-cancer-dataset/breast_cancer.csv
PROJECT ROOT      : /kaggle/input/datasets/adityagp7/q-medai-project-source/Q-MedAI


In [34]:
# ============================================================
# CELL 2 — SAFE DEPENDENCY SETUP + QUANTUM RUNTIME TEST
# Q-MedAI | SIH 2026 | PS 26139
# ============================================================

import sys
import subprocess
import importlib
import importlib.util
from importlib.metadata import version, PackageNotFoundError

print("=" * 70)
print("Q-MedAI — Dependency & Quantum Runtime Check")
print("=" * 70)

# ------------------------------------------------------------
# 1. Pin the PennyLane version used by Q-MedAI
# ------------------------------------------------------------

REQUIRED_PENNYLANE = "0.45.1"


def installed_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


current_pl = installed_version("pennylane")

print(f"Python version          : {sys.version.split()[0]}")
print(f"PennyLane before setup : {current_pl}")


# ------------------------------------------------------------
# 2. Install PennyLane ONLY if necessary
#
# Important:
# At this stage NumPy / sklearn have NOT been imported yet,
# so dependency installation is much safer.
# ------------------------------------------------------------

if current_pl != REQUIRED_PENNYLANE:
    print()
    print(
        f"Installing PennyLane {REQUIRED_PENNYLANE} "
        f"(current={current_pl})..."
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--upgrade-strategy",
        "only-if-needed",
        f"pennylane=={REQUIRED_PENNYLANE}",
    ])

    importlib.invalidate_caches()

else:
    print("Required PennyLane version already installed.")


# ------------------------------------------------------------
# 3. Import scientific stack AFTER dependency setup
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import sklearn
import matplotlib
import pennylane as qml


# ------------------------------------------------------------
# 4. Version report
# ------------------------------------------------------------

print()
print("Verified package versions")
print("-" * 70)

print(f"Python       : {sys.version.split()[0]}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"Matplotlib   : {matplotlib.__version__}")
print(f"PennyLane    : {qml.__version__}")


# ------------------------------------------------------------
# 5. PennyLane default.qubit smoke test
# ------------------------------------------------------------

print()
print("Testing PennyLane default.qubit...")


dev_smoke = qml.device(
    "default.qubit",
    wires=2
)


@qml.qnode(dev_smoke)
def smoke_circuit(theta):
    qml.RY(theta, wires=0)
    qml.RY(theta / 2, wires=1)

    qml.CNOT(wires=[0, 1])

    return qml.expval(qml.PauliZ(0))


test_theta = 0.123
test_result = smoke_circuit(test_theta)

assert np.isfinite(float(test_result)), (
    "PennyLane returned a non-finite value."
)


# ------------------------------------------------------------
# 6. Tiny fidelity kernel smoke test
#
# This checks the primitive needed later by the
# Quantum Kernel SVM.
# ------------------------------------------------------------

kernel_dev = qml.device(
    "default.qubit",
    wires=2
)


def feature_map(x):
    qml.RY(x[0], wires=0)
    qml.RY(x[1], wires=1)

    qml.CNOT(wires=[0, 1])

    # data re-uploading after entanglement
    qml.RY(x[0] * x[1], wires=0)
    qml.RY(x[0] - x[1], wires=1)


@qml.qnode(kernel_dev)
def kernel_overlap(x1, x2):

    feature_map(x1)

    qml.adjoint(feature_map)(x2)

    return qml.probs(wires=[0, 1])


x_a = np.array([0.20, 0.40])
x_b = np.array([0.25, 0.35])

kernel_value = float(kernel_overlap(x_a, x_b)[0])

assert np.isfinite(kernel_value)
assert 0.0 <= kernel_value <= 1.0 + 1e-8


# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------

print()
print(f"default.qubit output    : {float(test_result):.8f}")
print(f"Kernel fidelity example : {kernel_value:.8f}")

print()
print("=" * 70)
print("DEPENDENCY CHECK PASSED")
print("QUANTUM RUNTIME CHECK PASSED")
print("=" * 70)

print(
    "\nBackend: PennyLane default.qubit "
    "(classical simulation of quantum circuits)"
)

Q-MedAI — Dependency & Quantum Runtime Check
Python version          : 3.12.13
PennyLane before setup : 0.45.1
Required PennyLane version already installed.

Verified package versions
----------------------------------------------------------------------
Python       : 3.12.13
NumPy        : 2.0.2
Pandas       : 2.3.3
scikit-learn : 1.6.1
Matplotlib   : 3.10.0
PennyLane    : 0.45.1

Testing PennyLane default.qubit...

default.qubit output    : 0.99244503
Kernel fidelity example : 0.99868367

DEPENDENCY CHECK PASSED
QUANTUM RUNTIME CHECK PASSED

Backend: PennyLane default.qubit (classical simulation of quantum circuits)


In [35]:
# ============================================================
# CELL 3 — PREPARE WRITABLE Q-MedAI PROJECT + VERIFY IMPORTS
# Q-MedAI | SIH 2026 | PS 26139
# ============================================================

from pathlib import Path
import shutil
import sys
import os
import importlib

print("=" * 70)
print("Q-MedAI — Project Source Setup")
print("=" * 70)

# PROJECT_ROOT came from Cell 1:
# /kaggle/input/datasets/adityagp7/q-medai-project-source/Q-MedAI

assert PROJECT_ROOT is not None
assert PROJECT_ROOT.exists(), f"Source project not found: {PROJECT_ROOT}"

# ------------------------------------------------------------
# 1. Kaggle /kaggle/input is read-only.
#    Copy the project to /kaggle/working before execution.
# ------------------------------------------------------------

WORK_PROJECT_ROOT = Path("/kaggle/working/Q-MedAI")

if WORK_PROJECT_ROOT.exists():
    print(f"Removing previous working copy: {WORK_PROJECT_ROOT}")
    shutil.rmtree(WORK_PROJECT_ROOT)

print(f"Copying project:\n  FROM: {PROJECT_ROOT}\n  TO:   {WORK_PROJECT_ROOT}")

shutil.copytree(
    PROJECT_ROOT,
    WORK_PROJECT_ROOT
)

assert WORK_PROJECT_ROOT.exists()


# ------------------------------------------------------------
# 2. Confirm expected project structure
# ------------------------------------------------------------

required_paths = [
    WORK_PROJECT_ROOT / "src",
    WORK_PROJECT_ROOT / "src" / "__init__.py",
    WORK_PROJECT_ROOT / "src" / "config.py",
    WORK_PROJECT_ROOT / "src" / "final_experiment.py",
    WORK_PROJECT_ROOT / "src" / "models",
    WORK_PROJECT_ROOT / "src" / "preprocessing",
    WORK_PROJECT_ROOT / "src" / "evaluation",
]

print()
print("Project structure check:")

missing = []

for path in required_paths:
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{status:8} {path.relative_to(WORK_PROJECT_ROOT)}")

    if not path.exists():
        missing.append(path)

if missing:
    raise FileNotFoundError(
        "Required project files are missing:\n"
        + "\n".join(str(p) for p in missing)
    )


# ------------------------------------------------------------
# 3. Put writable project first on Python import path
# ------------------------------------------------------------

project_string = str(WORK_PROJECT_ROOT)

# Remove duplicate if this cell is rerun
while project_string in sys.path:
    sys.path.remove(project_string)

sys.path.insert(0, project_string)

# Work from writable project directory
os.chdir(WORK_PROJECT_ROOT)

print()
print("Current working directory:")
print(Path.cwd())

print()
print("sys.path[0]:")
print(sys.path[0])


# ------------------------------------------------------------
# 4. Import Q-MedAI core modules
# ------------------------------------------------------------

importlib.invalidate_caches()

try:
    import src
    import src.config as config_module
    import src.final_experiment as final_experiment_module

except Exception as exc:
    print()
    print("PROJECT IMPORT FAILED")
    print(type(exc).__name__, ":", exc)
    raise


# ------------------------------------------------------------
# 5. Verify important experiment interfaces
# ------------------------------------------------------------

expected_final_symbols = [
    "FINAL_CONFIG",
    "run_final_experiment",
]

print()
print("Core import checks:")
print(f"src package              : {src.__file__}")
print(f"src.config               : {config_module.__file__}")
print(f"src.final_experiment     : {final_experiment_module.__file__}")

print()
print("Final experiment API:")

missing_symbols = []

for symbol in expected_final_symbols:
    exists = hasattr(final_experiment_module, symbol)

    print(
        f"{'FOUND' if exists else 'MISSING':8} "
        f"src.final_experiment.{symbol}"
    )

    if not exists:
        missing_symbols.append(symbol)

if missing_symbols:
    raise AttributeError(
        "The attached repository does not expose the expected "
        "final experiment API:\n"
        + "\n".join(missing_symbols)
    )


# ------------------------------------------------------------
# 6. Inspect reproducibility configuration
# ------------------------------------------------------------

RANDOM_SEED = getattr(
    config_module,
    "RANDOM_SEED",
    42
)

FINAL_CONFIG = final_experiment_module.FINAL_CONFIG
run_final_experiment = final_experiment_module.run_final_experiment

print()
print("Reproducibility:")
print(f"Random seed : {RANDOM_SEED}")

print()
print("FINAL_CONFIG:")
print(FINAL_CONFIG)


# ------------------------------------------------------------
# 7. Confirm writable results location
# ------------------------------------------------------------

RESULTS_DIR = WORK_PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

test_file = RESULTS_DIR / ".write_test"

try:
    test_file.write_text(
        "Q-MedAI Kaggle write test",
        encoding="utf-8"
    )

    assert test_file.exists()

finally:
    if test_file.exists():
        test_file.unlink()


print()
print(f"Writable results directory: {RESULTS_DIR}")


# ------------------------------------------------------------
# 8. Final status
# ------------------------------------------------------------

print()
print("=" * 70)
print("PROJECT COPY PASSED")
print("PROJECT IMPORT PASSED")
print("RESULTS DIRECTORY WRITE TEST PASSED")
print("=" * 70)

print()
print("Execution project:")
print(WORK_PROJECT_ROOT)

print()
print("Dataset:")
print(DATASET_PATH)

Q-MedAI — Project Source Setup
Copying project:
  FROM: /kaggle/input/datasets/adityagp7/q-medai-project-source/Q-MedAI
  TO:   /kaggle/working/Q-MedAI

Project structure check:
FOUND    src
FOUND    src/__init__.py
FOUND    src/config.py
FOUND    src/final_experiment.py
FOUND    src/models
FOUND    src/preprocessing
FOUND    src/evaluation

Current working directory:
/kaggle/working/Q-MedAI

sys.path[0]:
/kaggle/working/Q-MedAI

Core import checks:
src package              : /kaggle/input/datasets/adityagp7/q-medai-project-source/Q-MedAI/src/__init__.py
src.config               : /kaggle/input/datasets/adityagp7/q-medai-project-source/Q-MedAI/src/config.py
src.final_experiment     : /kaggle/input/datasets/adityagp7/q-medai-project-source/Q-MedAI/src/final_experiment.py

Final experiment API:
FOUND    src.final_experiment.FINAL_CONFIG
FOUND    src.final_experiment.run_final_experiment

Reproducibility:
Random seed : 42

FINAL_CONFIG:
ExperimentConfig(reduction_method='SelectKBest', n_f

In [36]:
# ============================================================
# CELL 3B — FORCE Q-MedAI IMPORTS FROM WRITABLE WORKING COPY
# ============================================================

from pathlib import Path
import sys
import os
import importlib

print("=" * 70)
print("Q-MedAI — Fix Python Module Source")
print("=" * 70)

WORK_PROJECT_ROOT = Path("/kaggle/working/Q-MedAI")

assert WORK_PROJECT_ROOT.exists(), (
    f"Working project does not exist: {WORK_PROJECT_ROOT}"
)

# ------------------------------------------------------------
# 1. Change working directory
# ------------------------------------------------------------

os.chdir(WORK_PROJECT_ROOT)

# ------------------------------------------------------------
# 2. Put writable project first on sys.path
# ------------------------------------------------------------

working_path = str(WORK_PROJECT_ROOT)

while working_path in sys.path:
    sys.path.remove(working_path)

sys.path.insert(0, working_path)

# ------------------------------------------------------------
# 3. IMPORTANT:
#    Remove previously cached src modules.
#
#    Python caches imported modules in sys.modules.
#    Merely changing sys.path does NOT change an already-imported src.
# ------------------------------------------------------------

cached_src_modules = [
    name
    for name in list(sys.modules)
    if name == "src" or name.startswith("src.")
]

print(f"Removing {len(cached_src_modules)} cached src modules...")

for module_name in cached_src_modules:
    del sys.modules[module_name]

importlib.invalidate_caches()

# ------------------------------------------------------------
# 4. Import again
# ------------------------------------------------------------

import src
import src.config as config_module
import src.final_experiment as final_experiment_module

# ------------------------------------------------------------
# 5. Verify imports point to /kaggle/working
# ------------------------------------------------------------

print()
print("Resolved imports:")
print(f"src                  : {src.__file__}")
print(f"src.config           : {config_module.__file__}")
print(
    "src.final_experiment:"
    f" {final_experiment_module.__file__}"
)

expected_root = str(WORK_PROJECT_ROOT.resolve())

module_paths = [
    src.__file__,
    config_module.__file__,
    final_experiment_module.__file__,
]

for module_path in module_paths:
    resolved = str(Path(module_path).resolve())

    assert resolved.startswith(expected_root), (
        "\nWRONG MODULE SOURCE DETECTED\n"
        f"Expected module under:\n  {expected_root}\n"
        f"Actually loaded:\n  {resolved}"
    )

# ------------------------------------------------------------
# 6. Re-bind important objects
# ------------------------------------------------------------

RANDOM_SEED = getattr(
    config_module,
    "RANDOM_SEED",
    42,
)

FINAL_CONFIG = final_experiment_module.FINAL_CONFIG
run_final_experiment = (
    final_experiment_module.run_final_experiment
)

# ------------------------------------------------------------
# 7. Final check
# ------------------------------------------------------------

print()
print(f"Current directory : {Path.cwd()}")
print(f"Random seed       : {RANDOM_SEED}")
print(f"FINAL_CONFIG      : {FINAL_CONFIG}")

print()
print("=" * 70)
print("WRITABLE PROJECT IMPORT PASSED")
print("=" * 70)

Q-MedAI — Fix Python Module Source
Removing 16 cached src modules...

Resolved imports:
src                  : /kaggle/working/Q-MedAI/src/__init__.py
src.config           : /kaggle/working/Q-MedAI/src/config.py
src.final_experiment: /kaggle/working/Q-MedAI/src/final_experiment.py

Current directory : /kaggle/working/Q-MedAI
Random seed       : 42
FINAL_CONFIG      : ExperimentConfig(reduction_method='SelectKBest', n_features=4, test_size=0.2, vqc_layers=3, vqc_iterations=100, kernel_subset_size=150, quantum_timeout_seconds=600, random_seed=42)

WRITABLE PROJECT IMPORT PASSED


In [37]:
# ============================================================
# CELL 4 — DATASET INSPECTION + VALIDATION
# Q-MedAI | SIH 2026 | PS 26139
# ============================================================

from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np

print("=" * 70)
print("Q-MedAI — Dataset Inspection & Validation")
print("=" * 70)

# DATASET_PATH comes from Cell 1
assert DATASET_PATH is not None
assert Path(DATASET_PATH).exists(), f"Dataset not found: {DATASET_PATH}"

# ------------------------------------------------------------
# 1. Load RAW CSV exactly as supplied
# ------------------------------------------------------------

raw_df = pd.read_csv(DATASET_PATH)

print()
print("Dataset source:")
print(f"local CSV ({DATASET_PATH})")

print()
print("Raw shape:")
print(raw_df.shape)

print()
print("Raw columns:")
for i, column in enumerate(raw_df.columns, start=1):
    print(f"{i:02d}. {column}")

print()
print("First 3 rows:")
display(raw_df.head(3))


# ------------------------------------------------------------
# 2. Basic integrity checks
# ------------------------------------------------------------

assert len(raw_df) > 0, "Dataset is empty."
assert raw_df.columns.is_unique, "Duplicate column names detected."

duplicate_rows = int(raw_df.duplicated().sum())

print()
print(f"Duplicate rows: {duplicate_rows}")


# ------------------------------------------------------------
# 3. Detect target column
# ------------------------------------------------------------

TARGET_CANDIDATES = [
    "diagnosis",
    "target",
    "label",
    "class",
    "outcome",
]

lower_to_original = {
    str(col).strip().lower(): col
    for col in raw_df.columns
}

target_column = None

for candidate in TARGET_CANDIDATES:
    if candidate in lower_to_original:
        target_column = lower_to_original[candidate]
        break

if target_column is None:
    raise ValueError(
        "Could not automatically detect a target column.\n"
        f"Available columns: {list(raw_df.columns)}"
    )

print()
print(f"Detected target column: {target_column}")


# ------------------------------------------------------------
# 4. Inspect original target values
# ------------------------------------------------------------

target_values = raw_df[target_column]

print()
print("Original target values:")
print(target_values.value_counts(dropna=False))


if target_values.isna().any():
    raise ValueError(
        f"Target column '{target_column}' contains missing values."
    )


# ------------------------------------------------------------
# 5. Encode target
#
# Wisconsin Kaggle/UCI convention:
# B = benign    -> 0
# M = malignant -> 1
# ------------------------------------------------------------

unique_target_values = set(
    target_values.astype(str).str.strip().str.upper().unique()
)

if unique_target_values == {"B", "M"}:

    TARGET_ENCODING = {
        "B": 0,
        "M": 1,
    }

    y = (
        target_values
        .astype(str)
        .str.strip()
        .str.upper()
        .map(TARGET_ENCODING)
        .astype(int)
    )

elif unique_target_values.issubset({"0", "1"}):

    TARGET_ENCODING = {
        "0": 0,
        "1": 1,
    }

    y = (
        target_values
        .astype(str)
        .str.strip()
        .map(TARGET_ENCODING)
        .astype(int)
    )

else:
    raise ValueError(
        "Unsupported target encoding.\n"
        f"Observed values: {sorted(unique_target_values)}"
    )


assert set(y.unique()) == {0, 1}

print()
print("Target encoding:")
print(TARGET_ENCODING)

print()
print("Encoded class distribution:")
print(y.value_counts().sort_index())

print()
print("Encoded class percentages:")
print(
    (y.value_counts(normalize=True).sort_index() * 100)
    .round(2)
    .astype(str)
    + "%"
)


# ------------------------------------------------------------
# 6. Detect obvious non-feature columns
# ------------------------------------------------------------

columns_to_remove = []

removal_reasons = {}

for column in raw_df.columns:

    normalized = str(column).strip().lower()

    if column == target_column:
        continue

    if (
        normalized == "id"
        or normalized.endswith("_id")
        or normalized.startswith("unnamed")
        or normalized in {"index", "row", "row_id"}
    ):
        columns_to_remove.append(column)

        if normalized.startswith("unnamed"):
            removal_reasons[column] = (
                "Unnamed/index artifact; not a predictive biomarker."
            )

        elif "id" in normalized:
            removal_reasons[column] = (
                "Identifier column; not a predictive biomarker."
            )

        else:
            removal_reasons[column] = (
                "Index/row identifier; not a predictive biomarker."
            )


print()
print("Columns marked for removal:")

if columns_to_remove:
    for column in columns_to_remove:
        print(
            f"- {column}: "
            f"{removal_reasons[column]}"
        )
else:
    print("None")


# ------------------------------------------------------------
# 7. Build candidate feature matrix
# ------------------------------------------------------------

feature_columns = [
    column
    for column in raw_df.columns
    if column != target_column
    and column not in columns_to_remove
]

X_raw = raw_df[feature_columns].copy()

print()
print(f"Candidate feature count: {X_raw.shape[1]}")


# ------------------------------------------------------------
# 8. Check feature types
# ------------------------------------------------------------

non_numeric_columns = [
    column
    for column in X_raw.columns
    if not pd.api.types.is_numeric_dtype(X_raw[column])
]

if non_numeric_columns:

    print()
    print("Unexpected non-numeric feature columns:")

    for column in non_numeric_columns:
        print(
            f"- {column}: "
            f"{X_raw[column].dtype}"
        )

    raise ValueError(
        "Unexpected non-numeric feature columns found. "
        "They will NOT be silently removed."
    )


# ------------------------------------------------------------
# 9. Missing / infinite value inspection
# ------------------------------------------------------------

missing_by_column = X_raw.isna().sum()
missing_by_column = missing_by_column[
    missing_by_column > 0
]

print()
print(
    f"Total missing feature values: "
    f"{int(X_raw.isna().sum().sum())}"
)

if len(missing_by_column):
    print()
    print("Missing values by feature:")
    print(missing_by_column.sort_values(ascending=False))


numeric_array = X_raw.to_numpy(dtype=float)

infinite_count = int(
    np.isinf(numeric_array).sum()
)

print(f"Infinite feature values: {infinite_count}")

if infinite_count > 0:
    raise ValueError(
        "Infinite values detected in feature matrix."
    )


# ------------------------------------------------------------
# 10. Detect constant features
# ------------------------------------------------------------

constant_features = [
    column
    for column in X_raw.columns
    if X_raw[column].nunique(dropna=False) <= 1
]

print()
print(f"Constant features: {len(constant_features)}")

if constant_features:
    for column in constant_features:
        print(f"- {column}")


# ------------------------------------------------------------
# 11. Final validation summary
# ------------------------------------------------------------

DATASET_REPORT = {
    "source": f"local CSV ({DATASET_PATH})",
    "samples": int(len(raw_df)),
    "raw_columns": int(raw_df.shape[1]),
    "feature_count": int(X_raw.shape[1]),
    "target_column": str(target_column),
    "target_encoding": TARGET_ENCODING,
    "class_distribution": {
        int(k): int(v)
        for k, v in y.value_counts().sort_index().items()
    },
    "removed_columns": removal_reasons,
    "missing_feature_values": int(
        X_raw.isna().sum().sum()
    ),
    "duplicate_rows": duplicate_rows,
}

print()
print("=" * 70)
print("DATASET VALIDATION PASSED")
print("=" * 70)

print(f"Samples             : {DATASET_REPORT['samples']}")
print(f"Usable features     : {DATASET_REPORT['feature_count']}")
print(f"Target column       : {DATASET_REPORT['target_column']}")
print(f"Target encoding     : {DATASET_REPORT['target_encoding']}")
print(f"Class distribution  : {DATASET_REPORT['class_distribution']}")
print(f"Removed columns     : {list(removal_reasons.keys())}")
print(
    f"Missing feature vals: "
    f"{DATASET_REPORT['missing_feature_values']}"
)

print()
print("IMPORTANT:")
print(
    "No preprocessing has been fitted yet. "
    "The train/test split will happen BEFORE "
    "imputation, scaling, or feature selection."
)

Q-MedAI — Dataset Inspection & Validation

Dataset source:
local CSV (/kaggle/input/datasets/adityagp7/q-medai-breast-cancer-dataset/breast_cancer.csv)

Raw shape:
(569, 33)

Raw columns:
01. id
02. diagnosis
03. radius_mean
04. texture_mean
05. perimeter_mean
06. area_mean
07. smoothness_mean
08. compactness_mean
09. concavity_mean
10. concave points_mean
11. symmetry_mean
12. fractal_dimension_mean
13. radius_se
14. texture_se
15. perimeter_se
16. area_se
17. smoothness_se
18. compactness_se
19. concavity_se
20. concave points_se
21. symmetry_se
22. fractal_dimension_se
23. radius_worst
24. texture_worst
25. perimeter_worst
26. area_worst
27. smoothness_worst
28. compactness_worst
29. concavity_worst
30. concave points_worst
31. symmetry_worst
32. fractal_dimension_worst
33. Unnamed: 32

First 3 rows:


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN



Duplicate rows: 0

Detected target column: diagnosis

Original target values:
diagnosis
B    357
M    212
Name: count, dtype: int64

Target encoding:
{'B': 0, 'M': 1}

Encoded class distribution:
diagnosis
0    357
1    212
Name: count, dtype: int64

Encoded class percentages:
diagnosis
0    62.74%
1    37.26%
Name: proportion, dtype: object

Columns marked for removal:
- id: Identifier column; not a predictive biomarker.
- Unnamed: 32: Unnamed/index artifact; not a predictive biomarker.

Candidate feature count: 30

Total missing feature values: 0
Infinite feature values: 0

Constant features: 0

DATASET VALIDATION PASSED
Samples             : 569
Usable features     : 30
Target column       : diagnosis
Target encoding     : {'B': 0, 'M': 1}
Class distribution  : {0: 357, 1: 212}
Removed columns     : ['id', 'Unnamed: 32']
Missing feature vals: 0

IMPORTANT:
No preprocessing has been fitted yet. The train/test split will happen BEFORE imputation, scaling, or feature selection.


In [38]:
# ============================================================
# CELL 5 — STRATIFIED SPLIT + LEAKAGE-SAFE PREPROCESSING
# Q-MedAI | SIH 2026 | PS 26139
# ============================================================

from dataclasses import dataclass
from typing import Any

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA

print("=" * 70)
print("Q-MedAI — Leakage-Safe Split & Preprocessing")
print("=" * 70)

# ------------------------------------------------------------
# 1. Experiment configuration
# ------------------------------------------------------------

RANDOM_SEED = 42
TEST_SIZE = 0.20

# Keep SAFE until the complete notebook works once.
PROFILE = "SAFE"

PROFILE_CONFIGS = {
    "SAFE": {
        "feature_count": 4,
        "vqc_layers": 2,
        "vqc_iterations": 30,
        "matched_train_size": 100,
        "vqc_batch_size": 20,
        "quantum_timeout_seconds": 180,
    },
    "SIH": {
        "feature_count": 4,
        "vqc_layers": 3,
        "vqc_iterations": 100,
        "matched_train_size": 150,
        "vqc_batch_size": 24,
        "quantum_timeout_seconds": 600,
    },
}

assert PROFILE in PROFILE_CONFIGS

CFG = PROFILE_CONFIGS[PROFILE]

PREPROCESSING = "SelectKBest"

assert PREPROCESSING in {
    "SelectKBest",
    "PCA",
}

np.random.seed(RANDOM_SEED)

print(f"Profile            : {PROFILE}")
print(f"Preprocessing      : {PREPROCESSING}")
print(f"Feature count      : {CFG['feature_count']}")
print(f"Matched train size : {CFG['matched_train_size']}")
print(f"Random seed        : {RANDOM_SEED}")


# ------------------------------------------------------------
# 2. Perform the split BEFORE preprocessing
# ------------------------------------------------------------

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_SEED,
)

# Defensive copies
X_train_raw = X_train_raw.copy()
X_test_raw = X_test_raw.copy()

y_train = y_train.copy()
y_test = y_test.copy()


# ------------------------------------------------------------
# 3. Confirm split integrity
# ------------------------------------------------------------

assert len(X_train_raw) + len(X_test_raw) == len(X_raw)

assert set(X_train_raw.index).isdisjoint(
    set(X_test_raw.index)
), "Train/test overlap detected."


print()
print("Train/test split:")
print(f"Training rows : {len(X_train_raw)}")
print(f"Test rows     : {len(X_test_raw)}")

print()
print("Training class distribution:")
print(
    y_train.value_counts()
    .sort_index()
    .to_dict()
)

print()
print("Test class distribution:")
print(
    y_test.value_counts()
    .sort_index()
    .to_dict()
)


# ------------------------------------------------------------
# 4. Stratified matched subset helper
#
# Quantum models use fewer training examples because circuit
# simulation is computationally expensive.
#
# Classical models will ALSO be trained on the same matched
# subset later for a fair direct comparison.
# ------------------------------------------------------------

def stratified_subset(
    X: pd.DataFrame,
    y_local: pd.Series,
    n: int,
):
    n = min(
        int(n),
        len(X),
    )

    if n < 2:
        raise ValueError(
            "Matched training size must be at least 2."
        )

    if n == len(X):
        return (
            X.copy(),
            y_local.copy(),
        )

    X_selected, _, y_selected, _ = train_test_split(
        X,
        y_local,
        train_size=n,
        stratify=y_local,
        random_state=RANDOM_SEED,
    )

    return (
        X_selected.copy(),
        y_selected.copy(),
    )


# ------------------------------------------------------------
# 5. Preprocessor object
#
# Important:
# Every fitted object below is learned ONLY from X_fit.
# ------------------------------------------------------------

@dataclass
class FittedPreprocessor:

    imputer: Any
    scaler: Any
    reducer: Any
    angle_scaler: Any

    feature_names_out: list[str]
    method: str

    fit_row_count: int
    fit_indices: tuple

    def transform_reduced(
        self,
        X: pd.DataFrame,
    ) -> np.ndarray:
        """
        Impute -> standardize -> feature reduction.

        This representation is suitable for classical models.
        """

        arr = self.imputer.transform(X)

        arr = self.scaler.transform(arr)

        arr = self.reducer.transform(arr)

        return np.asarray(
            arr,
            dtype=float,
        )

    def transform_quantum(
        self,
        X: pd.DataFrame,
    ) -> np.ndarray:
        """
        Classical preprocessing followed by angle scaling
        into [-pi, pi] for quantum encoding.
        """

        reduced = self.transform_reduced(X)

        return np.asarray(
            self.angle_scaler.transform(reduced),
            dtype=float,
        )


# ------------------------------------------------------------
# 6. Fit preprocessing ONLY on supplied training rows
# ------------------------------------------------------------

def fit_preprocessor(
    X_fit: pd.DataFrame,
    y_fit: pd.Series,
    method: str,
    k: int,
):

    if len(X_fit) != len(y_fit):
        raise ValueError(
            "X_fit and y_fit row counts differ."
        )

    if k < 1:
        raise ValueError(
            "feature_count must be >= 1."
        )

    if k > X_fit.shape[1]:
        raise ValueError(
            f"Requested {k} features, but only "
            f"{X_fit.shape[1]} input features exist."
        )

    # ----------------------------------------
    # Median imputer
    # ----------------------------------------

    imputer = SimpleImputer(
        strategy="median"
    )

    X_imputed = imputer.fit_transform(
        X_fit
    )

    # ----------------------------------------
    # Standardization
    # ----------------------------------------

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        X_imputed
    )

    # ----------------------------------------
    # Feature reduction
    # ----------------------------------------

    if method == "SelectKBest":

        reducer = SelectKBest(
            score_func=f_classif,
            k=k,
        )

        X_reduced = reducer.fit_transform(
            X_scaled,
            y_fit,
        )

        selected_indices = (
            reducer
            .get_support(indices=True)
        )

        feature_names = [
            str(X_fit.columns[i])
            for i in selected_indices
        ]

    elif method == "PCA":

        reducer = PCA(
            n_components=k,
            random_state=RANDOM_SEED,
        )

        X_reduced = reducer.fit_transform(
            X_scaled
        )

        feature_names = [
            f"PC{i + 1}"
            for i in range(k)
        ]

    else:

        raise ValueError(
            "method must be "
            "'SelectKBest' or 'PCA'."
        )

    # ----------------------------------------
    # Quantum angle scaler
    #
    # Fit ONLY on reduced training data.
    # ----------------------------------------

    angle_scaler = MinMaxScaler(
        feature_range=(
            -np.pi,
            np.pi,
        ),
        clip=True,
    )

    angle_scaler.fit(
        X_reduced
    )

    fitted = FittedPreprocessor(
        imputer=imputer,
        scaler=scaler,
        reducer=reducer,
        angle_scaler=angle_scaler,
        feature_names_out=feature_names,
        method=method,
        fit_row_count=len(X_fit),
        fit_indices=tuple(X_fit.index),
    )

    return fitted


# ------------------------------------------------------------
# 7. FULL classical preprocessing
#
# Fitted only on the 455-row training partition.
# ------------------------------------------------------------

full_preprocessor = fit_preprocessor(
    X_train_raw,
    y_train,
    PREPROCESSING,
    CFG["feature_count"],
)

X_train_full = (
    full_preprocessor
    .transform_reduced(X_train_raw)
)

X_test_full = (
    full_preprocessor
    .transform_reduced(X_test_raw)
)


# ------------------------------------------------------------
# 8. Create matched raw training subset FIRST
# ------------------------------------------------------------

X_train_match_raw, y_train_match = (
    stratified_subset(
        X_train_raw,
        y_train,
        CFG["matched_train_size"],
    )
)


# ------------------------------------------------------------
# 9. Fit an entirely separate preprocessor
#    ONLY on the matched subset
# ------------------------------------------------------------

matched_preprocessor = fit_preprocessor(
    X_train_match_raw,
    y_train_match,
    PREPROCESSING,
    CFG["feature_count"],
)


# Classical matched representation
X_train_match_classical = (
    matched_preprocessor
    .transform_reduced(
        X_train_match_raw
    )
)

X_test_match_classical = (
    matched_preprocessor
    .transform_reduced(
        X_test_raw
    )
)


# Quantum angle representation
X_train_match_quantum = (
    matched_preprocessor
    .transform_quantum(
        X_train_match_raw
    )
)

X_test_match_quantum = (
    matched_preprocessor
    .transform_quantum(
        X_test_raw
    )
)


# ------------------------------------------------------------
# 10. Structural checks
# ------------------------------------------------------------

assert X_train_full.shape == (
    len(y_train),
    CFG["feature_count"],
)

assert X_test_full.shape == (
    len(y_test),
    CFG["feature_count"],
)

assert X_train_match_classical.shape == (
    len(y_train_match),
    CFG["feature_count"],
)

assert X_test_match_classical.shape == (
    len(y_test),
    CFG["feature_count"],
)

assert X_train_match_quantum.shape == (
    len(y_train_match),
    CFG["feature_count"],
)

assert X_test_match_quantum.shape == (
    len(y_test),
    CFG["feature_count"],
)


# Quantum angles should be bounded.
assert np.all(
    X_train_match_quantum
    <= np.pi + 1e-10
)

assert np.all(
    X_train_match_quantum
    >= -np.pi - 1e-10
)


# ------------------------------------------------------------
# 11. Confirm fitted row identities
# ------------------------------------------------------------

assert set(
    full_preprocessor.fit_indices
) == set(
    X_train_raw.index
)

assert set(
    matched_preprocessor.fit_indices
) == set(
    X_train_match_raw.index
)

assert set(
    matched_preprocessor.fit_indices
).isdisjoint(
    set(X_test_raw.index)
)


# ------------------------------------------------------------
# 12. Report
# ------------------------------------------------------------

print()
print("=" * 70)
print("PREPROCESSING COMPLETE")
print("=" * 70)

print()
print("Full-data condition")
print("-------------------")

print(
    f"Fit rows       : "
    f"{full_preprocessor.fit_row_count}"
)

print(
    f"Train shape    : "
    f"{X_train_full.shape}"
)

print(
    f"Test shape     : "
    f"{X_test_full.shape}"
)

print(
    "Selected features/components:"
)

for feature in (
    full_preprocessor
    .feature_names_out
):
    print(f"  - {feature}")


print()
print("Matched-data condition")
print("----------------------")

print(
    f"Fit rows       : "
    f"{matched_preprocessor.fit_row_count}"
)

print(
    "Class balance  :",
    y_train_match
    .value_counts()
    .sort_index()
    .to_dict(),
)

print(
    f"Classical train: "
    f"{X_train_match_classical.shape}"
)

print(
    f"Quantum train  : "
    f"{X_train_match_quantum.shape}"
)

print(
    f"Held-out test  : "
    f"{X_test_match_quantum.shape}"
)

print(
    "Selected features/components:"
)

for feature in (
    matched_preprocessor
    .feature_names_out
):
    print(f"  - {feature}")


print()
print("=" * 70)
print("STRATIFIED SPLIT PASSED")
print("TRAIN-ONLY PREPROCESSING PASSED")
print("MATCHED PREPROCESSING PASSED")
print("=" * 70)

Q-MedAI — Leakage-Safe Split & Preprocessing
Profile            : SAFE
Preprocessing      : SelectKBest
Feature count      : 4
Matched train size : 100
Random seed        : 42

Train/test split:
Training rows : 455
Test rows     : 114

Training class distribution:
{0: 285, 1: 170}

Test class distribution:
{0: 72, 1: 42}

PREPROCESSING COMPLETE

Full-data condition
-------------------
Fit rows       : 455
Train shape    : (455, 4)
Test shape     : (114, 4)
Selected features/components:
  - concave points_mean
  - radius_worst
  - perimeter_worst
  - concave points_worst

Matched-data condition
----------------------
Fit rows       : 100
Class balance  : {0: 63, 1: 37}
Classical train: (100, 4)
Quantum train  : (100, 4)
Held-out test  : (114, 4)
Selected features/components:
  - concave points_mean
  - radius_worst
  - perimeter_worst
  - concave points_worst

STRATIFIED SPLIT PASSED
TRAIN-ONLY PREPROCESSING PASSED
MATCHED PREPROCESSING PASSED


In [39]:
# ============================================================
# CELL 6 — EXPLICIT DATA-LEAKAGE SAFETY TEST
# Q-MedAI | SIH 2026 | PS 26139
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
import numpy as np
import pandas as pd

print("=" * 70)
print("Q-MedAI — Explicit Data Leakage Safety Test")
print("=" * 70)


# ------------------------------------------------------------
# 1. Verify training/test index separation
# ------------------------------------------------------------

train_indices = set(X_train_raw.index)
test_indices = set(X_test_raw.index)

assert train_indices.isdisjoint(test_indices)

print()
print("Index isolation:")
print(f"Training rows : {len(train_indices)}")
print(f"Test rows     : {len(test_indices)}")
print("Overlap       : 0")


# ------------------------------------------------------------
# 2. Verify full preprocessor was fitted only on training rows
# ------------------------------------------------------------

assert set(
    full_preprocessor.fit_indices
) == train_indices

assert set(
    full_preprocessor.fit_indices
).isdisjoint(test_indices)

print()
print("Full preprocessor fit audit:")
print(
    f"Rows seen during fit : "
    f"{full_preprocessor.fit_row_count}"
)
print(
    "Test rows seen during fit : 0"
)


# ------------------------------------------------------------
# 3. Verify matched preprocessor was fitted ONLY on
#    the 100-row matched training subset
# ------------------------------------------------------------

matched_indices = set(
    X_train_match_raw.index
)

assert set(
    matched_preprocessor.fit_indices
) == matched_indices

assert matched_indices.issubset(
    train_indices
)

assert matched_indices.isdisjoint(
    test_indices
)

print()
print("Matched preprocessor fit audit:")
print(
    f"Rows seen during fit : "
    f"{matched_preprocessor.fit_row_count}"
)
print(
    "Test rows seen during fit : 0"
)


# ------------------------------------------------------------
# 4. Record fitted scaler statistics BEFORE transforming test
# ------------------------------------------------------------

full_mean_before = (
    full_preprocessor
    .scaler
    .mean_
    .copy()
)

full_scale_before = (
    full_preprocessor
    .scaler
    .scale_
    .copy()
)

full_selector_scores_before = (
    full_preprocessor
    .reducer
    .scores_
    .copy()
)

matched_mean_before = (
    matched_preprocessor
    .scaler
    .mean_
    .copy()
)

matched_scale_before = (
    matched_preprocessor
    .scaler
    .scale_
    .copy()
)

matched_selector_scores_before = (
    matched_preprocessor
    .reducer
    .scores_
    .copy()
)


# ------------------------------------------------------------
# 5. Transform held-out test data AGAIN
#
# If transform() incorrectly refitted preprocessing,
# the fitted statistics would change here.
# ------------------------------------------------------------

_ = full_preprocessor.transform_reduced(
    X_test_raw
)

_ = matched_preprocessor.transform_reduced(
    X_test_raw
)

_ = matched_preprocessor.transform_quantum(
    X_test_raw
)


# ------------------------------------------------------------
# 6. Verify fitted parameters DID NOT CHANGE
# ------------------------------------------------------------

assert np.array_equal(
    full_mean_before,
    full_preprocessor.scaler.mean_,
)

assert np.array_equal(
    full_scale_before,
    full_preprocessor.scaler.scale_,
)

assert np.array_equal(
    full_selector_scores_before,
    full_preprocessor.reducer.scores_,
)

assert np.array_equal(
    matched_mean_before,
    matched_preprocessor.scaler.mean_,
)

assert np.array_equal(
    matched_scale_before,
    matched_preprocessor.scaler.scale_,
)

assert np.array_equal(
    matched_selector_scores_before,
    matched_preprocessor.reducer.scores_,
)

print()
print("Transform-only audit:")
print(
    "Scaler means unchanged after test transformation : PASS"
)
print(
    "Scaler scales unchanged after test transformation: PASS"
)
print(
    "SelectKBest scores unchanged                   : PASS"
)


# ------------------------------------------------------------
# 7. Controlled comparison:
#
# Create an intentionally WRONG preprocessing pipeline
# fitted on train + test.
#
# This is NOT used for modeling.
# It exists only to demonstrate why fitting before splitting
# would cause leakage.
# ------------------------------------------------------------

X_combined_wrong = pd.concat(
    [
        X_train_raw,
        X_test_raw,
    ],
    axis=0,
)

y_combined_wrong = pd.concat(
    [
        y_train,
        y_test,
    ],
    axis=0,
)


wrong_imputer = SimpleImputer(
    strategy="median"
)

wrong_imputed = (
    wrong_imputer
    .fit_transform(
        X_combined_wrong
    )
)

wrong_scaler = StandardScaler()

wrong_scaled = (
    wrong_scaler
    .fit_transform(
        wrong_imputed
    )
)

wrong_selector = SelectKBest(
    score_func=f_classif,
    k=CFG["feature_count"],
)

wrong_selector.fit(
    wrong_scaled,
    y_combined_wrong,
)


# ------------------------------------------------------------
# 8. Demonstrate the difference
# ------------------------------------------------------------

correct_scaler_mean = (
    full_preprocessor
    .scaler
    .mean_
)

wrong_scaler_mean = (
    wrong_scaler
    .mean_
)

mean_difference = np.abs(
    correct_scaler_mean
    - wrong_scaler_mean
)

max_mean_difference = float(
    np.max(mean_difference)
)

correct_scores = (
    full_preprocessor
    .reducer
    .scores_
)

wrong_scores = (
    wrong_selector
    .scores_
)

score_difference = np.abs(
    correct_scores
    - wrong_scores
)

max_score_difference = float(
    np.max(score_difference)
)


print()
print("Controlled leakage demonstration:")
print(
    "Correct scaler fit rows :",
    len(X_train_raw),
)

print(
    "Wrong scaler fit rows   :",
    len(X_combined_wrong),
)

print(
    "Max scaler-mean difference:",
    f"{max_mean_difference:.12f}",
)

print(
    "Max SelectKBest-score difference:",
    f"{max_score_difference:.12f}",
)


# ------------------------------------------------------------
# 9. Prove that leaked preprocessing actually differs
# ------------------------------------------------------------

assert max_mean_difference > 0.0, (
    "Unexpected: leaked and correct scaler statistics "
    "were numerically identical."
)

assert max_score_difference > 0.0, (
    "Unexpected: leaked and correct SelectKBest scores "
    "were numerically identical."
)


# ------------------------------------------------------------
# 10. Selected-feature comparison
# ------------------------------------------------------------

correct_selected = list(
    full_preprocessor
    .feature_names_out
)

wrong_selected_indices = (
    wrong_selector
    .get_support(indices=True)
)

wrong_selected = [
    str(X_train_raw.columns[i])
    for i in wrong_selected_indices
]

print()
print("Correct train-only selected features:")

for name in correct_selected:
    print(f"  - {name}")

print()
print("Leaked train+test selected features:")

for name in wrong_selected:
    print(f"  - {name}")

if correct_selected == wrong_selected:
    print()
    print(
        "NOTE: The selected feature names happened to remain "
        "the same, but their fitted statistics/scores changed. "
        "This is still data leakage."
    )

else:
    print()
    print(
        "The leaked preprocessing even changed which features "
        "were selected."
    )


# ------------------------------------------------------------
# 11. Quantum preprocessing audit
# ------------------------------------------------------------

quantum_train_min = float(
    np.min(
        X_train_match_quantum
    )
)

quantum_train_max = float(
    np.max(
        X_train_match_quantum
    )
)

quantum_test_min = float(
    np.min(
        X_test_match_quantum
    )
)

quantum_test_max = float(
    np.max(
        X_test_match_quantum
    )
)

print()
print("Quantum angle ranges:")
print(
    f"Training: "
    f"[{quantum_train_min:.6f}, "
    f"{quantum_train_max:.6f}]"
)

print(
    f"Test    : "
    f"[{quantum_test_min:.6f}, "
    f"{quantum_test_max:.6f}]"
)

assert quantum_train_min >= (
    -np.pi - 1e-10
)

assert quantum_train_max <= (
    np.pi + 1e-10
)

assert quantum_test_min >= (
    -np.pi - 1e-10
)

assert quantum_test_max <= (
    np.pi + 1e-10
)


# ------------------------------------------------------------
# 12. Final status
# ------------------------------------------------------------

print()
print("=" * 70)
print("DATA LEAKAGE SAFETY TEST PASSED")
print("=" * 70)

print(
    "Train/test split occurred BEFORE preprocessing."
)

print(
    "Imputer, scaler, SelectKBest, and quantum angle "
    "scaler were fitted using training data only."
)

print(
    "Held-out test transformation did not refit "
    "any preprocessing component."
)

Q-MedAI — Explicit Data Leakage Safety Test

Index isolation:
Training rows : 455
Test rows     : 114
Overlap       : 0

Full preprocessor fit audit:
Rows seen during fit : 455
Test rows seen during fit : 0

Matched preprocessor fit audit:
Rows seen during fit : 100
Test rows seen during fit : 0

Transform-only audit:
Scaler means unchanged after test transformation : PASS
Scaler scales unchanged after test transformation: PASS
SelectKBest scores unchanged                   : PASS

Controlled leakage demonstration:
Correct scaler fit rows : 455
Wrong scaler fit rows   : 569
Max scaler-mean difference: 9.986102473976
Max SelectKBest-score difference: 230.660460877756

Correct train-only selected features:
  - concave points_mean
  - radius_worst
  - perimeter_worst
  - concave points_worst

Leaked train+test selected features:
  - concave points_mean
  - radius_worst
  - perimeter_worst
  - concave points_worst

NOTE: The selected feature names happened to remain the same, but their fit

In [41]:
# ============================================================
# CELL 7 — CLASSICAL BASELINES + EVALUATION FRAMEWORK
# Q-MedAI | SIH 2026 | PS 26139
# ============================================================

import time
import warnings

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
)

print("=" * 70)
print("Q-MedAI — Classical Model Benchmark")
print("=" * 70)


# ------------------------------------------------------------
# 1. Metric helper
# ------------------------------------------------------------

def compute_binary_metrics(
    y_true,
    y_pred,
    y_score,
):
    """
    y_score MUST be continuous:
      - predict_proba[:, 1]
      - decision_function
      - quantum probability-like score

    Hard labels are never used for ROC-AUC.
    """

    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_score = np.asarray(y_score, dtype=float)

    if len(y_true) != len(y_pred):
        raise ValueError(
            "y_true and y_pred lengths differ."
        )

    if len(y_true) != len(y_score):
        raise ValueError(
            "y_true and y_score lengths differ."
        )

    if not np.all(np.isfinite(y_score)):
        raise ValueError(
            "Continuous prediction scores contain "
            "NaN or infinite values."
        )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    tn, fp, fn, tp = cm.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    fpr, tpr, thresholds = roc_curve(
        y_true,
        y_score,
    )

    metrics = {
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "precision": float(
            precision_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "recall_sensitivity": float(
            sensitivity
        ),
        "specificity": float(
            specificity
        ),
        "f1": float(
            f1_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                y_score,
            )
        ),
    }

    return {
        "metrics": metrics,
        "confusion_matrix": cm,
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
    }


# ------------------------------------------------------------
# 2. Model factory
# ------------------------------------------------------------

def make_classical_models():
    """
    Return NEW model instances each time.

    We do not reuse already-fitted estimators between
    full-data and matched-data conditions.
    """

    return {
        "Logistic Regression": LogisticRegression(
            max_iter=5000,
            solver="lbfgs",
            random_state=RANDOM_SEED,
        ),

        "RBF SVM": SVC(
            kernel="rbf",
            probability=True,
            random_state=RANDOM_SEED,
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        ),
    }


# ------------------------------------------------------------
# 3. Single classical training/evaluation function
# ------------------------------------------------------------

def run_classical_model(
    model_name,
    model,
    X_train,
    y_train_local,
    X_test,
    y_test_local,
):
    result = {
        "model": model_name,
        "status": "NOT TRAINED",
        "error": None,
        "training_seconds": None,
        "metrics": None,
        "confusion_matrix": None,
        "fpr": None,
        "tpr": None,
        "thresholds": None,
        "y_pred": None,
        "y_score": None,
        "estimator": None,
    }

    try:
        result["status"] = "TRAINING"

        start = time.perf_counter()

        model.fit(
            X_train,
            y_train_local,
        )

        training_seconds = (
            time.perf_counter() - start
        )

        y_pred = model.predict(
            X_test
        )

        # ----------------------------------------
        # Continuous ROC-AUC scores
        # ----------------------------------------

        if hasattr(model, "predict_proba"):

            probabilities = model.predict_proba(
                X_test
            )

            if probabilities.ndim != 2:
                raise ValueError(
                    "predict_proba did not return "
                    "a 2-D matrix."
                )

            positive_index = list(
                model.classes_
            ).index(1)

            y_score = probabilities[
                :,
                positive_index,
            ]

        elif hasattr(
            model,
            "decision_function",
        ):

            y_score = model.decision_function(
                X_test
            )

        else:
            raise RuntimeError(
                f"{model_name} exposes neither "
                "predict_proba nor decision_function."
            )

        evaluation = compute_binary_metrics(
            y_test_local,
            y_pred,
            y_score,
        )

        result.update(
            {
                "status": "COMPLETED",
                "training_seconds": float(
                    training_seconds
                ),
                "metrics": evaluation["metrics"],
                "confusion_matrix": evaluation[
                    "confusion_matrix"
                ],
                "fpr": evaluation["fpr"],
                "tpr": evaluation["tpr"],
                "thresholds": evaluation[
                    "thresholds"
                ],
                "y_pred": np.asarray(
                    y_pred,
                    dtype=int,
                ),
                "y_score": np.asarray(
                    y_score,
                    dtype=float,
                ),
                "estimator": model,
            }
        )

    except Exception as exc:

        result["status"] = "FAILED"
        result["error"] = (
            f"{type(exc).__name__}: {exc}"
        )

    return result


# ------------------------------------------------------------
# 4. Run a condition
# ------------------------------------------------------------

def run_classical_condition(
    condition_name,
    X_train,
    y_train_local,
    X_test,
    y_test_local,
):
    print()
    print("-" * 70)
    print(condition_name)
    print("-" * 70)

    models = make_classical_models()

    results = {}

    for model_name, model in models.items():

        print(
            f"Training {model_name}...",
            end=" ",
        )

        result = run_classical_model(
            model_name=model_name,
            model=model,
            X_train=X_train,
            y_train_local=y_train_local,
            X_test=X_test,
            y_test_local=y_test_local,
        )

        results[model_name] = result

        if result["status"] == "COMPLETED":

            m = result["metrics"]

            print(
                "COMPLETED "
                f"| accuracy={m['accuracy']:.4f} "
                f"| f1={m['f1']:.4f} "
                f"| auc={m['roc_auc']:.4f} "
                f"| {result['training_seconds']:.4f}s"
            )

        else:

            print(
                f"{result['status']} "
                f"| {result['error']}"
            )

    return results


# ------------------------------------------------------------
# 5. FULL-DATA classical benchmark
# ------------------------------------------------------------

classical_full_results = run_classical_condition(
    condition_name=(
        "FULL-DATA CLASSICAL BASELINE "
        f"({len(y_train)} training rows)"
    ),
    X_train=X_train_full,
    y_train_local=y_train,
    X_test=X_test_full,
    y_test_local=y_test,
)


# ------------------------------------------------------------
# 6. MATCHED-DATA classical benchmark
#
# IMPORTANT:
# This is the fair comparison condition for VQC/QSVM.
# Classical and quantum models use the same 100 raw
# training observations.
# ------------------------------------------------------------

classical_matched_results = run_classical_condition(
    condition_name=(
        "MATCHED-DATA CLASSICAL BASELINE "
        f"({len(y_train_match)} training rows)"
    ),
    X_train=X_train_match_classical,
    y_train_local=y_train_match,
    X_test=X_test_match_classical,
    y_test_local=y_test,
)


# ------------------------------------------------------------
# 7. Build summary table
# ------------------------------------------------------------

def results_to_dataframe(
    results,
    condition,
    training_rows,
):

    rows = []

    for model_name, result in results.items():

        row = {
            "Condition": condition,
            "Model": model_name,
            "Status": result["status"],
            "Training Rows": int(training_rows),
            "Training Time (s)": result[
                "training_seconds"
            ],
            "Accuracy": np.nan,
            "Precision": np.nan,
            "Sensitivity": np.nan,
            "Specificity": np.nan,
            "F1": np.nan,
            "ROC-AUC": np.nan,
            "Error": result["error"],
        }

        if result["metrics"] is not None:

            m = result["metrics"]

            row.update(
                {
                    "Accuracy": m[
                        "accuracy"
                    ],
                    "Precision": m[
                        "precision"
                    ],
                    "Sensitivity": m[
                        "recall_sensitivity"
                    ],
                    "Specificity": m[
                        "specificity"
                    ],
                    "F1": m["f1"],
                    "ROC-AUC": m[
                        "roc_auc"
                    ],
                }
            )

        rows.append(row)

    return pd.DataFrame(rows)


full_classical_table = (
    results_to_dataframe(
        classical_full_results,
        condition="FULL DATA",
        training_rows=len(y_train),
    )
)

matched_classical_table = (
    results_to_dataframe(
        classical_matched_results,
        condition="MATCHED DATA",
        training_rows=len(
            y_train_match
        ),
    )
)

classical_comparison_table = pd.concat(
    [
        full_classical_table,
        matched_classical_table,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 8. Verification assertions
# ------------------------------------------------------------

for condition_results in [
    classical_full_results,
    classical_matched_results,
]:

    for model_name, result in (
        condition_results.items()
    ):

        assert result["status"] == "COMPLETED", (
            f"{model_name} did not complete: "
            f"{result['error']}"
        )

        assert result["metrics"] is not None

        assert (
            result["confusion_matrix"].shape
            == (2, 2)
        )

        assert len(result["y_score"]) == len(
            y_test
        )

        # Explicitly prove scores aren't simply
        # hard predictions.
        unique_scores = np.unique(
            np.round(
                result["y_score"],
                12,
            )
        )

        assert len(unique_scores) > 2, (
            f"{model_name} appears to use hard "
            "labels rather than continuous scores."
        )

        for metric_value in (
            result["metrics"].values()
        ):
            assert np.isfinite(
                metric_value
            )


# ------------------------------------------------------------
# 9. Display clean table
# ------------------------------------------------------------

display_columns = [
    "Condition",
    "Model",
    "Status",
    "Training Rows",
    "Accuracy",
    "Precision",
    "Sensitivity",
    "Specificity",
    "F1",
    "ROC-AUC",
    "Training Time (s)",
]

display_table = (
    classical_comparison_table[
        display_columns
    ]
    .copy()
)

numeric_columns = [
    "Accuracy",
    "Precision",
    "Sensitivity",
    "Specificity",
    "F1",
    "ROC-AUC",
    "Training Time (s)",
]

display_table[
    numeric_columns
] = display_table[
    numeric_columns
].round(4)

print()
print("=" * 70)
print("CLASSICAL BENCHMARK RESULTS")
print("=" * 70)

display(display_table)


# ------------------------------------------------------------
# 10. Save classical results
# ------------------------------------------------------------

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

classical_csv_path = (
    RESULTS_DIR
    / "classical_benchmark.csv"
)

classical_comparison_table.to_csv(
    classical_csv_path,
    index=False,
)

print()
print(
    "Saved classical benchmark:"
)
print(classical_csv_path)


# ------------------------------------------------------------
# 11. Final status
# ------------------------------------------------------------

print()
print("=" * 70)
print("FULL-DATA CLASSICAL BENCHMARK PASSED")
print("MATCHED-DATA CLASSICAL BENCHMARK PASSED")
print("CONTINUOUS ROC-AUC SCORE CHECK PASSED")
print("=" * 70)

Q-MedAI — Classical Model Benchmark

----------------------------------------------------------------------
FULL-DATA CLASSICAL BASELINE (455 training rows)
----------------------------------------------------------------------
Training Logistic Regression... COMPLETED | accuracy=0.9561 | f1=0.9398 | auc=0.9954 | 0.0059s
Training RBF SVM... COMPLETED | accuracy=0.9386 | f1=0.9114 | auc=0.9954 | 0.0146s
Training Random Forest... COMPLETED | accuracy=0.9298 | f1=0.9024 | auc=0.9912 | 0.6272s

----------------------------------------------------------------------
MATCHED-DATA CLASSICAL BASELINE (100 training rows)
----------------------------------------------------------------------
Training Logistic Regression... COMPLETED | accuracy=0.9561 | f1=0.9398 | auc=0.9960 | 0.0037s
Training RBF SVM... COMPLETED | accuracy=0.9474 | f1=0.9250 | auc=0.9957 | 0.0031s
Training Random Forest... COMPLETED | accuracy=0.9298 | f1=0.8974 | auc=0.9901 | 0.6164s

CLASSICAL BENCHMARK RESULTS


,Condition,Model,Status,Training Rows,Accuracy,Precision,Sensitivity,Specificity,F1,ROC-AUC,Training Time (s)
0,FULL DATA,Logistic Regression,COMPLETED,455,0.9561,0.9512,0.9286,0.9722,0.9398,0.9954,0.0059
1,FULL DATA,RBF SVM,COMPLETED,455,0.9386,0.9730,0.8571,0.9861,0.9114,0.9954,0.0146
2,FULL DATA,Random Forest,COMPLETED,455,0.9298,0.9250,0.8810,0.9583,0.9024,0.9912,0.6272
3,MATCHED DATA,Logistic Regression,COMPLETED,100,0.9561,0.9512,0.9286,0.9722,0.9398,0.9960,0.0037
4,MATCHED DATA,RBF SVM,COMPLETED,100,0.9474,0.9737,0.8810,0.9861,0.9250,0.9957,0.0031
5,MATCHED DATA,Random Forest,COMPLETED,100,0.9298,0.9722,0.8333,0.9861,0.8974,0.9901,0.6164



Saved classical benchmark:
/kaggle/working/Q-MedAI/results/classical_benchmark.csv

FULL-DATA CLASSICAL BENCHMARK PASSED
MATCHED-DATA CLASSICAL BENCHMARK PASSED
CONTINUOUS ROC-AUC SCORE CHECK PASSED


In [44]:
# ============================================================
# CELL 8 — VQC CIRCUIT + GRADIENT SMOKE TEST
# Q-MedAI | SIH 2026 | PS 26139
# ============================================================

import time
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

print("=" * 70)
print("Q-MedAI — VQC Circuit & Gradient Smoke Test")
print("=" * 70)


# ============================================================
# 1. CONFIGURATION
# ============================================================

N_QUBITS = int(CFG["feature_count"])
VQC_LAYERS = int(CFG["vqc_layers"])
VQC_ITERATIONS = int(CFG["vqc_iterations"])
VQC_BATCH_SIZE = int(CFG["vqc_batch_size"])
QUANTUM_TIMEOUT_SECONDS = float(
    CFG["quantum_timeout_seconds"]
)

assert X_train_match_quantum.ndim == 2

assert N_QUBITS == X_train_match_quantum.shape[1], (
    f"Qubit count ({N_QUBITS}) does not match "
    f"quantum feature count "
    f"({X_train_match_quantum.shape[1]})."
)

assert N_QUBITS >= 2
assert VQC_LAYERS >= 1
assert VQC_ITERATIONS >= 1
assert VQC_BATCH_SIZE >= 1


print()
print(f"Profile          : {PROFILE}")
print(f"Qubits           : {N_QUBITS}")
print(f"Layers           : {VQC_LAYERS}")
print(f"Iterations       : {VQC_ITERATIONS}")
print(f"Batch size       : {VQC_BATCH_SIZE}")
print(
    f"Timeout (s)      : "
    f"{QUANTUM_TIMEOUT_SECONDS:g}"
)

print()
print(
    "Backend          : PennyLane default.qubit "
    "(classical quantum-circuit simulation)"
)


# ============================================================
# 2. DEVICE
# ============================================================

vqc_device = qml.device(
    "default.qubit",
    wires=N_QUBITS,
)


# ============================================================
# 3. FEATURE ENCODING
#
# RY(x_i) on each qubit
# ============================================================

def encode_features(x):

    if len(x) != N_QUBITS:
        raise ValueError(
            f"Expected {N_QUBITS} features, "
            f"received {len(x)}."
        )

    for wire in range(N_QUBITS):

        qml.RY(
            x[wire],
            wires=wire,
        )


# ============================================================
# 4. VARIATIONAL LAYER
#
# Per qubit:
#
# RX(theta_1)
# RY(theta_2)
# RZ(theta_3)
#
# followed by CNOT ring
# ============================================================

def variational_layer(layer_weights):

    expected_shape = (
        N_QUBITS,
        3,
    )

    if tuple(layer_weights.shape) != expected_shape:

        raise ValueError(
            f"Expected layer shape "
            f"{expected_shape}, "
            f"received "
            f"{tuple(layer_weights.shape)}."
        )

    # ----------------------------------------
    # Trainable rotations
    # ----------------------------------------

    for wire in range(N_QUBITS):

        qml.RX(
            layer_weights[
                wire,
                0,
            ],
            wires=wire,
        )

        qml.RY(
            layer_weights[
                wire,
                1,
            ],
            wires=wire,
        )

        qml.RZ(
            layer_weights[
                wire,
                2,
            ],
            wires=wire,
        )

    # ----------------------------------------
    # CNOT entanglement ring
    # ----------------------------------------

    for wire in range(N_QUBITS):

        next_wire = (
            wire + 1
        ) % N_QUBITS

        qml.CNOT(
            wires=[
                wire,
                next_wire,
            ]
        )


# ============================================================
# 5. VQC QNODE
#
# Measurement:
#
#     <Z_0>
#
# Probability-like score:
#
#     p = (1 + <Z_0>) / 2
#
# ============================================================

@qml.qnode(
    vqc_device,
    interface="autograd",
    diff_method="backprop",
)
def vqc_circuit(
    x,
    weights,
):

    encode_features(x)

    for layer_index in range(
        VQC_LAYERS
    ):

        variational_layer(
            weights[
                layer_index
            ]
        )

    return qml.expval(
        qml.PauliZ(0)
    )


# ============================================================
# 6. CONTINUOUS VQC SCORE
# ============================================================

def expectation_to_probability(z):

    return (
        1.0 + z
    ) / 2.0


def vqc_probability(
    x,
    weights,
):

    expectation = vqc_circuit(
        x,
        weights,
    )

    return expectation_to_probability(
        expectation
    )


# ============================================================
# 7. PARAMETER INITIALIZATION
# ============================================================

rng = np.random.default_rng(
    RANDOM_SEED
)

initial_weights_numpy = rng.normal(
    loc=0.0,
    scale=0.10,
    size=(
        VQC_LAYERS,
        N_QUBITS,
        3,
    ),
)

initial_weights = pnp.array(
    initial_weights_numpy,
    requires_grad=True,
)

assert initial_weights.shape == (
    VQC_LAYERS,
    N_QUBITS,
    3,
)

print()
print(
    "Weight shape     :",
    initial_weights.shape,
)


# ============================================================
# 8. REAL TRAINING SAMPLE FOR SMOKE TEST
# ============================================================

x_smoke = pnp.array(
    np.asarray(
        X_train_match_quantum[0],
        dtype=float,
    ),
    requires_grad=False,
)

# IMPORTANT:
#
# Keep the label as a plain Python float.
# Do NOT use a PennyLane tensor here.
#
# This avoids the NonDifferentiableError seen previously.

y_smoke = float(
    np.asarray(
        y_train_match
    )[0]
)

assert y_smoke in (
    0.0,
    1.0,
)


# ============================================================
# 9. FORWARD-PASS SMOKE TEST
# ============================================================

start_time = time.perf_counter()

z_smoke = vqc_circuit(
    x_smoke,
    initial_weights,
)

p_smoke = vqc_probability(
    x_smoke,
    initial_weights,
)

forward_seconds = (
    time.perf_counter()
    - start_time
)

z_value = float(
    z_smoke
)

p_value = float(
    p_smoke
)

print()
print(
    "Forward-pass smoke test:"
)

print(
    f"Target                 : "
    f"{int(y_smoke)}"
)

print(
    f"<Z>                    : "
    f"{z_value:.8f}"
)

print(
    f"Probability-like score : "
    f"{p_value:.8f}"
)

print(
    f"Forward time           : "
    f"{forward_seconds:.6f}s"
)


# ============================================================
# 10. FORWARD OUTPUT VALIDATION
# ============================================================

assert np.isfinite(
    z_value
)

assert np.isfinite(
    p_value
)

assert (
    -1.0 - 1e-9
    <= z_value
    <= 1.0 + 1e-9
), (
    f"Invalid Pauli-Z expectation: "
    f"{z_value}"
)

assert (
    -1e-9
    <= p_value
    <= 1.0 + 1e-9
), (
    f"Invalid probability-like "
    f"score: {p_value}"
)


# ============================================================
# 11. BINARY CROSS-ENTROPY LOSS
#
# Important Autograd rule:
#
# - weights: differentiable PennyLane tensor
# - x: fixed non-differentiable data
# - target: plain Python float
#
# ============================================================

EPS = 1e-7


def binary_cross_entropy_from_score(
    probability,
    target,
):

    # Keep target as ordinary scalar.
    target = float(target)

    # Avoid log(0).
    probability = pnp.clip(
        probability,
        EPS,
        1.0 - EPS,
    )

    return -(
        target
        * pnp.log(
            probability
        )
        +
        (
            1.0 - target
        )
        * pnp.log(
            1.0 - probability
        )
    )


def single_sample_loss(
    weights,
    x,
    target,
):

    probability = vqc_probability(
        x,
        weights,
    )

    return (
        binary_cross_entropy_from_score(
            probability,
            target,
        )
    )


# ============================================================
# 12. LOSS FOR DIFFERENTIATION
#
# CRITICAL FIX
#
# The function passed to qml.grad has exactly ONE argument:
#
#     weights
#
# x_smoke and y_smoke are captured constants.
#
# Therefore Autograd cannot accidentally try to
# differentiate with respect to data or labels.
# ============================================================

def smoke_loss_weights_only(
    weights,
):

    return single_sample_loss(
        weights,
        x_smoke,
        y_smoke,
    )


# ============================================================
# 13. LOSS FORWARD TEST
# ============================================================

loss_value = smoke_loss_weights_only(
    initial_weights
)

loss_float = float(
    loss_value
)

print()
print(
    "Loss smoke test:"
)

print(
    f"Binary cross-entropy   : "
    f"{loss_float:.10f}"
)

assert np.isfinite(
    loss_float
)

assert loss_float >= 0.0


# ============================================================
# 14. GRADIENT SMOKE TEST
#
# Differentiate ONLY the single weights argument.
# ============================================================

gradient_function = qml.grad(
    smoke_loss_weights_only
)

start_time = time.perf_counter()

gradient = gradient_function(
    initial_weights
)

gradient_seconds = (
    time.perf_counter()
    - start_time
)

gradient_numpy = np.asarray(
    gradient,
    dtype=float,
)

gradient_norm = float(
    np.linalg.norm(
        gradient_numpy
    )
)


print()
print(
    "Gradient smoke test:"
)

print(
    "Gradient shape :",
    gradient_numpy.shape,
)

print(
    "Gradient norm  :",
    f"{gradient_norm:.10f}",
)

print(
    "Gradient time  :",
    f"{gradient_seconds:.6f}s",
)


# ============================================================
# 15. GRADIENT VALIDATION
# ============================================================

expected_gradient_shape = (
    VQC_LAYERS,
    N_QUBITS,
    3,
)

assert (
    gradient_numpy.shape
    ==
    expected_gradient_shape
), (
    f"Unexpected gradient shape. "
    f"Expected "
    f"{expected_gradient_shape}, "
    f"received "
    f"{gradient_numpy.shape}."
)

assert np.all(
    np.isfinite(
        gradient_numpy
    )
), (
    "Gradient contains NaN "
    "or infinite values."
)

assert gradient_norm > 1e-12, (
    "Gradient is numerically zero. "
    "VQC optimization could not "
    "progress from this initialization."
)


# ============================================================
# 16. ONE MANUAL GRADIENT STEP TEST
#
# This proves the weights can actually be updated.
# ============================================================

SMOKE_LEARNING_RATE = 0.05

updated_weights = pnp.array(
    initial_weights
    -
    SMOKE_LEARNING_RATE
    * gradient,
    requires_grad=True,
)

updated_loss = (
    smoke_loss_weights_only(
        updated_weights
    )
)

updated_loss_float = float(
    updated_loss
)

parameter_change = float(
    np.linalg.norm(
        np.asarray(
            updated_weights
            -
            initial_weights,
            dtype=float,
        )
    )
)


print()
print(
    "One-step optimizer smoke test:"
)

print(
    f"Initial loss    : "
    f"{loss_float:.10f}"
)

print(
    f"Updated loss    : "
    f"{updated_loss_float:.10f}"
)

print(
    f"Parameter change: "
    f"{parameter_change:.10f}"
)


assert np.isfinite(
    updated_loss_float
)

assert parameter_change > 0.0


# ============================================================
# 17. TINY BATCH SCORE TEST
# ============================================================

tiny_batch = np.asarray(
    X_train_match_quantum[:5],
    dtype=float,
)

tiny_scores = []

for row in tiny_batch:

    row_tensor = pnp.array(
        row,
        requires_grad=False,
    )

    score = vqc_probability(
        row_tensor,
        initial_weights,
    )

    tiny_scores.append(
        float(score)
    )

tiny_scores = np.asarray(
    tiny_scores,
    dtype=float,
)


print()
print(
    "Tiny-batch score test:"
)

print(
    "Shape  :",
    tiny_scores.shape,
)

print(
    "Scores :",
    np.round(
        tiny_scores,
        6,
    ),
)


# ============================================================
# 18. BATCH VALIDATION
# ============================================================

assert tiny_scores.shape == (
    5,
)

assert np.all(
    np.isfinite(
        tiny_scores
    )
)

assert np.all(
    tiny_scores >= -1e-9
)

assert np.all(
    tiny_scores
    <= 1.0 + 1e-9
)


# ============================================================
# 19. VERIFY CONTINUOUS OUTPUTS
# ============================================================

unique_scores = np.unique(
    np.round(
        tiny_scores,
        12,
    )
)

assert len(
    unique_scores
) > 2, (
    "VQC appears to output "
    "only binary/hard predictions."
)

print()
print(
    f"Unique continuous scores: "
    f"{len(unique_scores)}"
)


# ============================================================
# 20. CIRCUIT SPECIFICATIONS
# ============================================================

try:

    circuit_specs = qml.specs(
        vqc_circuit
    )(
        x_smoke,
        initial_weights,
    )

    print()
    print(
        "Circuit specifications:"
    )

    resources = (
        circuit_specs
        .get(
            "resources",
            None,
        )
    )

    if resources is not None:

        print(
            resources
        )

    else:

        print(
            "Resource metadata not "
            "available."
        )

except Exception as exc:

    # Diagnostic only.
    # A qml.specs problem must not invalidate
    # a working differentiable circuit.

    print()
    print(
        "Circuit specification inspection "
        "was skipped:"
    )

    print(
        f"{type(exc).__name__}: "
        f"{exc}"
    )


# ============================================================
# 21. FINAL STATUS
# ============================================================

print()
print(
    "=" * 70
)

print(
    "VQC FORWARD PASS PASSED"
)

print(
    "VQC PROBABILITY SCORE PASSED"
)

print(
    "VQC LOSS FUNCTION PASSED"
)

print(
    "VQC GRADIENT PASSED"
)

print(
    "VQC PARAMETER UPDATE PASSED"
)

print(
    "VQC OUTPUT-SHAPE TEST PASSED"
)

print(
    "VQC CONTINUOUS-SCORE TEST PASSED"
)

print(
    "=" * 70
)

print()
print(
    "No full VQC training has "
    "been performed yet."
)

print(
    "The circuit, loss, gradient, "
    "and parameter-update paths "
    "are ready for training."
)

Q-MedAI — VQC Circuit & Gradient Smoke Test

Profile          : SAFE
Qubits           : 4
Layers           : 2
Iterations       : 30
Batch size       : 20
Timeout (s)      : 180

Backend          : PennyLane default.qubit (classical quantum-circuit simulation)

Weight shape     : (2, 4, 3)

Forward-pass smoke test:
Target                 : 1
<Z>                    : 0.07273360
Probability-like score : 0.53636680
Forward time           : 0.025107s

Loss smoke test:
Binary cross-entropy   : 0.6229370274

Gradient smoke test:
Gradient shape : (2, 4, 3)
Gradient norm  : 0.7302146388
Gradient time  : 0.030285s

One-step optimizer smoke test:
Initial loss    : 0.6229370274
Updated loss    : 0.5965629303
Parameter change: 0.0365107319

Tiny-batch score test:
Shape  : (5,)
Scores : [0.536367 0.574561 0.563516 0.371843 0.575697]

Unique continuous scores: 5

Circuit specifications:

Circuit specification inspection was skipped:
AttributeError: 'CircuitSpecs' object has no attribute 'get'

VQC F

In [45]:
# ============================================================
# CELL 9 — VQC TRAINING + EVALUATION
# Q-MedAI | SIH 2026 | PS 26139
# ============================================================

import time
import json
import numpy as np
import pandas as pd
from pennylane import numpy as pnp

print("=" * 70)
print("Q-MedAI — VQC Training & Evaluation")
print("=" * 70)


# ============================================================
# 1. TRAINING CONFIGURATION
# ============================================================

VQC_LEARNING_RATE = 0.03

ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.999
ADAM_EPS = 1e-8

VQC_THRESHOLD = 0.50

print()
print(f"Profile          : {PROFILE}")
print(f"Training rows    : {len(y_train_match)}")
print(f"Test rows        : {len(y_test)}")
print(f"Qubits           : {N_QUBITS}")
print(f"Layers           : {VQC_LAYERS}")
print(f"Iterations       : {VQC_ITERATIONS}")
print(f"Batch size       : {VQC_BATCH_SIZE}")
print(f"Learning rate    : {VQC_LEARNING_RATE}")
print(f"Timeout (s)      : {QUANTUM_TIMEOUT_SECONDS:g}")
print(f"Threshold        : {VQC_THRESHOLD}")

print()
print(
    "Backend          : PennyLane default.qubit "
    "(classical simulation)"
)


# ============================================================
# 2. PREPARE TRAINING ARRAYS
# ============================================================

X_vqc_train = np.asarray(
    X_train_match_quantum,
    dtype=float,
)

y_vqc_train = np.asarray(
    y_train_match,
    dtype=float,
)

X_vqc_test = np.asarray(
    X_test_match_quantum,
    dtype=float,
)

y_vqc_test = np.asarray(
    y_test,
    dtype=int,
)

assert X_vqc_train.shape == (
    len(y_vqc_train),
    N_QUBITS,
)

assert X_vqc_test.shape == (
    len(y_vqc_test),
    N_QUBITS,
)

assert set(
    np.unique(y_vqc_train)
).issubset(
    {0.0, 1.0}
)


# ============================================================
# 3. MINI-BATCH LOSS
#
# CRITICAL AUTOGRAD DESIGN:
#
# The function differentiated by qml.grad will receive
# ONLY `weights`.
#
# Features and labels are captured by the closure as fixed
# ordinary NumPy/Python values.
# ============================================================

def make_batch_loss(
    batch_x,
    batch_y,
):
    """
    Return a differentiable function:

        loss(weights)

    with batch data captured as constants.
    """

    batch_x = np.asarray(
        batch_x,
        dtype=float,
    )

    batch_y = np.asarray(
        batch_y,
        dtype=float,
    )

    def batch_loss_weights_only(
        weights,
    ):
        losses = []

        for row, target in zip(
            batch_x,
            batch_y,
        ):
            x_tensor = pnp.array(
                row,
                requires_grad=False,
            )

            probability = (
                vqc_probability(
                    x_tensor,
                    weights,
                )
            )

            probability = pnp.clip(
                probability,
                EPS,
                1.0 - EPS,
            )

            # IMPORTANT:
            # target stays a plain Python float.
            target_float = float(
                target
            )

            sample_loss = -(
                target_float
                * pnp.log(
                    probability
                )
                +
                (
                    1.0
                    - target_float
                )
                * pnp.log(
                    1.0
                    - probability
                )
            )

            losses.append(
                sample_loss
            )

        return pnp.mean(
            pnp.stack(
                losses
            )
        )

    return batch_loss_weights_only


# ============================================================
# 4. SCORE/PREDICTION HELPERS
# ============================================================

def vqc_predict_scores(
    X,
    weights,
):
    """
    Continuous probability-like scores.

        p = (1 + <Z>) / 2

    These scores are used for ROC-AUC.
    """

    X = np.asarray(
        X,
        dtype=float,
    )

    scores = []

    for row in X:
        x_tensor = pnp.array(
            row,
            requires_grad=False,
        )

        score = (
            vqc_probability(
                x_tensor,
                weights,
            )
        )

        score_float = float(
            score
        )

        if not np.isfinite(
            score_float
        ):
            raise ValueError(
                "VQC produced a non-finite score."
            )

        scores.append(
            score_float
        )

    return np.asarray(
        scores,
        dtype=float,
    )


def vqc_hard_predictions(
    scores,
    threshold=0.50,
):
    return (
        np.asarray(
            scores
        )
        >= threshold
    ).astype(int)


# ============================================================
# 5. INITIALIZE WEIGHTS
#
# Fresh copy of the SAME reproducible initialization used
# by the smoke test.
# ============================================================

vqc_weights = pnp.array(
    initial_weights_numpy.copy(),
    requires_grad=True,
)

assert vqc_weights.shape == (
    VQC_LAYERS,
    N_QUBITS,
    3,
)


# ============================================================
# 6. MANUAL ADAM STATE
#
# We implement Adam ourselves instead of relying on a
# PennyLane optimizer API so this notebook stays stable across
# PennyLane optimizer-interface changes.
# ============================================================

adam_m = np.zeros_like(
    initial_weights_numpy,
    dtype=float,
)

adam_v = np.zeros_like(
    initial_weights_numpy,
    dtype=float,
)


# ============================================================
# 7. REPRODUCIBLE MINI-BATCH RNG
# ============================================================

batch_rng = np.random.default_rng(
    RANDOM_SEED
)


# ============================================================
# 8. RESULT OBJECT
# ============================================================

vqc_result = {
    "model": "VQC",
    "status": "NOT TRAINED",
    "error": None,
    "training_seconds": None,
    "iterations_requested": int(
        VQC_ITERATIONS
    ),
    "iterations_completed": 0,
    "metrics": None,
    "confusion_matrix": None,
    "fpr": None,
    "tpr": None,
    "thresholds": None,
    "y_pred": None,
    "y_score": None,
    "loss_history": [],
    "gradient_norm_history": [],
    "final_weights": None,
}


# ============================================================
# 9. TRAINING LOOP
# ============================================================

print()
print("-" * 70)
print("Starting VQC optimization")
print("-" * 70)

training_start = (
    time.perf_counter()
)

vqc_result["status"] = "TRAINING"

try:

    for iteration in range(
        1,
        VQC_ITERATIONS + 1,
    ):

        # ----------------------------------------
        # Cooperative timeout BEFORE iteration
        # ----------------------------------------

        elapsed_before = (
            time.perf_counter()
            - training_start
        )

        if (
            elapsed_before
            >= QUANTUM_TIMEOUT_SECONDS
        ):
            vqc_result[
                "status"
            ] = "TIMED OUT"

            vqc_result[
                "error"
            ] = (
                "Cooperative VQC wall-clock "
                f"timeout reached after "
                f"{elapsed_before:.2f} seconds."
            )

            break


        # ----------------------------------------
        # Reproducible random mini-batch
        # ----------------------------------------

        current_batch_size = min(
            VQC_BATCH_SIZE,
            len(X_vqc_train),
        )

        batch_indices = (
            batch_rng.choice(
                len(X_vqc_train),
                size=current_batch_size,
                replace=False,
            )
        )

        batch_x = (
            X_vqc_train[
                batch_indices
            ]
        )

        batch_y = (
            y_vqc_train[
                batch_indices
            ]
        )


        # ----------------------------------------
        # Create weights-only loss closure
        # ----------------------------------------

        batch_loss_fn = (
            make_batch_loss(
                batch_x,
                batch_y,
            )
        )


        # ----------------------------------------
        # Forward batch loss
        # ----------------------------------------

        current_loss_tensor = (
            batch_loss_fn(
                vqc_weights
            )
        )

        current_loss = float(
            current_loss_tensor
        )

        if not np.isfinite(
            current_loss
        ):
            raise ValueError(
                "VQC batch loss became "
                "non-finite."
            )


        # ----------------------------------------
        # Compute gradient ONLY w.r.t. weights
        # ----------------------------------------

        grad_fn = qml.grad(
            batch_loss_fn
        )

        gradient = grad_fn(
            vqc_weights
        )

        gradient_np = np.asarray(
            gradient,
            dtype=float,
        )

        if gradient_np.shape != (
            VQC_LAYERS,
            N_QUBITS,
            3,
        ):
            raise ValueError(
                "Unexpected VQC gradient shape: "
                f"{gradient_np.shape}"
            )

        if not np.all(
            np.isfinite(
                gradient_np
            )
        ):
            raise ValueError(
                "VQC gradient contains "
                "NaN or infinity."
            )

        gradient_norm = float(
            np.linalg.norm(
                gradient_np
            )
        )


        # ----------------------------------------
        # Adam optimizer update
        # ----------------------------------------

        adam_m = (
            ADAM_BETA1
            * adam_m
            +
            (
                1.0
                - ADAM_BETA1
            )
            * gradient_np
        )

        adam_v = (
            ADAM_BETA2
            * adam_v
            +
            (
                1.0
                - ADAM_BETA2
            )
            * (
                gradient_np ** 2
            )
        )

        m_hat = (
            adam_m
            /
            (
                1.0
                - ADAM_BETA1
                ** iteration
            )
        )

        v_hat = (
            adam_v
            /
            (
                1.0
                - ADAM_BETA2
                ** iteration
            )
        )

        updated_weights_np = (
            np.asarray(
                vqc_weights,
                dtype=float,
            )
            -
            VQC_LEARNING_RATE
            * m_hat
            /
            (
                np.sqrt(
                    v_hat
                )
                + ADAM_EPS
            )
        )

        vqc_weights = pnp.array(
            updated_weights_np,
            requires_grad=True,
        )


        # ----------------------------------------
        # Record iteration
        # ----------------------------------------

        vqc_result[
            "loss_history"
        ].append(
            current_loss
        )

        vqc_result[
            "gradient_norm_history"
        ].append(
            gradient_norm
        )

        vqc_result[
            "iterations_completed"
        ] = iteration


        # ----------------------------------------
        # Progress output
        # ----------------------------------------

        elapsed_now = (
            time.perf_counter()
            - training_start
        )

        print(
            f"Iteration "
            f"{iteration:02d}/"
            f"{VQC_ITERATIONS} "
            f"| loss="
            f"{current_loss:.6f} "
            f"| grad="
            f"{gradient_norm:.6f} "
            f"| elapsed="
            f"{elapsed_now:.1f}s"
        )


        # ----------------------------------------
        # Cooperative timeout AFTER iteration
        # ----------------------------------------

        if (
            elapsed_now
            >= QUANTUM_TIMEOUT_SECONDS
            and iteration
            < VQC_ITERATIONS
        ):
            vqc_result[
                "status"
            ] = "TIMED OUT"

            vqc_result[
                "error"
            ] = (
                "Cooperative VQC wall-clock "
                f"timeout reached after "
                f"{elapsed_now:.2f} seconds."
            )

            break


    # ========================================================
    # 10. DETERMINE TRAINING STATUS
    # ========================================================

    training_seconds = (
        time.perf_counter()
        - training_start
    )

    vqc_result[
        "training_seconds"
    ] = float(
        training_seconds
    )

    if (
        vqc_result["status"]
        != "TIMED OUT"
    ):

        if (
            vqc_result[
                "iterations_completed"
            ]
            ==
            VQC_ITERATIONS
        ):
            vqc_result[
                "status"
            ] = "COMPLETED"

        else:
            raise RuntimeError(
                "VQC training exited before "
                "completing all iterations "
                "without a timeout."
            )


except Exception as exc:

    training_seconds = (
        time.perf_counter()
        - training_start
    )

    vqc_result[
        "training_seconds"
    ] = float(
        training_seconds
    )

    vqc_result[
        "status"
    ] = "FAILED"

    vqc_result[
        "error"
    ] = (
        f"{type(exc).__name__}: "
        f"{exc}"
    )


# ============================================================
# 11. EVALUATE ONLY IF TRAINING COMPLETED
#
# IMPORTANT:
# TIMED OUT / FAILED models receive NO fabricated metrics.
# ============================================================

if (
    vqc_result["status"]
    == "COMPLETED"
):

    print()
    print("-" * 70)
    print("Evaluating completed VQC")
    print("-" * 70)

    try:

        # ----------------------------------------
        # Continuous test scores
        # ----------------------------------------

        y_vqc_score = (
            vqc_predict_scores(
                X_vqc_test,
                vqc_weights,
            )
        )

        # ----------------------------------------
        # Hard predictions at documented threshold
        # ----------------------------------------

        y_vqc_pred = (
            vqc_hard_predictions(
                y_vqc_score,
                threshold=VQC_THRESHOLD,
            )
        )


        # ----------------------------------------
        # Verify scores are genuinely continuous
        # ----------------------------------------

        unique_scores = np.unique(
            np.round(
                y_vqc_score,
                12,
            )
        )

        if len(
            unique_scores
        ) <= 2:

            raise ValueError(
                "VQC ROC-AUC scores appear "
                "to be hard/binary outputs."
            )


        # ----------------------------------------
        # Shared metric function from Cell 7
        # ----------------------------------------

        evaluation = (
            compute_binary_metrics(
                y_vqc_test,
                y_vqc_pred,
                y_vqc_score,
            )
        )

        vqc_result.update(
            {
                "metrics": evaluation[
                    "metrics"
                ],
                "confusion_matrix": evaluation[
                    "confusion_matrix"
                ],
                "fpr": evaluation[
                    "fpr"
                ],
                "tpr": evaluation[
                    "tpr"
                ],
                "thresholds": evaluation[
                    "thresholds"
                ],
                "y_pred": y_vqc_pred,
                "y_score": y_vqc_score,
                "final_weights": np.asarray(
                    vqc_weights,
                    dtype=float,
                ),
            }
        )

    except Exception as exc:

        # If evaluation itself fails, do NOT leave
        # status as COMPLETED.

        vqc_result[
            "status"
        ] = "FAILED"

        vqc_result[
            "error"
        ] = (
            "VQC evaluation failed: "
            f"{type(exc).__name__}: "
            f"{exc}"
        )

        vqc_result[
            "metrics"
        ] = None


# ============================================================
# 12. REPORT STATUS
# ============================================================

print()
print("=" * 70)
print("VQC TRAINING RESULT")
print("=" * 70)

print(
    f"Status               : "
    f"{vqc_result['status']}"
)

print(
    f"Iterations requested : "
    f"{vqc_result['iterations_requested']}"
)

print(
    f"Iterations completed : "
    f"{vqc_result['iterations_completed']}"
)

print(
    f"Training time        : "
    f"{vqc_result['training_seconds']:.3f}s"
)

if (
    vqc_result["error"]
    is not None
):
    print(
        f"Error                : "
        f"{vqc_result['error']}"
    )


# ============================================================
# 13. DISPLAY REAL METRICS
# ============================================================

if (
    vqc_result["status"]
    == "COMPLETED"
):

    metrics = (
        vqc_result[
            "metrics"
        ]
    )

    print()
    print("Held-out test metrics")
    print("-" * 70)

    print(
        f"Accuracy     : "
        f"{metrics['accuracy']:.4f}"
    )

    print(
        f"Precision    : "
        f"{metrics['precision']:.4f}"
    )

    print(
        f"Sensitivity  : "
        f"{metrics['recall_sensitivity']:.4f}"
    )

    print(
        f"Specificity  : "
        f"{metrics['specificity']:.4f}"
    )

    print(
        f"F1           : "
        f"{metrics['f1']:.4f}"
    )

    print(
        f"ROC-AUC      : "
        f"{metrics['roc_auc']:.4f}"
    )

    print()
    print(
        "Confusion matrix "
        "[[TN, FP], [FN, TP]]:"
    )

    print(
        vqc_result[
            "confusion_matrix"
        ]
    )

    print()
    print(
        "Continuous score range:"
    )

    print(
        f"min="
        f"{np.min(vqc_result['y_score']):.6f}, "
        f"max="
        f"{np.max(vqc_result['y_score']):.6f}"
    )

else:

    print()
    print(
        "No VQC performance metrics "
        "are reported because training "
        f"status is {vqc_result['status']}."
    )


# ============================================================
# 14. LOSS HISTORY TABLE
# ============================================================

vqc_history_df = pd.DataFrame(
    {
        "iteration": np.arange(
            1,
            len(
                vqc_result[
                    "loss_history"
                ]
            )
            + 1,
        ),
        "batch_loss": (
            vqc_result[
                "loss_history"
            ]
        ),
        "gradient_norm": (
            vqc_result[
                "gradient_norm_history"
            ]
        ),
    }
)

if len(
    vqc_history_df
) > 0:

    print()
    print("Training history:")

    display(
        vqc_history_df
        .round(6)
    )


# ============================================================
# 15. SAVE TRAINING HISTORY
# ============================================================

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

vqc_history_path = (
    RESULTS_DIR
    / "vqc_training_history.csv"
)

vqc_history_df.to_csv(
    vqc_history_path,
    index=False,
)

print()
print(
    "Saved VQC training history:"
)
print(
    vqc_history_path
)


# ============================================================
# 16. SAVE SAFE JSON SUMMARY
#
# Arrays/weights are intentionally excluded from JSON.
# ============================================================

vqc_summary = {
    "model": "VQC",
    "status": (
        vqc_result[
            "status"
        ]
    ),
    "profile": PROFILE,
    "qubits": N_QUBITS,
    "layers": VQC_LAYERS,
    "iterations_requested": (
        VQC_ITERATIONS
    ),
    "iterations_completed": (
        vqc_result[
            "iterations_completed"
        ]
    ),
    "batch_size": (
        VQC_BATCH_SIZE
    ),
    "learning_rate": (
        VQC_LEARNING_RATE
    ),
    "training_rows": int(
        len(
            y_vqc_train
        )
    ),
    "test_rows": int(
        len(
            y_vqc_test
        )
    ),
    "threshold": (
        VQC_THRESHOLD
    ),
    "training_seconds": (
        vqc_result[
            "training_seconds"
        ]
    ),
    "backend": (
        "PennyLane default.qubit "
        "classical simulation"
    ),
    "metrics": (
        vqc_result[
            "metrics"
        ]
    ),
    "error": (
        vqc_result[
            "error"
        ]
    ),
}

vqc_summary_path = (
    RESULTS_DIR
    / "vqc_summary.json"
)

with open(
    vqc_summary_path,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        vqc_summary,
        handle,
        indent=2,
    )


print()
print(
    "Saved VQC summary:"
)
print(
    vqc_summary_path
)


# ============================================================
# 17. FINAL VERIFICATION
# ============================================================

assert (
    vqc_result[
        "status"
    ]
    in {
        "COMPLETED",
        "FAILED",
        "TIMED OUT",
    }
)

if (
    vqc_result[
        "status"
    ]
    == "COMPLETED"
):

    assert (
        vqc_result[
            "metrics"
        ]
        is not None
    )

    assert (
        vqc_result[
            "confusion_matrix"
        ].shape
        == (2, 2)
    )

    assert len(
        vqc_result[
            "y_score"
        ]
    ) == len(
        y_test
    )

    assert len(
        np.unique(
            np.round(
                vqc_result[
                    "y_score"
                ],
                12,
            )
        )
    ) > 2

else:

    # Honesty requirement:
    # failed/timed-out model must not expose
    # invented performance metrics.

    assert (
        vqc_result[
            "metrics"
        ]
        is None
    )


print()
print("=" * 70)

if (
    vqc_result["status"]
    == "COMPLETED"
):
    print(
        "VQC TRAINING COMPLETED"
    )
    print(
        "VQC EVALUATION PASSED"
    )
    print(
        "VQC CONTINUOUS ROC-AUC SCORE CHECK PASSED"
    )

elif (
    vqc_result["status"]
    == "TIMED OUT"
):
    print(
        "VQC TIMED OUT CLEANLY"
    )
    print(
        "NO VQC METRICS WERE FABRICATED"
    )

else:
    print(
        "VQC FAILED CLEANLY"
    )
    print(
        "NO VQC METRICS WERE FABRICATED"
    )

print("=" * 70)

Q-MedAI — VQC Training & Evaluation

Profile          : SAFE
Training rows    : 100
Test rows        : 114
Qubits           : 4
Layers           : 2
Iterations       : 30
Batch size       : 20
Learning rate    : 0.03
Timeout (s)      : 180
Threshold        : 0.5

Backend          : PennyLane default.qubit (classical simulation)

----------------------------------------------------------------------
Starting VQC optimization
----------------------------------------------------------------------
Iteration 01/30 | loss=0.725807 | grad=0.340521 | elapsed=0.8s
Iteration 02/30 | loss=0.565524 | grad=0.303623 | elapsed=1.5s
Iteration 03/30 | loss=0.688402 | grad=0.254287 | elapsed=2.2s
Iteration 04/30 | loss=0.762183 | grad=0.283294 | elapsed=6.2s
Iteration 05/30 | loss=0.693359 | grad=0.286614 | elapsed=7.1s
Iteration 06/30 | loss=0.658677 | grad=0.305500 | elapsed=8.1s
Iteration 07/30 | loss=0.565686 | grad=0.248167 | elapsed=9.0s
Iteration 08/30 | loss=0.609334 | grad=0.238122 | elapsed=9.

,iteration,batch_loss,gradient_norm
0,1,0.725807,0.340521
1,2,0.565524,0.303623
2,3,0.688402,0.254287
3,4,0.762183,0.283294
4,5,0.693359,0.286614
5,6,0.658677,0.305500
6,7,0.565686,0.248167
7,8,0.609334,0.238122
8,9,0.544250,0.234126
9,10,0.595637,0.241531



Saved VQC training history:
/kaggle/working/Q-MedAI/results/vqc_training_history.csv

Saved VQC summary:
/kaggle/working/Q-MedAI/results/vqc_summary.json

VQC TRAINING COMPLETED
VQC EVALUATION PASSED
VQC CONTINUOUS ROC-AUC SCORE CHECK PASSED


In [46]:
# ============================================================
# CELL 10 — QUANTUM KERNEL SVM + EVALUATION
# Q-MedAI | SIH 2026 | PS 26139
# ============================================================

import time
import json
import numpy as np
import pandas as pd
import pennylane as qml

from sklearn.svm import SVC

print("=" * 70)
print("Q-MedAI — Quantum Kernel SVM")
print("=" * 70)


# ============================================================
# 1. CONFIGURATION
# ============================================================

QK_N_QUBITS = int(
    CFG["feature_count"]
)

QK_TIMEOUT_SECONDS = float(
    CFG["quantum_timeout_seconds"]
)

QK_TRAIN_SIZE = len(
    y_train_match
)

QK_TEST_SIZE = len(
    y_test
)

assert QK_N_QUBITS == (
    X_train_match_quantum.shape[1]
)

print()
print(f"Profile          : {PROFILE}")
print(f"Qubits           : {QK_N_QUBITS}")
print(f"Training rows    : {QK_TRAIN_SIZE}")
print(f"Test rows        : {QK_TEST_SIZE}")
print(f"Timeout (s)      : {QK_TIMEOUT_SECONDS:g}")

print()
print(
    "Backend          : PennyLane default.qubit "
    "(classical quantum-circuit simulation)"
)


# ============================================================
# 2. PREPARE DATA
#
# IMPORTANT:
# These are the SAME matched training rows used by:
#
# - matched Logistic Regression
# - matched RBF SVM
# - matched Random Forest
# - VQC
#
# This makes the comparison fair.
# ============================================================

X_qk_train = np.asarray(
    X_train_match_quantum,
    dtype=float,
)

y_qk_train = np.asarray(
    y_train_match,
    dtype=int,
)

X_qk_test = np.asarray(
    X_test_match_quantum,
    dtype=float,
)

y_qk_test = np.asarray(
    y_test,
    dtype=int,
)

assert X_qk_train.shape == (
    QK_TRAIN_SIZE,
    QK_N_QUBITS,
)

assert X_qk_test.shape == (
    QK_TEST_SIZE,
    QK_N_QUBITS,
)


# ============================================================
# 3. QUANTUM KERNEL DEVICE
# ============================================================

qk_device = qml.device(
    "default.qubit",
    wires=QK_N_QUBITS,
)


# ============================================================
# 4. QUANTUM FEATURE MAP
#
# IMPORTANT DESIGN:
#
# Stage 1:
#     RY(x_i)
#
# Stage 2:
#     CNOT ring
#
# Stage 3:
#     data re-uploading using RY(x_i)
#
# Why re-upload data after entanglement?
#
# If we used only:
#
#     D(x) -> fixed E
#
# then in a fidelity construction:
#
#     U(y)^dagger U(x)
#
# the fixed entangler can cancel.
#
# With:
#
#     D2(x) E D1(x)
#
# data-dependent gates remain around the entangling operation,
# so the fidelity kernel retains a nontrivial entangled feature map.
# ============================================================

def quantum_kernel_feature_map(x):

    if len(x) != QK_N_QUBITS:
        raise ValueError(
            f"Expected {QK_N_QUBITS} features, "
            f"received {len(x)}."
        )

    # ----------------------------------------
    # First data encoding
    # ----------------------------------------

    for wire in range(
        QK_N_QUBITS
    ):
        qml.RY(
            x[wire],
            wires=wire,
        )

    # ----------------------------------------
    # Entangling CNOT ring
    # ----------------------------------------

    for wire in range(
        QK_N_QUBITS
    ):

        next_wire = (
            wire + 1
        ) % QK_N_QUBITS

        qml.CNOT(
            wires=[
                wire,
                next_wire,
            ]
        )

    # ----------------------------------------
    # Data re-uploading
    # ----------------------------------------

    for wire in range(
        QK_N_QUBITS
    ):

        next_wire = (
            wire + 1
        ) % QK_N_QUBITS

        # Re-upload same selected feature
        qml.RY(
            0.5 * x[wire],
            wires=wire,
        )

        # Add a data-dependent phase coupling
        qml.RZ(
            0.25
            * x[wire]
            * x[next_wire],
            wires=wire,
        )


# ============================================================
# 5. FIDELITY KERNEL CIRCUIT
#
# We evaluate:
#
#     | <phi(x2) | phi(x1)> |^2
#
# by applying:
#
#     U(x1)
#     U(x2)^dagger
#
# and measuring probability of |0000>.
# ============================================================

@qml.qnode(
    qk_device,
    interface=None,
)
def quantum_kernel_circuit(
    x1,
    x2,
):

    quantum_kernel_feature_map(
        x1
    )

    qml.adjoint(
        quantum_kernel_feature_map
    )(
        x2
    )

    return qml.probs(
        wires=range(
            QK_N_QUBITS
        )
    )


def quantum_kernel_value(
    x1,
    x2,
):
    """
    Fidelity-based quantum kernel.

    Returns a scalar approximately in [0, 1].
    """

    probabilities = (
        quantum_kernel_circuit(
            np.asarray(
                x1,
                dtype=float,
            ),
            np.asarray(
                x2,
                dtype=float,
            ),
        )
    )

    value = float(
        probabilities[0]
    )

    # Numerical precision may produce tiny
    # values such as 1.0000000000000002.
    value = float(
        np.clip(
            value,
            0.0,
            1.0,
        )
    )

    return value


# ============================================================
# 6. SINGLE-KERNEL SMOKE TEST
# ============================================================

print()
print("-" * 70)
print("Quantum kernel smoke test")
print("-" * 70)

kernel_self = quantum_kernel_value(
    X_qk_train[0],
    X_qk_train[0],
)

kernel_cross = quantum_kernel_value(
    X_qk_train[0],
    X_qk_train[1],
)

print(
    f"K(x0, x0) = "
    f"{kernel_self:.10f}"
)

print(
    f"K(x0, x1) = "
    f"{kernel_cross:.10f}"
)

assert np.isfinite(
    kernel_self
)

assert np.isfinite(
    kernel_cross
)

assert np.isclose(
    kernel_self,
    1.0,
    atol=1e-7,
), (
    "Quantum fidelity kernel self-similarity "
    f"should be approximately 1, got {kernel_self}."
)

assert (
    0.0
    <= kernel_cross
    <= 1.0
)


# ============================================================
# 7. RESULT OBJECT
# ============================================================

quantum_kernel_result = {
    "model": "Quantum Kernel SVM",
    "status": "NOT TRAINED",
    "error": None,

    "training_seconds": None,
    "kernel_train_seconds": None,
    "kernel_test_seconds": None,
    "svm_fit_seconds": None,

    "metrics": None,
    "confusion_matrix": None,

    "fpr": None,
    "tpr": None,
    "thresholds": None,

    "y_pred": None,
    "y_score": None,

    "K_train": None,
    "K_test": None,

    "train_kernel_rows_completed": 0,
    "test_kernel_rows_completed": 0,
}


# ============================================================
# 8. TRAIN KERNEL MATRIX
#
# Required shape:
#
#     K_train[i, j]
#       = K(x_train_i, x_train_j)
#
# We exploit symmetry:
#
#     K(i,j) = K(j,i)
#
# so only the upper triangle needs a quantum evaluation.
# ============================================================

def build_train_kernel_matrix(
    X,
    timeout_seconds,
):

    X = np.asarray(
        X,
        dtype=float,
    )

    n = len(X)

    K = np.empty(
        (
            n,
            n,
        ),
        dtype=float,
    )

    start = (
        time.perf_counter()
    )

    completed_rows = 0

    for i in range(n):

        # ----------------------------------------
        # Cooperative timeout
        # ----------------------------------------

        elapsed = (
            time.perf_counter()
            - start
        )

        if (
            elapsed
            >= timeout_seconds
        ):
            raise TimeoutError(
                "Quantum training-kernel "
                f"computation exceeded "
                f"{timeout_seconds:.1f} seconds "
                f"after completing {completed_rows} rows."
            )

        # Fidelity with itself = 1 exactly
        K[i, i] = 1.0

        for j in range(
            i + 1,
            n,
        ):

            elapsed = (
                time.perf_counter()
                - start
            )

            if (
                elapsed
                >= timeout_seconds
            ):
                raise TimeoutError(
                    "Quantum training-kernel "
                    f"computation exceeded "
                    f"{timeout_seconds:.1f} seconds "
                    f"during row {i}."
                )

            value = (
                quantum_kernel_value(
                    X[i],
                    X[j],
                )
            )

            K[i, j] = value
            K[j, i] = value

        completed_rows = (
            i + 1
        )

        if (
            completed_rows == 1
            or completed_rows % 10 == 0
            or completed_rows == n
        ):

            elapsed = (
                time.perf_counter()
                - start
            )

            print(
                f"Train kernel row "
                f"{completed_rows:03d}/{n} "
                f"| elapsed="
                f"{elapsed:.1f}s"
            )

    seconds = (
        time.perf_counter()
        - start
    )

    return (
        K,
        seconds,
        completed_rows,
    )


# ============================================================
# 9. TEST-TRAIN KERNEL MATRIX
#
# CRITICAL LEAKAGE RULE:
#
# Correct:
#
#     K_test[i,j]
#       = K(x_test_i, x_train_j)
#
# Shape:
#
#     test x train
#
# WRONG:
#
#     test x test
#
# We never build a test-test kernel for SVM evaluation.
# ============================================================

def build_test_train_kernel_matrix(
    X_test,
    X_train,
    timeout_seconds,
):

    X_test = np.asarray(
        X_test,
        dtype=float,
    )

    X_train = np.asarray(
        X_train,
        dtype=float,
    )

    n_test = len(
        X_test
    )

    n_train = len(
        X_train
    )

    K = np.empty(
        (
            n_test,
            n_train,
        ),
        dtype=float,
    )

    start = (
        time.perf_counter()
    )

    completed_rows = 0

    for i in range(
        n_test
    ):

        elapsed = (
            time.perf_counter()
            - start
        )

        if (
            elapsed
            >= timeout_seconds
        ):

            raise TimeoutError(
                "Quantum test-train kernel "
                f"computation exceeded "
                f"{timeout_seconds:.1f} seconds "
                f"after completing "
                f"{completed_rows} test rows."
            )

        for j in range(
            n_train
        ):

            elapsed = (
                time.perf_counter()
                - start
            )

            if (
                elapsed
                >= timeout_seconds
            ):
                raise TimeoutError(
                    "Quantum test-train kernel "
                    f"computation exceeded "
                    f"{timeout_seconds:.1f} seconds "
                    f"during test row {i}."
                )

            K[i, j] = (
                quantum_kernel_value(
                    X_test[i],
                    X_train[j],
                )
            )

        completed_rows = (
            i + 1
        )

        if (
            completed_rows == 1
            or completed_rows % 10 == 0
            or completed_rows == n_test
        ):

            elapsed = (
                time.perf_counter()
                - start
            )

            print(
                f"Test kernel row "
                f"{completed_rows:03d}/"
                f"{n_test} "
                f"| elapsed="
                f"{elapsed:.1f}s"
            )

    seconds = (
        time.perf_counter()
        - start
    )

    return (
        K,
        seconds,
        completed_rows,
    )


# ============================================================
# 10. RUN QUANTUM KERNEL SVM
# ============================================================

print()
print("-" * 70)
print("Building quantum training kernel")
print("-" * 70)

overall_start = (
    time.perf_counter()
)

quantum_kernel_result[
    "status"
] = "TRAINING"

try:

    # --------------------------------------------------------
    # Training kernel
    # --------------------------------------------------------

    (
        K_train,
        train_kernel_seconds,
        train_rows_completed,
    ) = build_train_kernel_matrix(
        X_qk_train,
        timeout_seconds=(
            QK_TIMEOUT_SECONDS
        ),
    )

    quantum_kernel_result[
        "train_kernel_rows_completed"
    ] = (
        train_rows_completed
    )

    quantum_kernel_result[
        "kernel_train_seconds"
    ] = float(
        train_kernel_seconds
    )


    # --------------------------------------------------------
    # Validate K_train
    # --------------------------------------------------------

    assert K_train.shape == (
        QK_TRAIN_SIZE,
        QK_TRAIN_SIZE,
    )

    assert np.all(
        np.isfinite(
            K_train
        )
    )

    assert np.allclose(
        K_train,
        K_train.T,
        atol=1e-8,
    ), (
        "Quantum training kernel is "
        "not symmetric."
    )

    assert np.allclose(
        np.diag(
            K_train
        ),
        1.0,
        atol=1e-7,
    )


    # --------------------------------------------------------
    # Fit precomputed-kernel SVM
    # --------------------------------------------------------

    print()
    print("-" * 70)
    print("Fitting SVC(kernel='precomputed')")
    print("-" * 70)

    qk_svm = SVC(
        kernel="precomputed",
        random_state=RANDOM_SEED,
    )

    svm_fit_start = (
        time.perf_counter()
    )

    qk_svm.fit(
        K_train,
        y_qk_train,
    )

    svm_fit_seconds = (
        time.perf_counter()
        - svm_fit_start
    )

    quantum_kernel_result[
        "svm_fit_seconds"
    ] = float(
        svm_fit_seconds
    )


    # --------------------------------------------------------
    # Remaining timeout for test kernel
    # --------------------------------------------------------

    elapsed_total = (
        time.perf_counter()
        - overall_start
    )

    remaining_seconds = (
        QK_TIMEOUT_SECONDS
        - elapsed_total
    )

    if remaining_seconds <= 0:

        raise TimeoutError(
            "Quantum kernel timeout reached "
            "before test-kernel computation."
        )


    # --------------------------------------------------------
    # Build test x train matrix
    # --------------------------------------------------------

    print()
    print("-" * 70)
    print("Building quantum test-train kernel")
    print("-" * 70)

    (
        K_test,
        test_kernel_seconds,
        test_rows_completed,
    ) = (
        build_test_train_kernel_matrix(
            X_qk_test,
            X_qk_train,
            timeout_seconds=(
                remaining_seconds
            ),
        )
    )

    quantum_kernel_result[
        "test_kernel_rows_completed"
    ] = (
        test_rows_completed
    )

    quantum_kernel_result[
        "kernel_test_seconds"
    ] = float(
        test_kernel_seconds
    )


    # --------------------------------------------------------
    # Validate K_test
    # --------------------------------------------------------

    assert K_test.shape == (
        QK_TEST_SIZE,
        QK_TRAIN_SIZE,
    ), (
        "Quantum test kernel has wrong shape. "
        "It must be test x train."
    )

    assert np.all(
        np.isfinite(
            K_test
        )
    )


    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    y_qk_pred = (
        qk_svm.predict(
            K_test
        )
    )


    # --------------------------------------------------------
    # CONTINUOUS ROC-AUC SCORE
    #
    # Required:
    #
    # decision_function()
    #
    # Do NOT invent probabilities.
    # --------------------------------------------------------

    y_qk_score = (
        qk_svm
        .decision_function(
            K_test
        )
    )

    y_qk_score = np.asarray(
        y_qk_score,
        dtype=float,
    )

    assert len(
        y_qk_score
    ) == len(
        y_qk_test
    )

    assert np.all(
        np.isfinite(
            y_qk_score
        )
    )

    unique_scores = np.unique(
        np.round(
            y_qk_score,
            12,
        )
    )

    assert len(
        unique_scores
    ) > 2, (
        "Quantum Kernel SVM appears "
        "to use hard predictions for ROC-AUC."
    )


    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------

    qk_evaluation = (
        compute_binary_metrics(
            y_qk_test,
            y_qk_pred,
            y_qk_score,
        )
    )


    # --------------------------------------------------------
    # Complete result
    # --------------------------------------------------------

    total_seconds = (
        time.perf_counter()
        - overall_start
    )

    quantum_kernel_result.update(
        {
            "status": "COMPLETED",

            "training_seconds": float(
                total_seconds
            ),

            "metrics": (
                qk_evaluation[
                    "metrics"
                ]
            ),

            "confusion_matrix": (
                qk_evaluation[
                    "confusion_matrix"
                ]
            ),

            "fpr": (
                qk_evaluation[
                    "fpr"
                ]
            ),

            "tpr": (
                qk_evaluation[
                    "tpr"
                ]
            ),

            "thresholds": (
                qk_evaluation[
                    "thresholds"
                ]
            ),

            "y_pred": np.asarray(
                y_qk_pred,
                dtype=int,
            ),

            "y_score": (
                y_qk_score
            ),

            "K_train": (
                K_train
            ),

            "K_test": (
                K_test
            ),
        }
    )


# ============================================================
# 11. HANDLE TIMEOUT
# ============================================================

except TimeoutError as exc:

    total_seconds = (
        time.perf_counter()
        - overall_start
    )

    quantum_kernel_result[
        "status"
    ] = "TIMED OUT"

    quantum_kernel_result[
        "training_seconds"
    ] = float(
        total_seconds
    )

    quantum_kernel_result[
        "error"
    ] = (
        f"TimeoutError: {exc}"
    )

    # Honesty requirement
    quantum_kernel_result[
        "metrics"
    ] = None


# ============================================================
# 12. HANDLE FAILURE
# ============================================================

except Exception as exc:

    total_seconds = (
        time.perf_counter()
        - overall_start
    )

    quantum_kernel_result[
        "status"
    ] = "FAILED"

    quantum_kernel_result[
        "training_seconds"
    ] = float(
        total_seconds
    )

    quantum_kernel_result[
        "error"
    ] = (
        f"{type(exc).__name__}: "
        f"{exc}"
    )

    quantum_kernel_result[
        "metrics"
    ] = None


# ============================================================
# 13. REPORT RESULT
# ============================================================

print()
print("=" * 70)
print("QUANTUM KERNEL SVM RESULT")
print("=" * 70)

print(
    f"Status              : "
    f"{quantum_kernel_result['status']}"
)

print(
    f"Training rows       : "
    f"{QK_TRAIN_SIZE}"
)

print(
    f"Test rows           : "
    f"{QK_TEST_SIZE}"
)

print(
    f"Total time          : "
    f"{quantum_kernel_result['training_seconds']:.3f}s"
)

if (
    quantum_kernel_result[
        "kernel_train_seconds"
    ]
    is not None
):

    print(
        f"Train kernel time   : "
        f"{quantum_kernel_result['kernel_train_seconds']:.3f}s"
    )

if (
    quantum_kernel_result[
        "svm_fit_seconds"
    ]
    is not None
):

    print(
        f"SVM fit time        : "
        f"{quantum_kernel_result['svm_fit_seconds']:.6f}s"
    )

if (
    quantum_kernel_result[
        "kernel_test_seconds"
    ]
    is not None
):

    print(
        f"Test kernel time    : "
        f"{quantum_kernel_result['kernel_test_seconds']:.3f}s"
    )

if (
    quantum_kernel_result[
        "error"
    ]
    is not None
):

    print(
        f"Error               : "
        f"{quantum_kernel_result['error']}"
    )


# ============================================================
# 14. DISPLAY METRICS ONLY IF COMPLETED
# ============================================================

if (
    quantum_kernel_result[
        "status"
    ]
    == "COMPLETED"
):

    metrics = (
        quantum_kernel_result[
            "metrics"
        ]
    )

    print()
    print("Held-out test metrics")
    print("-" * 70)

    print(
        f"Accuracy     : "
        f"{metrics['accuracy']:.4f}"
    )

    print(
        f"Precision    : "
        f"{metrics['precision']:.4f}"
    )

    print(
        f"Sensitivity  : "
        f"{metrics['recall_sensitivity']:.4f}"
    )

    print(
        f"Specificity  : "
        f"{metrics['specificity']:.4f}"
    )

    print(
        f"F1           : "
        f"{metrics['f1']:.4f}"
    )

    print(
        f"ROC-AUC      : "
        f"{metrics['roc_auc']:.4f}"
    )

    print()
    print(
        "Confusion matrix "
        "[[TN, FP], [FN, TP]]:"
    )

    print(
        quantum_kernel_result[
            "confusion_matrix"
        ]
    )

    print()
    print(
        "Kernel shapes:"
    )

    print(
        "K_train:",
        quantum_kernel_result[
            "K_train"
        ].shape,
    )

    print(
        "K_test :",
        quantum_kernel_result[
            "K_test"
        ].shape,
    )

    print()
    print(
        "Decision-function score range:"
    )

    print(
        f"min="
        f"{np.min(quantum_kernel_result['y_score']):.6f}, "
        f"max="
        f"{np.max(quantum_kernel_result['y_score']):.6f}"
    )

else:

    print()
    print(
        "No Quantum Kernel SVM metrics "
        "are reported because status is "
        f"{quantum_kernel_result['status']}."
    )


# ============================================================
# 15. SAVE KERNEL MATRICES IF COMPLETED
# ============================================================

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if (
    quantum_kernel_result[
        "status"
    ]
    == "COMPLETED"
):

    np.save(
        RESULTS_DIR
        / "quantum_kernel_train.npy",
        quantum_kernel_result[
            "K_train"
        ],
    )

    np.save(
        RESULTS_DIR
        / "quantum_kernel_test.npy",
        quantum_kernel_result[
            "K_test"
        ],
    )

    print()
    print(
        "Saved kernel matrices:"
    )

    print(
        RESULTS_DIR
        / "quantum_kernel_train.npy"
    )

    print(
        RESULTS_DIR
        / "quantum_kernel_test.npy"
    )


# ============================================================
# 16. SAVE JSON SUMMARY
# ============================================================

qk_summary = {
    "model": (
        "Quantum Kernel SVM"
    ),

    "status": (
        quantum_kernel_result[
            "status"
        ]
    ),

    "profile": PROFILE,

    "qubits": (
        QK_N_QUBITS
    ),

    "training_rows": (
        QK_TRAIN_SIZE
    ),

    "test_rows": (
        QK_TEST_SIZE
    ),

    "kernel_train_shape": (
        list(
            quantum_kernel_result[
                "K_train"
            ].shape
        )
        if quantum_kernel_result[
            "K_train"
        ]
        is not None
        else None
    ),

    "kernel_test_shape": (
        list(
            quantum_kernel_result[
                "K_test"
            ].shape
        )
        if quantum_kernel_result[
            "K_test"
        ]
        is not None
        else None
    ),

    "training_seconds": (
        quantum_kernel_result[
            "training_seconds"
        ]
    ),

    "kernel_train_seconds": (
        quantum_kernel_result[
            "kernel_train_seconds"
        ]
    ),

    "kernel_test_seconds": (
        quantum_kernel_result[
            "kernel_test_seconds"
        ]
    ),

    "svm_fit_seconds": (
        quantum_kernel_result[
            "svm_fit_seconds"
        ]
    ),

    "backend": (
        "PennyLane default.qubit "
        "classical simulation"
    ),

    "continuous_score": (
        "SVC.decision_function"
    ),

    "metrics": (
        quantum_kernel_result[
            "metrics"
        ]
    ),

    "error": (
        quantum_kernel_result[
            "error"
        ]
    ),
}

qk_summary_path = (
    RESULTS_DIR
    / "quantum_kernel_summary.json"
)

with open(
    qk_summary_path,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        qk_summary,
        handle,
        indent=2,
    )

print()
print(
    "Saved quantum-kernel summary:"
)

print(
    qk_summary_path
)


# ============================================================
# 17. FINAL SAFETY ASSERTIONS
# ============================================================

assert (
    quantum_kernel_result[
        "status"
    ]
    in {
        "COMPLETED",
        "TIMED OUT",
        "FAILED",
    }
)

if (
    quantum_kernel_result[
        "status"
    ]
    == "COMPLETED"
):

    assert (
        quantum_kernel_result[
            "metrics"
        ]
        is not None
    )

    assert (
        quantum_kernel_result[
            "K_train"
        ].shape
        ==
        (
            QK_TRAIN_SIZE,
            QK_TRAIN_SIZE,
        )
    )

    # Critical test:
    # test × train
    assert (
        quantum_kernel_result[
            "K_test"
        ].shape
        ==
        (
            QK_TEST_SIZE,
            QK_TRAIN_SIZE,
        )
    )

    assert (
        quantum_kernel_result[
            "confusion_matrix"
        ].shape
        == (2, 2)
    )

else:

    # No fabricated performance.
    assert (
        quantum_kernel_result[
            "metrics"
        ]
        is None
    )


print()
print("=" * 70)

if (
    quantum_kernel_result[
        "status"
    ]
    == "COMPLETED"
):

    print(
        "QUANTUM KERNEL MATRIX PASSED"
    )

    print(
        "QUANTUM KERNEL SVM COMPLETED"
    )

    print(
        "TEST x TRAIN KERNEL SHAPE PASSED"
    )

    print(
        "CONTINUOUS DECISION-FUNCTION ROC-AUC CHECK PASSED"
    )

elif (
    quantum_kernel_result[
        "status"
    ]
    == "TIMED OUT"
):

    print(
        "QUANTUM KERNEL SVM TIMED OUT CLEANLY"
    )

    print(
        "NO QUANTUM-KERNEL METRICS WERE FABRICATED"
    )

else:

    print(
        "QUANTUM KERNEL SVM FAILED CLEANLY"
    )

    print(
        "NO QUANTUM-KERNEL METRICS WERE FABRICATED"
    )

print("=" * 70)

Q-MedAI — Quantum Kernel SVM

Profile          : SAFE
Qubits           : 4
Training rows    : 100
Test rows        : 114
Timeout (s)      : 180

Backend          : PennyLane default.qubit (classical quantum-circuit simulation)

----------------------------------------------------------------------
Quantum kernel smoke test
----------------------------------------------------------------------
K(x0, x0) = 1.0000000000
K(x0, x1) = 0.6461275158

----------------------------------------------------------------------
Building quantum training kernel
----------------------------------------------------------------------
Train kernel row 001/100 | elapsed=0.5s
Train kernel row 010/100 | elapsed=4.4s
Train kernel row 020/100 | elapsed=8.4s
Train kernel row 030/100 | elapsed=11.8s
Train kernel row 040/100 | elapsed=14.8s
Train kernel row 050/100 | elapsed=17.4s
Train kernel row 060/100 | elapsed=19.5s
Train kernel row 070/100 | elapsed=21.1s
Train kernel row 080/100 | elapsed=22.2s
Train kernel

In [47]:
# ============================================================
# CELL 11 — FINAL FIVE-MODEL SIH COMPARISON
# Q-MedAI | SIH 2026 | PS 26139
# ============================================================

import json
import numpy as np
import pandas as pd

print("=" * 70)
print("Q-MedAI — Final Five-Model SIH Comparison")
print("=" * 70)


# ============================================================
# 1. HELPER TO CONVERT RESULT OBJECT INTO TABLE ROW
# ============================================================

def result_to_comparison_row(
    model_name,
    result,
    condition,
    training_rows,
    model_family,
):
    """
    Convert one model result into a comparison-table row.

    FAILED and TIMED OUT models remain visible.
    No metrics are fabricated.
    """

    row = {
        "Condition": condition,
        "Model": model_name,
        "Family": model_family,
        "Status": result.get(
            "status",
            "UNKNOWN",
        ),
        "Training Rows": int(
            training_rows
        ),
        "Accuracy": np.nan,
        "Precision": np.nan,
        "Sensitivity": np.nan,
        "Specificity": np.nan,
        "F1": np.nan,
        "ROC-AUC": np.nan,
        "Training Time (s)": result.get(
            "training_seconds",
            np.nan,
        ),
        "Error": result.get(
            "error",
            None,
        ),
    }

    metrics = result.get(
        "metrics",
        None,
    )

    if metrics is not None:
        row.update(
            {
                "Accuracy": metrics.get(
                    "accuracy",
                    np.nan,
                ),
                "Precision": metrics.get(
                    "precision",
                    np.nan,
                ),
                "Sensitivity": metrics.get(
                    "recall_sensitivity",
                    np.nan,
                ),
                "Specificity": metrics.get(
                    "specificity",
                    np.nan,
                ),
                "F1": metrics.get(
                    "f1",
                    np.nan,
                ),
                "ROC-AUC": metrics.get(
                    "roc_auc",
                    np.nan,
                ),
            }
        )

    return row


# ============================================================
# 2. BUILD MATCHED-DATA FIVE-MODEL COMPARISON
#
# FAIR comparison condition:
#
# All five models use the SAME 100 matched training rows.
# ============================================================

matched_rows = []


# ------------------------------------------------------------
# Logistic Regression
# ------------------------------------------------------------

matched_rows.append(
    result_to_comparison_row(
        model_name="Logistic Regression",
        result=classical_matched_results[
            "Logistic Regression"
        ],
        condition="MATCHED DATA",
        training_rows=len(
            y_train_match
        ),
        model_family="Classical",
    )
)


# ------------------------------------------------------------
# RBF SVM
# ------------------------------------------------------------

matched_rows.append(
    result_to_comparison_row(
        model_name="RBF SVM",
        result=classical_matched_results[
            "RBF SVM"
        ],
        condition="MATCHED DATA",
        training_rows=len(
            y_train_match
        ),
        model_family="Classical",
    )
)


# ------------------------------------------------------------
# Random Forest
# ------------------------------------------------------------

matched_rows.append(
    result_to_comparison_row(
        model_name="Random Forest",
        result=classical_matched_results[
            "Random Forest"
        ],
        condition="MATCHED DATA",
        training_rows=len(
            y_train_match
        ),
        model_family="Classical",
    )
)


# ------------------------------------------------------------
# VQC
# ------------------------------------------------------------

matched_rows.append(
    result_to_comparison_row(
        model_name="VQC",
        result=vqc_result,
        condition="MATCHED DATA",
        training_rows=len(
            y_train_match
        ),
        model_family="Quantum",
    )
)


# ------------------------------------------------------------
# Quantum Kernel SVM
# ------------------------------------------------------------

matched_rows.append(
    result_to_comparison_row(
        model_name="Quantum Kernel SVM",
        result=quantum_kernel_result,
        condition="MATCHED DATA",
        training_rows=len(
            y_train_match
        ),
        model_family="Quantum",
    )
)


matched_five_model_table = pd.DataFrame(
    matched_rows
)


# ============================================================
# 3. FULL-DATA CLASSICAL REFERENCE TABLE
#
# These models use all 455 training rows.
#
# They are useful as a performance ceiling/reference,
# but they are NOT used for direct quantum fairness claims.
# ============================================================

full_rows = []

for model_name in [
    "Logistic Regression",
    "RBF SVM",
    "Random Forest",
]:

    full_rows.append(
        result_to_comparison_row(
            model_name=model_name,
            result=classical_full_results[
                model_name
            ],
            condition="FULL DATA",
            training_rows=len(
                y_train
            ),
            model_family="Classical",
        )
    )


full_classical_reference = pd.DataFrame(
    full_rows
)


# ============================================================
# 4. DISPLAY CLEAN FIVE-MODEL FAIR COMPARISON
# ============================================================

display_columns = [
    "Model",
    "Family",
    "Status",
    "Training Rows",
    "Accuracy",
    "Precision",
    "Sensitivity",
    "Specificity",
    "F1",
    "ROC-AUC",
    "Training Time (s)",
]

matched_display = (
    matched_five_model_table[
        display_columns
    ]
    .copy()
)

metric_columns = [
    "Accuracy",
    "Precision",
    "Sensitivity",
    "Specificity",
    "F1",
    "ROC-AUC",
    "Training Time (s)",
]

matched_display[
    metric_columns
] = (
    matched_display[
        metric_columns
    ]
    .round(4)
)


print()
print("=" * 70)
print("FAIR MATCHED-DATA COMPARISON")
print(
    f"All models trained on "
    f"{len(y_train_match)} matched training rows"
)
print("=" * 70)

display(
    matched_display
)


# ============================================================
# 5. DISPLAY FULL-DATA CLASSICAL REFERENCE
# ============================================================

full_display = (
    full_classical_reference[
        display_columns
    ]
    .copy()
)

full_display[
    metric_columns
] = (
    full_display[
        metric_columns
    ]
    .round(4)
)

print()
print("=" * 70)
print("FULL-DATA CLASSICAL REFERENCE")
print(
    f"Classical models trained on "
    f"{len(y_train)} training rows"
)
print("=" * 70)

display(
    full_display
)


# ============================================================
# 6. VERIFY ALL REQUIRED MODEL ROWS EXIST
# ============================================================

required_models = {
    "Logistic Regression",
    "RBF SVM",
    "Random Forest",
    "VQC",
    "Quantum Kernel SVM",
}

actual_models = set(
    matched_five_model_table[
        "Model"
    ]
)

assert actual_models == required_models, (
    "Final comparison table does not "
    "contain exactly the five required models."
)


# ============================================================
# 7. STATUS REPORT
# ============================================================

print()
print("Model statuses")
print("-" * 70)

for _, row in (
    matched_five_model_table
    .iterrows()
):

    print(
        f"{row['Model']:<25} "
        f"{row['Status']}"
    )


# ============================================================
# 8. IDENTIFY BEST MATCHED CLASSICAL MODEL
#
# Use ROC-AUC as the primary ranking statistic because
# this is a binary medical classification task and ROC-AUC
# uses continuous discrimination scores.
# ============================================================

matched_classical_completed = (
    matched_five_model_table[
        (
            matched_five_model_table[
                "Family"
            ]
            == "Classical"
        )
        &
        (
            matched_five_model_table[
                "Status"
            ]
            == "COMPLETED"
        )
    ]
    .copy()
)

assert len(
    matched_classical_completed
) > 0

best_classical_index = (
    matched_classical_completed[
        "ROC-AUC"
    ]
    .astype(float)
    .idxmax()
)

best_classical = (
    matched_classical_completed
    .loc[
        best_classical_index
    ]
)

print()
print("=" * 70)
print("BEST MATCHED CLASSICAL REFERENCE")
print("=" * 70)

print(
    f"Model    : "
    f"{best_classical['Model']}"
)

print(
    f"Accuracy : "
    f"{best_classical['Accuracy']:.4f}"
)

print(
    f"F1       : "
    f"{best_classical['F1']:.4f}"
)

print(
    f"ROC-AUC  : "
    f"{best_classical['ROC-AUC']:.4f}"
)


# ============================================================
# 9. QUANTUM VS BEST-CLASSICAL DELTAS
#
# These are observed differences only.
#
# They are NOT evidence of quantum advantage.
# ============================================================

quantum_comparison_rows = []

for quantum_model in [
    "VQC",
    "Quantum Kernel SVM",
]:

    q_row = (
        matched_five_model_table[
            matched_five_model_table[
                "Model"
            ]
            == quantum_model
        ]
        .iloc[0]
    )

    comparison = {
        "Quantum Model": quantum_model,
        "Status": q_row[
            "Status"
        ],
        "Reference Classical Model": best_classical[
            "Model"
        ],
        "Accuracy Delta": np.nan,
        "F1 Delta": np.nan,
        "ROC-AUC Delta": np.nan,
    }

    if (
        q_row["Status"]
        == "COMPLETED"
    ):

        comparison.update(
            {
                "Accuracy Delta":
                    float(
                        q_row[
                            "Accuracy"
                        ]
                    )
                    -
                    float(
                        best_classical[
                            "Accuracy"
                        ]
                    ),

                "F1 Delta":
                    float(
                        q_row[
                            "F1"
                        ]
                    )
                    -
                    float(
                        best_classical[
                            "F1"
                        ]
                    ),

                "ROC-AUC Delta":
                    float(
                        q_row[
                            "ROC-AUC"
                        ]
                    )
                    -
                    float(
                        best_classical[
                            "ROC-AUC"
                        ]
                    ),
            }
        )

    quantum_comparison_rows.append(
        comparison
    )


quantum_vs_classical = pd.DataFrame(
    quantum_comparison_rows
)

for column in [
    "Accuracy Delta",
    "F1 Delta",
    "ROC-AUC Delta",
]:
    quantum_vs_classical[
        column
    ] = (
        quantum_vs_classical[
            column
        ]
        .round(6)
    )


print()
print("=" * 70)
print("QUANTUM VS MATCHED CLASSICAL REFERENCE")
print("=" * 70)

display(
    quantum_vs_classical
)


# ============================================================
# 10. SPECIAL QSVM SUMMARY
# ============================================================

if (
    quantum_kernel_result[
        "status"
    ]
    == "COMPLETED"
):

    qk_metrics = (
        quantum_kernel_result[
            "metrics"
        ]
    )

    print()
    print(
        "Quantum Kernel SVM verified Kaggle result:"
    )

    print(
        f"Accuracy    : "
        f"{qk_metrics['accuracy']:.4f}"
    )

    print(
        f"F1          : "
        f"{qk_metrics['f1']:.4f}"
    )

    print(
        f"ROC-AUC     : "
        f"{qk_metrics['roc_auc']:.4f}"
    )

    print(
        "K_train     :",
        quantum_kernel_result[
            "K_train"
        ].shape,
    )

    print(
        "K_test      :",
        quantum_kernel_result[
            "K_test"
        ].shape,
    )

    assert (
        quantum_kernel_result[
            "K_train"
        ].shape
        ==
        (
            len(
                y_train_match
            ),
            len(
                y_train_match
            ),
        )
    )

    assert (
        quantum_kernel_result[
            "K_test"
        ].shape
        ==
        (
            len(
                y_test
            ),
            len(
                y_train_match
            ),
        )
    )


# ============================================================
# 11. SPECIAL VQC SUMMARY
# ============================================================

if (
    vqc_result[
        "status"
    ]
    == "COMPLETED"
):

    vqc_metrics = (
        vqc_result[
            "metrics"
        ]
    )

    print()
    print(
        "VQC verified Kaggle result:"
    )

    print(
        f"Accuracy    : "
        f"{vqc_metrics['accuracy']:.4f}"
    )

    print(
        f"F1          : "
        f"{vqc_metrics['f1']:.4f}"
    )

    print(
        f"ROC-AUC     : "
        f"{vqc_metrics['roc_auc']:.4f}"
    )

    print(
        f"Iterations  : "
        f"{vqc_result['iterations_completed']}/"
        f"{vqc_result['iterations_requested']}"
    )

    print(
        f"Train time  : "
        f"{vqc_result['training_seconds']:.3f}s"
    )


# ============================================================
# 12. SAVE FINAL COMPARISON TABLES
# ============================================================

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

matched_csv = (
    RESULTS_DIR
    / "final_matched_five_model_comparison.csv"
)

full_csv = (
    RESULTS_DIR
    / "final_full_classical_reference.csv"
)

delta_csv = (
    RESULTS_DIR
    / "quantum_vs_classical_deltas.csv"
)


matched_five_model_table.to_csv(
    matched_csv,
    index=False,
)

full_classical_reference.to_csv(
    full_csv,
    index=False,
)

quantum_vs_classical.to_csv(
    delta_csv,
    index=False,
)


print()
print("Saved:")
print(matched_csv)
print(full_csv)
print(delta_csv)


# ============================================================
# 13. SAVE FINAL JSON SUMMARY
# ============================================================

def safe_number(value):

    if value is None:
        return None

    try:
        value = float(
            value
        )

        if np.isnan(
            value
        ):
            return None

        return value

    except Exception:
        return None


final_model_records = []

for _, row in (
    matched_five_model_table
    .iterrows()
):

    final_model_records.append(
        {
            "model": row[
                "Model"
            ],

            "family": row[
                "Family"
            ],

            "status": row[
                "Status"
            ],

            "training_rows": int(
                row[
                    "Training Rows"
                ]
            ),

            "accuracy": safe_number(
                row[
                    "Accuracy"
                ]
            ),

            "precision": safe_number(
                row[
                    "Precision"
                ]
            ),

            "sensitivity": safe_number(
                row[
                    "Sensitivity"
                ]
            ),

            "specificity": safe_number(
                row[
                    "Specificity"
                ]
            ),

            "f1": safe_number(
                row[
                    "F1"
                ]
            ),

            "roc_auc": safe_number(
                row[
                    "ROC-AUC"
                ]
            ),

            "training_seconds": safe_number(
                row[
                    "Training Time (s)"
                ]
            ),

            "error": (
                None
                if pd.isna(
                    row[
                        "Error"
                    ]
                )
                else str(
                    row[
                        "Error"
                    ]
                )
            ),
        }
    )


final_comparison_summary = {
    "project": "Q-MedAI",

    "competition": (
        "Smart India Hackathon 2026"
    ),

    "problem_statement": (
        "PS 26139"
    ),

    "profile": PROFILE,

    "dataset": {
        "samples": int(
            len(
                X_raw
            )
        ),

        "usable_features": int(
            X_raw.shape[1]
        ),

        "training_rows_full": int(
            len(
                y_train
            )
        ),

        "training_rows_matched": int(
            len(
                y_train_match
            )
        ),

        "test_rows": int(
            len(
                y_test
            )
        ),
    },

    "preprocessing": {
        "method": PREPROCESSING,

        "feature_count": int(
            CFG[
                "feature_count"
            ]
        ),

        "selected_features": (
            matched_preprocessor
            .feature_names_out
        ),

        "leakage_safe": True,
    },

    "random_seed": int(
        RANDOM_SEED
    ),

    "quantum_backend": (
        "PennyLane default.qubit "
        "classical simulation"
    ),

    "models": (
        final_model_records
    ),

    "best_matched_classical": {
        "model": best_classical[
            "Model"
        ],

        "accuracy": safe_number(
            best_classical[
                "Accuracy"
            ]
        ),

        "f1": safe_number(
            best_classical[
                "F1"
            ]
        ),

        "roc_auc": safe_number(
            best_classical[
                "ROC-AUC"
            ]
        ),
    },

    "quantum_advantage_claimed": False,

    "research_disclaimer": (
        "This software is a research prototype "
        "and is not a medical diagnostic device."
    ),
}


final_json_path = (
    RESULTS_DIR
    / "final_sih_summary.json"
)

with open(
    final_json_path,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        final_comparison_summary,
        handle,
        indent=2,
    )


print(
    final_json_path
)


# ============================================================
# 14. FINAL METHODOLOGY STATEMENTS
# ============================================================

print()
print("=" * 70)
print("INTERPRETATION")
print("=" * 70)

print(
    "1. FULL DATA results show classical performance "
    "when all 455 training rows are available."
)

print(
    "2. MATCHED DATA results provide the fair "
    "classical-vs-quantum comparison because all five "
    f"models use the same {len(y_train_match)} training rows."
)

print(
    "3. ROC-AUC is calculated from continuous model scores."
)

print(
    "4. Quantum Kernel SVM evaluation uses a "
    "test × train kernel, never a test × test fitting matrix."
)

print(
    "5. PennyLane default.qubit is a classical simulator "
    "of quantum circuits, not physical quantum hardware."
)

print(
    "6. Predictive performance alone does not establish "
    "quantum advantage."
)

print()
print(
    "Medical disclaimer:"
)
print(
    "This software is a research prototype and is not "
    "a medical diagnostic device."
)


# ============================================================
# 15. FINAL ASSERTIONS
# ============================================================

assert len(
    matched_five_model_table
) == 5

assert set(
    matched_five_model_table[
        "Status"
    ]
).issubset(
    {
        "COMPLETED",
        "FAILED",
        "TIMED OUT",
    }
)

assert (
    final_comparison_summary[
        "quantum_advantage_claimed"
    ]
    is False
)

print()
print("=" * 70)
print("FIVE-MODEL COMPARISON TABLE PASSED")
print("FAIR MATCHED-DATA COMPARISON PASSED")
print("FINAL SIH SUMMARY SAVED")
print("=" * 70)

Q-MedAI — Final Five-Model SIH Comparison

FAIR MATCHED-DATA COMPARISON
All models trained on 100 matched training rows


,Model,Family,Status,Training Rows,Accuracy,Precision,Sensitivity,Specificity,F1,ROC-AUC,Training Time (s)
0,Logistic Regression,Classical,COMPLETED,100,0.9561,0.9512,0.9286,0.9722,0.9398,0.9960,0.0037
1,RBF SVM,Classical,COMPLETED,100,0.9474,0.9737,0.8810,0.9861,0.9250,0.9957,0.0031
2,Random Forest,Classical,COMPLETED,100,0.9298,0.9722,0.8333,0.9861,0.8974,0.9901,0.6164
3,VQC,Quantum,COMPLETED,100,0.8596,0.9643,0.6429,0.9861,0.7714,0.9401,29.7464
4,Quantum Kernel SVM,Quantum,COMPLETED,100,0.9474,0.9500,0.9048,0.9722,0.9268,0.9944,75.7601



FULL-DATA CLASSICAL REFERENCE
Classical models trained on 455 training rows


,Model,Family,Status,Training Rows,Accuracy,Precision,Sensitivity,Specificity,F1,ROC-AUC,Training Time (s)
0,Logistic Regression,Classical,COMPLETED,455,0.9561,0.9512,0.9286,0.9722,0.9398,0.9954,0.0059
1,RBF SVM,Classical,COMPLETED,455,0.9386,0.9730,0.8571,0.9861,0.9114,0.9954,0.0146
2,Random Forest,Classical,COMPLETED,455,0.9298,0.9250,0.8810,0.9583,0.9024,0.9912,0.6272



Model statuses
----------------------------------------------------------------------
Logistic Regression       COMPLETED
RBF SVM                   COMPLETED
Random Forest             COMPLETED
VQC                       COMPLETED
Quantum Kernel SVM        COMPLETED

BEST MATCHED CLASSICAL REFERENCE
Model    : Logistic Regression
Accuracy : 0.9561
F1       : 0.9398
ROC-AUC  : 0.9960

QUANTUM VS MATCHED CLASSICAL REFERENCE


,Quantum Model,Status,Reference Classical Model,Accuracy Delta,F1 Delta,ROC-AUC Delta
0,VQC,COMPLETED,Logistic Regression,-0.096491,-0.16833,-0.055886
1,Quantum Kernel SVM,COMPLETED,Logistic Regression,-0.008772,-0.01293,-0.001653



Quantum Kernel SVM verified Kaggle result:
Accuracy    : 0.9474
F1          : 0.9268
ROC-AUC     : 0.9944
K_train     : (100, 100)
K_test      : (114, 100)

VQC verified Kaggle result:
Accuracy    : 0.8596
F1          : 0.7714
ROC-AUC     : 0.9401
Iterations  : 30/30
Train time  : 29.746s

Saved:
/kaggle/working/Q-MedAI/results/final_matched_five_model_comparison.csv
/kaggle/working/Q-MedAI/results/final_full_classical_reference.csv
/kaggle/working/Q-MedAI/results/quantum_vs_classical_deltas.csv
/kaggle/working/Q-MedAI/results/final_sih_summary.json

INTERPRETATION
1. FULL DATA results show classical performance when all 455 training rows are available.
2. MATCHED DATA results provide the fair classical-vs-quantum comparison because all five models use the same 100 training rows.
3. ROC-AUC is calculated from continuous model scores.
4. Quantum Kernel SVM evaluation uses a test × train kernel, never a test × test fitting matrix.
5. PennyLane default.qubit is a classical simulator of q

In [48]:
# ============================================================
# CELL 12 — JUDGE-READY VISUALS + EXPLAINABILITY
# Q-MedAI | SIH 2026 | PS 26139
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_curve, roc_auc_score

print("=" * 70)
print("Q-MedAI — Judge-Ready Visual Evidence")
print("=" * 70)


# ============================================================
# 1. OUTPUT DIRECTORY
# ============================================================

FIGURES_DIR = (
    RESULTS_DIR
    / "figures"
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print()
print(
    "Figures directory:"
)
print(
    FIGURES_DIR
)


# ============================================================
# 2. SELECTED BIOMARKER NAMES
# ============================================================

selected_features = list(
    matched_preprocessor
    .feature_names_out
)

assert len(
    selected_features
) == CFG["feature_count"]

print()
print(
    "Selected biomarkers:"
)

for index, feature in enumerate(
    selected_features,
    start=1,
):
    print(
        f"{index}. {feature}"
    )


# ============================================================
# 3. COLLECT ALL FIVE MATCHED MODEL RESULTS
# ============================================================

matched_model_results = {
    "Logistic Regression":
        classical_matched_results[
            "Logistic Regression"
        ],

    "RBF SVM":
        classical_matched_results[
            "RBF SVM"
        ],

    "Random Forest":
        classical_matched_results[
            "Random Forest"
        ],

    "VQC":
        vqc_result,

    "Quantum Kernel SVM":
        quantum_kernel_result,
}


# ------------------------------------------------------------
# Ensure all five models completed.
# ------------------------------------------------------------

for model_name, result in (
    matched_model_results.items()
):

    assert (
        result["status"]
        == "COMPLETED"
    ), (
        f"{model_name} is not COMPLETED: "
        f"{result['status']}"
    )

    assert (
        result["metrics"]
        is not None
    )

    assert (
        result["confusion_matrix"]
        is not None
    )

    assert (
        result["y_score"]
        is not None
    )


print()
print(
    "All five matched models are available."
)


# ============================================================
# 4. FIGURE REGISTRY
# ============================================================

figure_records = []


def register_figure(
    filename,
    category,
    description,
):

    figure_records.append(
        {
            "filename": filename,
            "category": category,
            "description": description,
        }
    )


# ============================================================
# 5. FIVE-MODEL ROC CURVE
# ============================================================

plt.figure(
    figsize=(9, 7)
)

for model_name, result in (
    matched_model_results.items()
):

    y_score = np.asarray(
        result["y_score"],
        dtype=float,
    )

    fpr, tpr, _ = roc_curve(
        y_test,
        y_score,
    )

    auc_value = roc_auc_score(
        y_test,
        y_score,
    )

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=(
            f"{model_name} "
            f"(AUC={auc_value:.4f})"
        ),
    )


# Random classifier reference
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Random classifier",
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "Q-MedAI — Matched-Data ROC Comparison\n"
    "All models trained on the same 100 training samples"
)

plt.legend(
    loc="lower right"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()

roc_path = (
    FIGURES_DIR
    / "roc_matched_five_models.png"
)

plt.savefig(
    roc_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

register_figure(
    roc_path.name,
    "Performance",
    (
        "ROC curves for all five matched-data models. "
        "ROC-AUC uses continuous scores."
    ),
)


# ============================================================
# 6. CONFUSION MATRIX FUNCTION
# ============================================================

def plot_confusion_matrix(
    cm,
    model_name,
    filename,
):

    cm = np.asarray(
        cm,
        dtype=int,
    )

    assert cm.shape == (
        2,
        2,
    )

    plt.figure(
        figsize=(5.5, 5)
    )

    plt.imshow(
        cm
    )

    plt.xticks(
        [0, 1],
        [
            "Benign (0)",
            "Malignant (1)",
        ],
    )

    plt.yticks(
        [0, 1],
        [
            "Benign (0)",
            "Malignant (1)",
        ],
    )

    plt.xlabel(
        "Predicted label"
    )

    plt.ylabel(
        "True label"
    )

    plt.title(
        f"{model_name}\nConfusion Matrix"
    )

    for i in range(2):
        for j in range(2):

            plt.text(
                j,
                i,
                str(
                    cm[i, j]
                ),
                ha="center",
                va="center",
                fontsize=16,
            )

    plt.tight_layout()

    output_path = (
        FIGURES_DIR
        / filename
    )

    plt.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()

    return output_path


# ============================================================
# 7. SAVE ALL FIVE CONFUSION MATRICES
# ============================================================

safe_filename_map = {
    "Logistic Regression":
        "confusion_logistic_regression.png",

    "RBF SVM":
        "confusion_rbf_svm.png",

    "Random Forest":
        "confusion_random_forest.png",

    "VQC":
        "confusion_vqc.png",

    "Quantum Kernel SVM":
        "confusion_quantum_kernel_svm.png",
}


for model_name, result in (
    matched_model_results.items()
):

    cm_path = (
        plot_confusion_matrix(
            cm=result[
                "confusion_matrix"
            ],
            model_name=model_name,
            filename=safe_filename_map[
                model_name
            ],
        )
    )

    register_figure(
        cm_path.name,
        "Performance",
        (
            f"Held-out confusion matrix "
            f"for {model_name}."
        ),
    )


# ============================================================
# 8. VQC LOSS CURVE
# ============================================================

vqc_losses = np.asarray(
    vqc_result[
        "loss_history"
    ],
    dtype=float,
)

assert len(
    vqc_losses
) == vqc_result[
    "iterations_completed"
]


plt.figure(
    figsize=(9, 5.5)
)

plt.plot(
    np.arange(
        1,
        len(vqc_losses) + 1,
    ),
    vqc_losses,
    marker="o",
)

plt.xlabel(
    "Training iteration"
)

plt.ylabel(
    "Mini-batch binary cross-entropy"
)

plt.title(
    "VQC Training Loss\n"
    f"{VQC_LAYERS} layers, "
    f"{N_QUBITS} qubits, "
    f"{len(y_train_match)} matched samples"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()

vqc_loss_path = (
    FIGURES_DIR
    / "vqc_training_loss.png"
)

plt.savefig(
    vqc_loss_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

register_figure(
    vqc_loss_path.name,
    "Quantum Training",
    (
        "Observed VQC mini-batch binary "
        "cross-entropy during Kaggle training."
    ),
)


# ============================================================
# 9. VQC GRADIENT-NORM CURVE
# ============================================================

vqc_gradient_norms = np.asarray(
    vqc_result[
        "gradient_norm_history"
    ],
    dtype=float,
)

assert len(
    vqc_gradient_norms
) == len(
    vqc_losses
)


plt.figure(
    figsize=(9, 5.5)
)

plt.plot(
    np.arange(
        1,
        len(
            vqc_gradient_norms
        ) + 1,
    ),
    vqc_gradient_norms,
    marker="o",
)

plt.xlabel(
    "Training iteration"
)

plt.ylabel(
    "Gradient norm"
)

plt.title(
    "VQC Optimization Gradient Norm"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()

gradient_path = (
    FIGURES_DIR
    / "vqc_gradient_norm.png"
)

plt.savefig(
    gradient_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

register_figure(
    gradient_path.name,
    "Quantum Training",
    (
        "Observed VQC parameter-gradient norm "
        "during optimization."
    ),
)


# ============================================================
# 10. CLASSICAL FEATURE IMPORTANCE
#
# Use model-appropriate explanation:
#
# Logistic Regression:
#     absolute standardized coefficient
#
# Random Forest:
#     impurity-based feature importance
#
# RBF SVM:
#     permutation importance
#
# ============================================================

logistic_model = (
    classical_matched_results[
        "Logistic Regression"
    ][
        "estimator"
    ]
)

rbf_model = (
    classical_matched_results[
        "RBF SVM"
    ][
        "estimator"
    ]
)

rf_model = (
    classical_matched_results[
        "Random Forest"
    ][
        "estimator"
    ]
)


# ============================================================
# 11. LOGISTIC REGRESSION IMPORTANCE
# ============================================================

logistic_importance = np.abs(
    np.asarray(
        logistic_model.coef_[0],
        dtype=float,
    )
)

logistic_df = pd.DataFrame(
    {
        "Feature": selected_features,
        "Importance": logistic_importance,
    }
).sort_values(
    "Importance",
    ascending=True,
)


plt.figure(
    figsize=(8, 5.5)
)

plt.barh(
    logistic_df[
        "Feature"
    ],
    logistic_df[
        "Importance"
    ],
)

plt.xlabel(
    "Absolute standardized coefficient"
)

plt.title(
    "Logistic Regression Feature Importance"
)

plt.tight_layout()

logistic_importance_path = (
    FIGURES_DIR
    / "importance_logistic_regression.png"
)

plt.savefig(
    logistic_importance_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

register_figure(
    logistic_importance_path.name,
    "Explainability",
    (
        "Logistic Regression importance based "
        "on absolute standardized coefficients."
    ),
)


# ============================================================
# 12. RANDOM FOREST IMPORTANCE
# ============================================================

rf_importance = np.asarray(
    rf_model.feature_importances_,
    dtype=float,
)

rf_df = pd.DataFrame(
    {
        "Feature": selected_features,
        "Importance": rf_importance,
    }
).sort_values(
    "Importance",
    ascending=True,
)


plt.figure(
    figsize=(8, 5.5)
)

plt.barh(
    rf_df[
        "Feature"
    ],
    rf_df[
        "Importance"
    ],
)

plt.xlabel(
    "Feature importance"
)

plt.title(
    "Random Forest Feature Importance"
)

plt.tight_layout()

rf_importance_path = (
    FIGURES_DIR
    / "importance_random_forest.png"
)

plt.savefig(
    rf_importance_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

register_figure(
    rf_importance_path.name,
    "Explainability",
    (
        "Random Forest native feature importance."
    ),
)


# ============================================================
# 13. RBF SVM PERMUTATION IMPORTANCE
#
# We use the matched TRAINING representation for this
# interpretability analysis so the held-out test metrics
# remain untouched.
# ============================================================

rbf_perm = permutation_importance(
    rbf_model,
    X_train_match_classical,
    np.asarray(
        y_train_match,
        dtype=int,
    ),
    scoring="roc_auc",
    n_repeats=20,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

rbf_df = pd.DataFrame(
    {
        "Feature": selected_features,
        "Importance": (
            rbf_perm.importances_mean
        ),
        "Std": (
            rbf_perm.importances_std
        ),
    }
).sort_values(
    "Importance",
    ascending=True,
)


plt.figure(
    figsize=(8, 5.5)
)

plt.barh(
    rbf_df[
        "Feature"
    ],
    rbf_df[
        "Importance"
    ],
    xerr=rbf_df[
        "Std"
    ],
)

plt.xlabel(
    "Decrease in ROC-AUC after permutation"
)

plt.title(
    "RBF SVM Permutation Importance"
)

plt.tight_layout()

rbf_importance_path = (
    FIGURES_DIR
    / "importance_rbf_svm_permutation.png"
)

plt.savefig(
    rbf_importance_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

register_figure(
    rbf_importance_path.name,
    "Explainability",
    (
        "RBF SVM permutation importance measured "
        "using matched training data."
    ),
)


# ============================================================
# 14. VQC PERTURBATION-BASED MODEL SENSITIVITY
#
# This is NOT conventional feature importance.
#
# For each selected quantum feature:
#
# 1. measure baseline VQC score
# 2. perturb the feature by +delta
# 3. perturb the feature by -delta
# 4. calculate mean absolute score change
#
# The interpretation is therefore:
#
#     Perturbation-based model sensitivity
#
# not:
#
#     Feature importance
#
# ============================================================

assert (
    vqc_result[
        "final_weights"
    ]
    is not None
)

final_vqc_weights = pnp.array(
    np.asarray(
        vqc_result[
            "final_weights"
        ],
        dtype=float,
    ),
    requires_grad=False,
)

VQC_SENSITIVITY_DELTA = 0.10


# Use all 100 matched training samples.
X_vqc_sensitivity = np.asarray(
    X_train_match_quantum,
    dtype=float,
)


def predict_vqc_scores_fixed(
    X,
):
    values = []

    for row in np.asarray(
        X,
        dtype=float,
    ):

        row_tensor = pnp.array(
            row,
            requires_grad=False,
        )

        score = float(
            vqc_probability(
                row_tensor,
                final_vqc_weights,
            )
        )

        values.append(
            score
        )

    return np.asarray(
        values,
        dtype=float,
    )


print()
print(
    "Calculating VQC perturbation-based sensitivity..."
)

baseline_vqc_scores = (
    predict_vqc_scores_fixed(
        X_vqc_sensitivity
    )
)

vqc_sensitivity_values = []


for feature_index in range(
    N_QUBITS
):

    X_plus = (
        X_vqc_sensitivity.copy()
    )

    X_minus = (
        X_vqc_sensitivity.copy()
    )


    X_plus[
        :,
        feature_index,
    ] = np.clip(
        X_plus[
            :,
            feature_index
        ]
        + VQC_SENSITIVITY_DELTA,
        -np.pi,
        np.pi,
    )


    X_minus[
        :,
        feature_index,
    ] = np.clip(
        X_minus[
            :,
            feature_index
        ]
        - VQC_SENSITIVITY_DELTA,
        -np.pi,
        np.pi,
    )


    plus_scores = (
        predict_vqc_scores_fixed(
            X_plus
        )
    )

    minus_scores = (
        predict_vqc_scores_fixed(
            X_minus
        )
    )


    plus_change = np.abs(
        plus_scores
        - baseline_vqc_scores
    )

    minus_change = np.abs(
        minus_scores
        - baseline_vqc_scores
    )


    sensitivity = float(
        np.mean(
            (
                plus_change
                +
                minus_change
            )
            / 2.0
        )
    )

    vqc_sensitivity_values.append(
        sensitivity
    )


vqc_sensitivity_df = pd.DataFrame(
    {
        "Feature": selected_features,
        "Sensitivity": (
            vqc_sensitivity_values
        ),
    }
).sort_values(
    "Sensitivity",
    ascending=True,
)


plt.figure(
    figsize=(8, 5.5)
)

plt.barh(
    vqc_sensitivity_df[
        "Feature"
    ],
    vqc_sensitivity_df[
        "Sensitivity"
    ],
)

plt.xlabel(
    "Mean absolute change in VQC score"
)

plt.title(
    "VQC Perturbation-Based Model Sensitivity\n"
    f"±{VQC_SENSITIVITY_DELTA:.2f} rad perturbation"
)

plt.tight_layout()

vqc_sensitivity_path = (
    FIGURES_DIR
    / "vqc_perturbation_sensitivity.png"
)

plt.savefig(
    vqc_sensitivity_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

register_figure(
    vqc_sensitivity_path.name,
    "Explainability",
    (
        "VQC perturbation-based model sensitivity. "
        "This is not presented as conventional "
        "feature importance."
    ),
)


print()
print(
    "VQC perturbation sensitivity:"
)

display(
    vqc_sensitivity_df
    .sort_values(
        "Sensitivity",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 15. QUANTUM KERNEL MATRIX VISUALIZATION
# ============================================================

K_train_final = np.asarray(
    quantum_kernel_result[
        "K_train"
    ],
    dtype=float,
)

assert K_train_final.shape == (
    len(
        y_train_match
    ),
    len(
        y_train_match
    ),
)


plt.figure(
    figsize=(7, 6)
)

plt.imshow(
    K_train_final,
    aspect="auto",
)

plt.xlabel(
    "Training sample index"
)

plt.ylabel(
    "Training sample index"
)

plt.title(
    "Quantum Fidelity Kernel Matrix\n"
    "100 × 100 matched training set"
)

plt.colorbar(
    label="Quantum-kernel similarity"
)

plt.tight_layout()

kernel_heatmap_path = (
    FIGURES_DIR
    / "quantum_kernel_train_matrix.png"
)

plt.savefig(
    kernel_heatmap_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

register_figure(
    kernel_heatmap_path.name,
    "Quantum Model",
    (
        "100 × 100 fidelity-based quantum "
        "training-kernel matrix."
    ),
)


# ============================================================
# 16. SAVE EXPLAINABILITY NUMBERS
# ============================================================

logistic_df.to_csv(
    RESULTS_DIR
    / "explainability_logistic_regression.csv",
    index=False,
)

rf_df.to_csv(
    RESULTS_DIR
    / "explainability_random_forest.csv",
    index=False,
)

rbf_df.to_csv(
    RESULTS_DIR
    / "explainability_rbf_svm.csv",
    index=False,
)

vqc_sensitivity_df.to_csv(
    RESULTS_DIR
    / "explainability_vqc_sensitivity.csv",
    index=False,
)


# ============================================================
# 17. SAVE FIGURE MANIFEST
# ============================================================

figure_manifest = pd.DataFrame(
    figure_records
)

figure_manifest_path = (
    RESULTS_DIR
    / "figure_manifest.csv"
)

figure_manifest.to_csv(
    figure_manifest_path,
    index=False,
)


# ============================================================
# 18. PERFORMANCE SUMMARY FOR JUDGES
# ============================================================

judge_summary = (
    matched_five_model_table[
        [
            "Model",
            "Family",
            "Status",
            "Training Rows",
            "Accuracy",
            "Sensitivity",
            "Specificity",
            "F1",
            "ROC-AUC",
            "Training Time (s)",
        ]
    ]
    .copy()
)

judge_summary = (
    judge_summary
    .sort_values(
        "ROC-AUC",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 70)
print("JUDGE-FACING MATCHED-DATA SUMMARY")
print("=" * 70)

display(
    judge_summary
    .round(4)
)


# ============================================================
# 19. TOP OBSERVED RESULTS
# ============================================================

best_auc_row = (
    judge_summary
    .iloc[0]
)

best_accuracy_index = (
    judge_summary[
        "Accuracy"
    ]
    .astype(float)
    .idxmax()
)

best_accuracy_row = (
    judge_summary
    .loc[
        best_accuracy_index
    ]
)


print()
print(
    "Highest observed matched-data ROC-AUC:"
)

print(
    f"{best_auc_row['Model']} "
    f"= {best_auc_row['ROC-AUC']:.4f}"
)


print()
print(
    "Highest observed matched-data accuracy:"
)

print(
    f"{best_accuracy_row['Model']} "
    f"= {best_accuracy_row['Accuracy']:.4f}"
)


# ============================================================
# 20. VERIFIED QUANTUM RESULTS
# ============================================================

print()
print("=" * 70)
print("VERIFIED QUANTUM RESULTS")
print("=" * 70)

print(
    "Quantum Kernel SVM:"
)

print(
    f"  Accuracy : "
    f"{quantum_kernel_result['metrics']['accuracy']:.4f}"
)

print(
    f"  F1       : "
    f"{quantum_kernel_result['metrics']['f1']:.4f}"
)

print(
    f"  ROC-AUC  : "
    f"{quantum_kernel_result['metrics']['roc_auc']:.4f}"
)

print(
    f"  Time     : "
    f"{quantum_kernel_result['training_seconds']:.3f}s"
)

print(
    "  K_train  : "
    f"{quantum_kernel_result['K_train'].shape}"
)

print(
    "  K_test   : "
    f"{quantum_kernel_result['K_test'].shape}"
)


print()
print(
    "Variational Quantum Classifier:"
)

print(
    f"  Accuracy : "
    f"{vqc_result['metrics']['accuracy']:.4f}"
)

print(
    f"  F1       : "
    f"{vqc_result['metrics']['f1']:.4f}"
)

print(
    f"  ROC-AUC  : "
    f"{vqc_result['metrics']['roc_auc']:.4f}"
)

print(
    f"  Time     : "
    f"{vqc_result['training_seconds']:.3f}s"
)

print(
    f"  Iterations: "
    f"{vqc_result['iterations_completed']}/"
    f"{vqc_result['iterations_requested']}"
)


# ============================================================
# 21. IMPORTANT INTERPRETATION
# ============================================================

print()
print("=" * 70)
print("INTERPRETATION GUARDRAILS")
print("=" * 70)

print(
    "• All five matched-data models use the same "
    "100 training observations."
)

print(
    "• All metrics are evaluated on the same "
    "114 held-out test observations."
)

print(
    "• ROC-AUC uses continuous scores."
)

print(
    "• The Quantum Kernel SVM uses "
    "K_test = test × train."
)

print(
    "• VQC sensitivity is perturbation-based "
    "model sensitivity, not conventional "
    "feature importance."
)

print(
    "• PennyLane default.qubit is a classical "
    "simulator of quantum circuits."
)

print(
    "• These results do not demonstrate "
    "quantum advantage."
)

print(
    "• Q-MedAI is a research prototype, "
    "not a medical diagnostic device."
)


# ============================================================
# 22. FINAL ASSERTIONS
# ============================================================

assert len(
    matched_model_results
) == 5

assert len(
    figure_manifest
) >= 10

assert (
    quantum_kernel_result[
        "K_test"
    ].shape
    ==
    (
        len(
            y_test
        ),
        len(
            y_train_match
        ),
    )
)

assert np.all(
    np.isfinite(
        vqc_sensitivity_values
    )
)


print()
print(
    "Saved figure manifest:"
)

print(
    figure_manifest_path
)

print()
print(
    "Generated figures:"
)

for record in (
    figure_records
):

    print(
        " -",
        FIGURES_DIR
        / record[
            "filename"
        ],
    )


print()
print("=" * 70)
print("ROC VISUALIZATION PASSED")
print("CONFUSION-MATRIX VISUALIZATIONS PASSED")
print("VQC TRAINING VISUALIZATION PASSED")
print("CLASSICAL EXPLAINABILITY PASSED")
print("VQC PERTURBATION SENSITIVITY PASSED")
print("QUANTUM KERNEL VISUALIZATION PASSED")
print("JUDGE-READY FIGURES SAVED")
print("=" * 70)

Q-MedAI — Judge-Ready Visual Evidence

Figures directory:
/kaggle/working/Q-MedAI/results/figures

Selected biomarkers:
1. concave points_mean
2. radius_worst
3. perimeter_worst
4. concave points_worst

All five matched models are available.


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration o


Calculating VQC perturbation-based sensitivity...

VQC perturbation sensitivity:


,Feature,Sensitivity
0,radius_worst,0.009638
1,concave points_worst,0.009502
2,concave points_mean,0.008318
3,perimeter_worst,0.007028



JUDGE-FACING MATCHED-DATA SUMMARY


,Model,Family,Status,Training Rows,Accuracy,Sensitivity,Specificity,F1,ROC-AUC,Training Time (s)
0,Logistic Regression,Classical,COMPLETED,100,0.9561,0.9286,0.9722,0.9398,0.9960,0.0037
1,RBF SVM,Classical,COMPLETED,100,0.9474,0.8810,0.9861,0.9250,0.9957,0.0031
2,Quantum Kernel SVM,Quantum,COMPLETED,100,0.9474,0.9048,0.9722,0.9268,0.9944,75.7601
3,Random Forest,Classical,COMPLETED,100,0.9298,0.8333,0.9861,0.8974,0.9901,0.6164
4,VQC,Quantum,COMPLETED,100,0.8596,0.6429,0.9861,0.7714,0.9401,29.7464



Highest observed matched-data ROC-AUC:
Logistic Regression = 0.9960

Highest observed matched-data accuracy:
Logistic Regression = 0.9561

VERIFIED QUANTUM RESULTS
Quantum Kernel SVM:
  Accuracy : 0.9474
  F1       : 0.9268
  ROC-AUC  : 0.9944
  Time     : 75.760s
  K_train  : (100, 100)
  K_test   : (114, 100)

Variational Quantum Classifier:
  Accuracy : 0.8596
  F1       : 0.7714
  ROC-AUC  : 0.9401
  Time     : 29.746s
  Iterations: 30/30

INTERPRETATION GUARDRAILS
• All five matched-data models use the same 100 training observations.
• All metrics are evaluated on the same 114 held-out test observations.
• ROC-AUC uses continuous scores.
• The Quantum Kernel SVM uses K_test = test × train.
• VQC sensitivity is perturbation-based model sensitivity, not conventional feature importance.
• PennyLane default.qubit is a classical simulator of quantum circuits.
• These results do not demonstrate quantum advantage.
• Q-MedAI is a research prototype, not a medical diagnostic device.

Save

In [59]:
# ============================================================
# CELL 13 — COLLEGE JUDGE PROGRESS SUMMARY
# Q-MedAI | SIH26139
# ============================================================

from IPython.display import display, HTML
import numpy as np

print("=" * 76)
print("Q-MedAI — COLLEGE JUDGE EVIDENCE SUMMARY")
print("=" * 76)

# Verify the configuration that produced the current Kaggle outputs.
assert PROFILE == "SAFE"
assert RANDOM_SEED == 42
assert CFG["feature_count"] == 4
assert CFG["vqc_layers"] == 2
assert CFG["vqc_iterations"] == 30
assert CFG["matched_train_size"] == 100

assert len(y_train_match) == 100
assert len(y_test) == 114

assert vqc_result["status"] == "COMPLETED"
assert vqc_result["iterations_completed"] == 30

assert quantum_kernel_result["status"] == "COMPLETED"
assert quantum_kernel_result["K_train"].shape == (100, 100)
assert quantum_kernel_result["K_test"].shape == (114, 100)

# Extract actual executed results.
judge_results = matched_five_model_table.set_index("Model")

qk = judge_results.loc["Quantum Kernel SVM"]
vqc = judge_results.loc["VQC"]
lr = judge_results.loc["Logistic Regression"]
rbf = judge_results.loc["RBF SVM"]
rf = judge_results.loc["Random Forest"]

# Evidence-supported improvements over Random Forest.
qk_rf_accuracy_gain = float(qk["Accuracy"] - rf["Accuracy"])
qk_rf_sensitivity_gain = float(qk["Sensitivity"] - rf["Sensitivity"])
qk_rf_f1_gain = float(qk["F1"] - rf["F1"])
qk_rf_auc_gain = float(qk["ROC-AUC"] - rf["ROC-AUC"])

qk_rbf_sensitivity_gain = float(
    qk["Sensitivity"] - rbf["Sensitivity"]
)

selected_biomarkers = "<br>".join(
    f"• {feature}"
    for feature in matched_preprocessor.feature_names_out
)

summary_html = f"""
<style>
.qmedai-container {{
    font-family: Arial, sans-serif;
    background: linear-gradient(135deg, #eff6ff 0%, #faf5ff 50%, #fff1f2 100%);
    border: 2px solid #6366f1;
    border-radius: 20px;
    padding: 26px;
    margin: 12px 0;
    color: #0f172a;
}}

.qmedai-title {{
    font-size: 30px;
    font-weight: 800;
    color: #312e81;
    margin-bottom: 4px;
}}

.qmedai-subtitle {{
    font-size: 17px;
    color: #475569;
    margin-bottom: 22px;
}}

.qmedai-grid {{
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 14px;
}}

.qmedai-card {{
    background: white;
    border-radius: 14px;
    padding: 18px;
    box-shadow: 0 5px 15px rgba(15, 23, 42, 0.10);
    border-top: 5px solid #6366f1;
}}

.quantum-card {{
    border-top-color: #e11d48;
}}

.success-card {{
    border-top-color: #16a34a;
}}

.qmedai-number {{
    font-size: 27px;
    font-weight: 800;
    color: #312e81;
}}

.qmedai-label {{
    color: #64748b;
    font-size: 13px;
    margin-top: 4px;
}}

.qmedai-section {{
    background: white;
    border-radius: 14px;
    padding: 18px;
    margin-top: 16px;
    box-shadow: 0 5px 15px rgba(15, 23, 42, 0.08);
}}

.qmedai-highlight {{
    background: #ecfdf5;
    border-left: 6px solid #16a34a;
    padding: 15px;
    margin-top: 16px;
    border-radius: 10px;
}}

.qmedai-warning {{
    background: #fff7ed;
    border-left: 6px solid #f97316;
    padding: 15px;
    margin-top: 16px;
    border-radius: 10px;
}}
</style>

<div class="qmedai-container">

  <div class="qmedai-title">
    Q-MedAI — Hybrid Quantum ML for Disease Detection
  </div>

  <div class="qmedai-subtitle">
    SIH26139 | Executed Kaggle Research Prototype |
    Classical and Quantum Models Compared Fairly
  </div>

  <div class="qmedai-grid">

    <div class="qmedai-card">
      <div class="qmedai-number">569</div>
      <div class="qmedai-label">
        Biomedical observations
      </div>
    </div>

    <div class="qmedai-card">
      <div class="qmedai-number">4</div>
      <div class="qmedai-label">
        Selected biomarkers mapped to 4 qubits
      </div>
    </div>

    <div class="qmedai-card">
      <div class="qmedai-number">5</div>
      <div class="qmedai-label">
        Classical and quantum models evaluated
      </div>
    </div>

    <div class="qmedai-card quantum-card">
      <div class="qmedai-number">
        {qk["Accuracy"]:.2%}
      </div>
      <div class="qmedai-label">
        Quantum Kernel accuracy
      </div>
    </div>

    <div class="qmedai-card quantum-card">
      <div class="qmedai-number">
        {qk["Sensitivity"]:.2%}
      </div>
      <div class="qmedai-label">
        Quantum Kernel malignant-class sensitivity
      </div>
    </div>

    <div class="qmedai-card quantum-card">
      <div class="qmedai-number">
        {qk["ROC-AUC"]:.4f}
      </div>
      <div class="qmedai-label">
        Quantum Kernel ROC-AUC
      </div>
    </div>

  </div>

  <div class="qmedai-section">
    <h3>Quantum implementation evidence</h3>

    <b>Variational Quantum Classifier</b><br>
    • PennyLane default.qubit simulator<br>
    • 4 qubits and 2 variational layers<br>
    • RX, RY and RZ trainable rotations<br>
    • CNOT entanglement ring<br>
    • 30/30 optimization iterations completed<br>
    • Gradients and parameter updates explicitly verified<br><br>

    <b>Quantum Kernel SVM</b><br>
    • Entangled, data-reuploading quantum feature map<br>
    • Fidelity-based quantum kernel<br>
    • K_train = 100 × 100<br>
    • K_test = 114 × 100<br>
    • SVC(kernel="precomputed")<br>
    • ROC-AUC calculated using decision_function()
  </div>

  <div class="qmedai-section">
    <h3>Selected biomedical features</h3>
    {selected_biomarkers}
  </div>

  <div class="qmedai-highlight">
    <b>Where the Quantum Kernel performed better:</b><br><br>

    Compared with Random Forest:<br>
    • Accuracy improvement: {qk_rf_accuracy_gain:+.4f}<br>
    • Sensitivity improvement: {qk_rf_sensitivity_gain:+.4f}<br>
    • F1 improvement: {qk_rf_f1_gain:+.4f}<br>
    • ROC-AUC improvement: {qk_rf_auc_gain:+.4f}<br><br>

    Compared with RBF SVM:<br>
    • Sensitivity improvement: {qk_rbf_sensitivity_gain:+.4f}
  </div>

  <div class="qmedai-warning">
    <b>Scientific conclusion:</b><br>
    The Quantum Kernel outperformed Random Forest across the displayed
    metrics and achieved higher sensitivity than RBF SVM. Logistic
    Regression remained the strongest overall classical reference.
    Therefore, this experiment demonstrates competitive quantum-machine-
    learning performance—not universal quantum advantage.
  </div>

  <div class="qmedai-warning">
    <b>Early-detection relevance:</b><br>
    Higher sensitivity means fewer malignant cases are missed in this
    benchmark. This supports early-detection research, but the current
    experiment is not clinical proof of earlier-in-time diagnosis.
  </div>

</div>
"""

display(HTML(summary_html))

print("\nCOLLEGE JUDGE EVIDENCE SUMMARY PASSED")
print("Next: classical-vs-quantum final comparison graph.")

Q-MedAI — COLLEGE JUDGE EVIDENCE SUMMARY



COLLEGE JUDGE EVIDENCE SUMMARY PASSED
Next: classical-vs-quantum final comparison graph.


In [60]:
# ============================================================
# CELL 14 — CLASSICAL VS QUANTUM COMPARISON DASHBOARD
# Q-MedAI | SIH26139
# ============================================================

from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 76)
print("Q-MedAI — FAIR CLASSICAL VS QUANTUM COMPARISON")
print("=" * 76)

dashboard_table = (
    matched_five_model_table
    .loc[matched_five_model_table["Status"] == "COMPLETED"]
    .copy()
    .reset_index(drop=True)
)

metric_columns = [
    "Accuracy",
    "Sensitivity",
    "Specificity",
    "F1",
    "ROC-AUC",
]

models = dashboard_table["Model"].tolist()

model_colors = {
    "Logistic Regression": "#2563EB",
    "RBF SVM": "#0891B2",
    "Random Forest": "#16A34A",
    "VQC": "#9333EA",
    "Quantum Kernel SVM": "#E11D48",
}

figure, axes = plt.subplots(
    1,
    2,
    figsize=(19, 7.5),
    gridspec_kw={"width_ratios": [1.7, 1]},
)

# ------------------------------------------------------------
# LEFT: all five models on the same held-out test set
# ------------------------------------------------------------

x_positions = np.arange(len(metric_columns))
bar_width = 0.15
offsets = np.linspace(-2, 2, len(models)) * bar_width

for offset, model_name in zip(offsets, models):

    row = dashboard_table.loc[
        dashboard_table["Model"] == model_name
    ].iloc[0]

    values = [
        float(row[metric])
        for metric in metric_columns
    ]

    bars = axes[0].bar(
        x_positions + offset,
        values,
        width=bar_width,
        label=model_name,
        color=model_colors[model_name],
        alpha=0.93,
        edgecolor="white",
        linewidth=0.8,
    )

    # Label only the two quantum models to avoid clutter.
    if model_name in {"VQC", "Quantum Kernel SVM"}:
        for bar, value in zip(bars, values):
            axes[0].text(
                bar.get_x() + bar.get_width() / 2,
                value + 0.004,
                f"{value:.3f}",
                ha="center",
                va="bottom",
                fontsize=8,
                rotation=90,
                fontweight="bold",
                color=model_colors[model_name],
            )

axes[0].set_title(
    "Fair Matched-Data Evaluation\n"
    "Same 100 training rows and 114 held-out test rows",
    fontsize=15,
    fontweight="bold",
)

axes[0].set_xticks(
    x_positions,
    metric_columns,
)

axes[0].set_ylim(0.72, 1.03)
axes[0].set_ylabel("Held-out score — higher is better")
axes[0].grid(axis="y", alpha=0.23)
axes[0].legend(loc="lower left", fontsize=9)

# ------------------------------------------------------------
# RIGHT: Quantum Kernel minus each classical baseline
# ------------------------------------------------------------

qk_row = dashboard_table.loc[
    dashboard_table["Model"] == "Quantum Kernel SVM"
].iloc[0]

classical_names = [
    "Logistic Regression",
    "RBF SVM",
    "Random Forest",
]

delta_matrix = np.asarray([
    [
        float(qk_row[metric])
        - float(
            dashboard_table.loc[
                dashboard_table["Model"] == classical_name,
                metric,
            ].iloc[0]
        )
        for metric in metric_columns
    ]
    for classical_name in classical_names
])

max_abs_delta = max(
    float(np.max(np.abs(delta_matrix))),
    0.001,
)

heatmap = axes[1].imshow(
    delta_matrix,
    cmap="RdYlGn",
    norm=TwoSlopeNorm(
        vmin=-max_abs_delta,
        vcenter=0.0,
        vmax=max_abs_delta,
    ),
    aspect="auto",
)

axes[1].set_title(
    "Quantum Kernel − Classical Baseline\n"
    "Green means the Quantum Kernel scored higher",
    fontsize=14,
    fontweight="bold",
)

axes[1].set_xticks(
    np.arange(len(metric_columns)),
    metric_columns,
    rotation=35,
    ha="right",
)

axes[1].set_yticks(
    np.arange(len(classical_names)),
    classical_names,
)

for row_index in range(delta_matrix.shape[0]):
    for column_index in range(delta_matrix.shape[1]):

        delta = delta_matrix[
            row_index,
            column_index,
        ]

        axes[1].text(
            column_index,
            row_index,
            f"{delta:+.4f}",
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
            color="black",
        )

figure.colorbar(
    heatmap,
    ax=axes[1],
    fraction=0.046,
    pad=0.04,
    label="Score difference",
)

figure.suptitle(
    "Q-MedAI | Classical and Quantum Evidence Dashboard",
    fontsize=20,
    fontweight="bold",
    y=1.02,
)

figure.tight_layout()

teacher_dashboard_path = (
    FIGURES_DIR
    / "teacher_comparison_dashboard.png"
)

figure.savefig(
    teacher_dashboard_path,
    dpi=220,
    bbox_inches="tight",
)

plt.show()

# Create a readable numerical difference table.
delta_records = []

for classical_name, row_values in zip(
    classical_names,
    delta_matrix,
):
    delta_records.append({
        "Comparison":
            f"Quantum Kernel − {classical_name}",
        **{
            metric: float(value)
            for metric, value
            in zip(metric_columns, row_values)
        },
    })

delta_table = pd.DataFrame(delta_records)

display(
    delta_table.style.format({
        metric: "{:+.4f}"
        for metric in metric_columns
    })
)

print(f"\nSaved dashboard: {teacher_dashboard_path}")
print(
    "Conclusion: the Quantum Kernel is selectively stronger, "
    "but the strongest classical baseline remains competitive."
)

Q-MedAI — FAIR CLASSICAL VS QUANTUM COMPARISON


,Comparison,Accuracy,Sensitivity,Specificity,F1,ROC-AUC
0,Quantum Kernel − Logistic Regression,-0.0088,-0.0238,+0.0000,-0.0129,-0.0017
1,Quantum Kernel − RBF SVM,+0.0000,+0.0238,-0.0139,+0.0018,-0.0013
2,Quantum Kernel − Random Forest,+0.0175,+0.0714,-0.0139,+0.0294,+0.0043



Saved dashboard: /kaggle/working/Q-MedAI/results/figures/teacher_comparison_dashboard.png
Conclusion: the Quantum Kernel is selectively stronger, but the strongest classical baseline remains competitive.


In [61]:
# ============================================================
# CELL 15 — QUANTUM EXECUTION AND CIRCUIT PROOF
# ============================================================

print("=" * 76)
print("Q-MedAI — QUANTUM EXECUTION PROOF")
print("=" * 76)

vqc_parameter_count = int(
    np.asarray(
        vqc_result["final_weights"]
    ).size
)

vqc_two_qubit_gates_per_inference = int(
    N_QUBITS * VQC_LAYERS
)

# The fidelity circuit applies the feature map and its adjoint.
qk_two_qubit_gates_per_kernel_evaluation = int(
    2 * QK_N_QUBITS
)

train_kernel_evaluations = int(
    QK_TRAIN_SIZE
    * (QK_TRAIN_SIZE + 1)
    / 2
)

test_kernel_evaluations = int(
    QK_TEST_SIZE
    * QK_TRAIN_SIZE
)

quantum_proof = pd.DataFrame([
    {
        "Quantum evidence": "VQC architecture",
        "Verified observation":
            f"{N_QUBITS} qubits and "
            f"{VQC_LAYERS} variational layers",
    },
    {
        "Quantum evidence": "Trainable VQC parameters",
        "Verified observation":
            vqc_parameter_count,
    },
    {
        "Quantum evidence":
            "VQC two-qubit gates per inference",
        "Verified observation":
            vqc_two_qubit_gates_per_inference,
    },
    {
        "Quantum evidence": "VQC optimization",
        "Verified observation":
            f"{vqc_result['iterations_completed']}/"
            f"{vqc_result['iterations_requested']} iterations",
    },
    {
        "Quantum evidence":
            "Quantum-kernel CNOT gates per overlap",
        "Verified observation":
            qk_two_qubit_gates_per_kernel_evaluation,
    },
    {
        "Quantum evidence":
            "Symmetric training-kernel evaluations",
        "Verified observation":
            train_kernel_evaluations,
    },
    {
        "Quantum evidence":
            "Test × train kernel evaluations",
        "Verified observation":
            test_kernel_evaluations,
    },
    {
        "Quantum evidence": "Quantum backend",
        "Verified observation":
            f"PennyLane {qml.__version__} "
            "default.qubit simulator",
    },
])

display(quantum_proof)

# ------------------------------------------------------------
# Draw the actual trained VQC architecture
# ------------------------------------------------------------

draw_input = pnp.array(
    X_train_match_quantum[0],
    requires_grad=False,
)

draw_weights = pnp.array(
    vqc_result["final_weights"],
    requires_grad=False,
)

try:

    circuit_figure, _ = (
        qml.draw_mpl(
            vqc_circuit
        )(
            draw_input,
            draw_weights,
        )
    )

    circuit_figure.suptitle(
        "Executed Q-MedAI Variational Quantum Circuit",
        fontsize=16,
        fontweight="bold",
    )

    circuit_path = (
        FIGURES_DIR
        / "executed_vqc_circuit.png"
    )

    circuit_figure.savefig(
        circuit_path,
        dpi=220,
        bbox_inches="tight",
    )

    plt.show()

    print(
        f"Saved quantum-circuit diagram: "
        f"{circuit_path}"
    )

except Exception as circuit_draw_error:

    print("Matplotlib circuit drawing was unavailable.")
    print("Text representation of the executed circuit:\n")

    print(
        qml.draw(
            vqc_circuit
        )(
            draw_input,
            draw_weights,
        )
    )

    print(
        type(circuit_draw_error).__name__,
        str(circuit_draw_error),
    )

# ------------------------------------------------------------
# Evidence assertions
# ------------------------------------------------------------

assert vqc_parameter_count == 24

assert (
    len(
        vqc_result[
            "gradient_norm_history"
        ]
    )
    == 30
)

assert np.all(
    np.isfinite(
        vqc_result[
            "gradient_norm_history"
        ]
    )
)

assert (
    quantum_kernel_result[
        "K_train"
    ].shape
    == (100, 100)
)

assert (
    quantum_kernel_result[
        "K_test"
    ].shape
    == (114, 100)
)

print("\nVQC GRADIENT HISTORY VERIFIED")
print("VQC TRAINABLE PARAMETERS VERIFIED")
print("QUANTUM KERNEL MATRIX SHAPES VERIFIED")
print(
    "Backend disclosure: default.qubit simulates "
    "quantum circuits on classical hardware."
)

Q-MedAI — QUANTUM EXECUTION PROOF


,Quantum evidence,Verified observation
0,VQC architecture,4 qubits and 2 variational layers
1,Trainable VQC parameters,24
2,VQC two-qubit gates per inference,8
3,VQC optimization,30/30 iterations
4,Quantum-kernel CNOT gates per overlap,8
5,Symmetric training-kernel evaluations,5050
6,Test × train kernel evaluations,11400
7,Quantum backend,PennyLane 0.45.1 default.qubit simulator


Saved quantum-circuit diagram: /kaggle/working/Q-MedAI/results/figures/executed_vqc_circuit.png

VQC GRADIENT HISTORY VERIFIED
VQC TRAINABLE PARAMETERS VERIFIED
QUANTUM KERNEL MATRIX SHAPES VERIFIED
Backend disclosure: default.qubit simulates quantum circuits on classical hardware.


In [62]:
# ============================================================
# CELL 16 — HELD-OUT INFERENCE DEMONSTRATION
# ============================================================

print("=" * 76)
print("Q-MedAI — HELD-OUT MALIGNANT-CASE DEMONSTRATION")
print("=" * 76)

# Select one malignant observation from the locked test set.
# This observation was not used for model training.
test_targets = np.asarray(
    y_test,
    dtype=int,
)

malignant_positions = np.flatnonzero(
    test_targets == 1
)

demo_position = int(
    malignant_positions[0]
)

actual_label = int(
    test_targets[demo_position]
)

selected_feature_names = list(
    matched_preprocessor.feature_names_out
)

selected_biomarker_values = (
    X_test_raw
    .iloc[demo_position][selected_feature_names]
    .to_frame(name="Observed biomarker value")
)

prediction_rows = []

for model_name, result in matched_model_results.items():

    prediction = int(
        np.asarray(
            result["y_pred"]
        )[demo_position]
    )

    score = float(
        np.asarray(
            result["y_score"]
        )[demo_position]
    )

    if model_name == "Quantum Kernel SVM":
        score_type = "SVM decision-function score"
    else:
        score_type = "Probability or continuous model score"

    prediction_rows.append({
        "Model": model_name,
        "Family": (
            "Quantum"
            if model_name
            in {"VQC", "Quantum Kernel SVM"}
            else "Classical"
        ),
        "Prediction": (
            "Malignant (M)"
            if prediction == 1
            else "Benign (B)"
        ),
        "Continuous score": score,
        "Score type": score_type,
        "Correct for this row":
            prediction == actual_label,
    })

prediction_demo = pd.DataFrame(
    prediction_rows
)

print(
    "Actual held-out class:",
    "Malignant (M)"
    if actual_label == 1
    else "Benign (B)",
)

print(
    "\nSelected biomarker values used "
    "by the fitted preprocessing pipeline:"
)

display(selected_biomarker_values)

print(
    "\nClassical and quantum predictions:"
)

display(
    prediction_demo.style.format({
        "Continuous score": "{:.6f}"
    })
)

agreement_count = int(
    prediction_demo[
        "Prediction"
    ]
    .value_counts()
    .max()
)

print(
    f"\nMajority agreement: "
    f"{agreement_count}/"
    f"{len(prediction_demo)} models"
)

print(
    "This is an inference demonstration on a benchmark "
    "test observation, not a clinical patient diagnosis."
)

Q-MedAI — HELD-OUT MALIGNANT-CASE DEMONSTRATION
Actual held-out class: Malignant (M)

Selected biomarker values used by the fitted preprocessing pipeline:


,Observed biomarker value
concave points_mean,0.1310
radius_worst,25.5800
perimeter_worst,165.3000
concave points_worst,0.2105



Classical and quantum predictions:


,Model,Family,Prediction,Continuous score,Score type,Correct for this row
0,Logistic Regression,Classical,Malignant (M),0.999402,Probability or continuous model score,True
1,RBF SVM,Classical,Malignant (M),0.967756,Probability or continuous model score,True
2,Random Forest,Classical,Malignant (M),1.000000,Probability or continuous model score,True
3,VQC,Quantum,Benign (B),0.312712,Probability or continuous model score,False
4,Quantum Kernel SVM,Quantum,Malignant (M),0.864710,SVM decision-function score,True



Majority agreement: 4/5 models
This is an inference demonstration on a benchmark test observation, not a clinical patient diagnosis.


In [63]:
# ============================================================
# CELL 19 — EXPORT COLLEGE / SIH EVIDENCE BUNDLE
# ============================================================

import json
import shutil
from pathlib import Path

print("=" * 76)
print("Q-MedAI — COLLEGE REVIEW EVIDENCE EXPORT")
print("=" * 76)

EVIDENCE_DIR = Path(
    "/kaggle/working/Q-MedAI_SIH26139_Evidence"
)

EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Convert the DataFrame through JSON so all values are
# standard JSON-compatible Python values.
json_safe_models = json.loads(
    dashboard_table.to_json(
        orient="records"
    )
)

teacher_summary = {
    "project": "Q-MedAI",
    "problem_statement": (
        "SIH26139 — Hybrid Quantum Machine Learning "
        "Platform for Early Disease Detection"
    ),
    "experiment": "Executed Kaggle SAFE profile",
    "configuration": {
        "random_seed": 42,
        "matched_training_rows": 100,
        "held_out_test_rows": 114,
        "selected_features": list(
            matched_preprocessor.feature_names_out
        ),
        "vqc_qubits": int(N_QUBITS),
        "vqc_layers": int(VQC_LAYERS),
        "vqc_iterations_completed": int(
            vqc_result[
                "iterations_completed"
            ]
        ),
        "quantum_backend": (
            "PennyLane default.qubit "
            "classical quantum-circuit simulator"
        ),
    },
    "models": json_safe_models,
    "scientific_conclusion": (
        "The Quantum Kernel SVM outperformed Random Forest "
        "across the displayed metrics and achieved higher "
        "sensitivity than RBF SVM. Logistic Regression "
        "remained the strongest overall classical reference. "
        "This experiment does not establish universal quantum "
        "advantage or clinical validity."
    ),
}

summary_path = (
    EVIDENCE_DIR
    / "teacher_summary.json"
)

summary_path.write_text(
    json.dumps(
        teacher_summary,
        indent=2,
    ),
    encoding="utf-8",
)

dashboard_table.to_csv(
    EVIDENCE_DIR
    / "matched_five_model_results.csv",
    index=False,
)

prediction_demo.to_csv(
    EVIDENCE_DIR
    / "held_out_inference_demo.csv",
    index=False,
)

delta_table.to_csv(
    EVIDENCE_DIR
    / "quantum_kernel_metric_differences.csv",
    index=False,
)

destination_figures = (
    EVIDENCE_DIR
    / "figures"
)

destination_figures.mkdir(
    parents=True,
    exist_ok=True,
)

if FIGURES_DIR.exists():

    for figure_path in FIGURES_DIR.glob(
        "*.png"
    ):
        shutil.copy2(
            figure_path,
            destination_figures
            / figure_path.name,
        )

evidence_readme = (
    "# Q-MedAI SIH26139 Evidence Bundle\n\n"
    "This bundle was generated from the executed Kaggle "
    "SAFE-profile experiment.\n\n"
    "It contains the fair matched-data result table, "
    "held-out inference demonstration, comparison figures, "
    "and a structured result summary.\n\n"
    "PennyLane default.qubit is a classical simulator of "
    "quantum circuits. The results do not establish universal "
    "quantum advantage or clinical validity.\n"
)

(
    EVIDENCE_DIR
    / "README.md"
).write_text(
    evidence_readme,
    encoding="utf-8",
)

evidence_zip = shutil.make_archive(
    "/kaggle/working/"
    "Q-MedAI_SIH26139_Evidence",
    "zip",
    EVIDENCE_DIR,
)

print(f"Evidence directory: {EVIDENCE_DIR}")
print(f"Downloadable ZIP:   {evidence_zip}")
print(
    "The final graph in the next cell will also "
    "be added to this ZIP."
)

Q-MedAI — COLLEGE REVIEW EVIDENCE EXPORT
Evidence directory: /kaggle/working/Q-MedAI_SIH26139_Evidence
Downloadable ZIP:   /kaggle/working/Q-MedAI_SIH26139_Evidence.zip
The final graph in the next cell will also be added to this ZIP.


In [64]:
# ============================================================
# CELL 20 — FINAL Q-MedAI JUDGE SUMMARY
# ============================================================

import shutil

print("=" * 76)
print("Q-MedAI — FINAL JUDGE SUMMARY")
print("=" * 76)

judge_models = (
    dashboard_table[
        "Model"
    ].tolist()
)

judge_metrics = [
    "Accuracy",
    "Sensitivity",
    "F1",
    "ROC-AUC",
]

judge_figure = plt.figure(
    figsize=(18, 9),
    facecolor="#F8FAFC",
)

judge_grid = judge_figure.add_gridspec(
    2,
    2,
    width_ratios=[1.55, 1],
    height_ratios=[1, 0.72],
    hspace=0.34,
    wspace=0.25,
)

# ------------------------------------------------------------
# MAIN METRIC COMPARISON
# ------------------------------------------------------------

metrics_axis = judge_figure.add_subplot(
    judge_grid[:, 0]
)

x = np.arange(
    len(judge_metrics)
)

width = 0.15

offsets = (
    np.linspace(
        -2,
        2,
        len(judge_models),
    )
    * width
)

for offset, model_name in zip(
    offsets,
    judge_models,
):

    row = dashboard_table.loc[
        dashboard_table["Model"]
        == model_name
    ].iloc[0]

    values = [
        float(row[metric])
        for metric in judge_metrics
    ]

    bars = metrics_axis.bar(
        x + offset,
        values,
        width,
        label=model_name,
        color=model_colors[model_name],
        edgecolor="white",
        linewidth=0.8,
    )

    # Highlight Quantum Kernel values.
    if model_name == "Quantum Kernel SVM":

        for bar, value in zip(
            bars,
            values,
        ):

            metrics_axis.text(
                bar.get_x()
                + bar.get_width() / 2,
                value + 0.006,
                f"{value:.3f}",
                ha="center",
                va="bottom",
                fontsize=9,
                fontweight="bold",
                color="#9F1239",
                rotation=90,
            )

metrics_axis.set_title(
    "Classical vs Quantum — Fair Matched Comparison",
    fontsize=17,
    fontweight="bold",
    color="#0F172A",
)

metrics_axis.set_xticks(
    x,
    judge_metrics,
)

metrics_axis.set_ylim(
    0.72,
    1.03,
)

metrics_axis.set_ylabel(
    "Held-out score — higher is better"
)

metrics_axis.grid(
    axis="y",
    alpha=0.22,
)

metrics_axis.legend(
    loc="lower left",
    fontsize=9,
    frameon=True,
)

metrics_axis.text(
    0.02,
    0.98,
    "Quantum Kernel highlighted in red",
    transform=metrics_axis.transAxes,
    va="top",
    fontsize=10,
    color="#9F1239",
    fontweight="bold",
)

# ------------------------------------------------------------
# FALSE NEGATIVES
# Lower false negatives = fewer malignant cases missed
# ------------------------------------------------------------

false_negative_axis = (
    judge_figure.add_subplot(
        judge_grid[0, 1]
    )
)

false_negatives = []

for model_name in judge_models:

    matrix = np.asarray(
        matched_model_results[
            model_name
        ][
            "confusion_matrix"
        ],
        dtype=int,
    )

    # Confusion matrix:
    # [[true negatives, false positives],
    #  [false negatives, true positives]]
    false_negatives.append(
        int(matrix[1, 0])
    )

fn_bars = false_negative_axis.barh(
    judge_models,
    false_negatives,
    color=[
        model_colors[name]
        for name in judge_models
    ],
)

false_negative_axis.invert_yaxis()

false_negative_axis.set_title(
    "Malignant Cases Missed\n"
    "(False Negatives)",
    fontsize=14,
    fontweight="bold",
)

false_negative_axis.set_xlabel(
    "Count on the same 114-row test set — lower is better"
)

false_negative_axis.grid(
    axis="x",
    alpha=0.22,
)

for bar, value in zip(
    fn_bars,
    false_negatives,
):

    false_negative_axis.text(
        value + 0.15,
        bar.get_y()
        + bar.get_height() / 2,
        str(value),
        va="center",
        fontweight="bold",
    )

# ------------------------------------------------------------
# JUDGE TAKEAWAY
# ------------------------------------------------------------

story_axis = judge_figure.add_subplot(
    judge_grid[1, 1]
)

story_axis.axis("off")

story_axis.text(
    0.02,
    0.98,
    "WHY THE QUANTUM RESULT MATTERS",
    va="top",
    fontsize=15,
    fontweight="bold",
    color="#312E81",
)

story_axis.text(
    0.02,
    0.78,
    "Quantum Kernel SVM\n"
    "• Accuracy: 94.74%\n"
    "• Sensitivity: 90.48%\n"
    "• F1: 0.9268\n"
    "• ROC-AUC: 0.9944\n\n"
    "It beat Random Forest across these displayed\n"
    "metrics and achieved higher sensitivity than\n"
    "RBF SVM.\n\n"
    "Sensitivity measures how many malignant cases\n"
    "are detected. This is relevant to early-detection\n"
    "research, but clinical early-diagnosis claims\n"
    "require external longitudinal validation.",
    va="top",
    fontsize=11,
    color="#1E293B",
    linespacing=1.35,
)

judge_figure.suptitle(
    "Q-MedAI | Quantum-Assisted Disease Detection Research",
    fontsize=21,
    fontweight="bold",
    color="#0F172A",
    y=0.99,
)

judge_figure.text(
    0.5,
    0.015,
    "PennyLane default.qubit simulator  •  "
    "Same 100 training rows  •  "
    "Same 114 held-out test rows  •  "
    "No universal quantum-advantage claim",
    ha="center",
    fontsize=10,
    color="#475569",
)

final_judge_graph_path = (
    FIGURES_DIR
    / "final_judge_summary.png"
)

judge_figure.savefig(
    final_judge_graph_path,
    dpi=220,
    bbox_inches="tight",
)

# ------------------------------------------------------------
# ADD FINAL GRAPH TO EVIDENCE ZIP
# ------------------------------------------------------------

destination_figures = (
    EVIDENCE_DIR
    / "figures"
)

destination_figures.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy2(
    final_judge_graph_path,
    destination_figures
    / final_judge_graph_path.name,
)

evidence_zip = shutil.make_archive(
    "/kaggle/working/"
    "Q-MedAI_SIH26139_Evidence",
    "zip",
    EVIDENCE_DIR,
)

print(
    f"Final judge graph saved: "
    f"{final_judge_graph_path}"
)

print(
    f"Updated evidence ZIP: "
    f"{evidence_zip}"
)

plt.show()

Q-MedAI — FINAL JUDGE SUMMARY


/tmp/ipykernel_58/1552731014.py:24: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  judge_figure = plt.figure(


Final judge graph saved: /kaggle/working/Q-MedAI/results/figures/final_judge_summary.png
Updated evidence ZIP: /kaggle/working/Q-MedAI_SIH26139_Evidence.zip
